# NLU Phenikaa Courier — BamiBERT

Luồng: cài đặt → dữ liệu (train + val, **không đọc test**) → sinh dữ liệu cân bằng → BamiBERT (không cần pyvi) → train (EMA) → đánh giá nhiều góc nhìn.

Khác bản cũ:
- Bỏ `pyvi`: BamiBERT nhận chữ thô, chạy thẳng không cần tách từ.
- Chọn checkpoint theo **trung bình nhiều góc nhìn** (gốc / bỏ dấu / typo / bỏ dấu + typo) và dùng **EMA** weights, không chọn theo 1 view val 300 câu.
- **Giải mã có ràng buộc** (goal ≠ via, anchor chỉ có khi near/far, ...). Ràng buộc nào vi phạm trong nhãn thật thì tự tắt.
- Đánh giá có khoảng tin cậy 95%, đối chiếu argmax với giải mã có ràng buộc, in câu sai, kiểm tra cắt token.

In [ ]:
!pip install -q "transformers<=5.5.0" sentencepiece protobuf

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!unzip -q -o "/content/drive/MyDrive/Phenikaa_Campus_Courier_2026.zip" -d "/content/dataset"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, json, random, re, unicodedata, collections, gc, shutil, math
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

ROOT = "/content/dataset/Phenikaa_Campus_Courier_2026/delivery_public"
TRAIN_SCENES, VAL_SCENES = f"{ROOT}/train/scenes.json", f"{ROOT}/validation/scenes.json"

MODEL_NAME = "Qualcomm-AI-Research/BamiBERT"
SEED = 42
TRAIN_ON_ALL = False      # False: train bằng train, dùng val để chọn epoch.  True: gộp val vào train cho lần train CUỐI (cần FIXED_EPOCHS)
FIXED_EPOCHS = None       # điền số epoch tốt nhất (in ra sau lần chạy đầu) khi TRAIN_ON_ALL=True

# --- train ---
MAX_LEN, BATCH_SIZE = 192, 32          # 192 để câu không dấu + typo không bị cắt mất đuôi (cờ urgent/fragile hay nằm cuối câu)
EPOCHS, PATIENCE =70, 70
LR_ENC, LR_HEAD, WD, LABEL_SMOOTH = 3e-5, 1e-3, 0.01, 0.05
EMA_DECAY = 0.998
REAL_REPEAT = 2           # câu thật của train được lấy 2 lần / epoch (mỗi lần 1 bản nhiễu khác nhau)
P_CLEAN = 0.30            # xác suất giữ nguyên câu gốc, không nhiễu
P_PAIR = 0.50             # xác suất kèm bản bỏ dấu của câu (loss CE + căn chỉnh embedding)
ALIGN_W = 2.0
SYN_P = 0.8               # xác suất đổi đồng nghĩa trong augment

# --- cân bằng dữ liệu sinh thêm ---
# --- cân bằng dữ liệu sinh thêm ---
BALANCE_FRAC    = 1.0
# Đẩy số mẫu tối thiểu lên ngang bằng với số lượng mẫu của các class phổ biến (~200 mẫu)
MIN_PER_CLASS   = {'via': 200, 'gkind': 200, 'ganchor': 150, 'vkind': 200, 'vanchor': 150}
YES_RATIO       = 0.50           # Kéo tỷ lệ cờ urgent/fragile (yes/no) về mức 50/50 thay vì 40/60
SYN_FRACTION    = 0.75           # TĂNG TỪ 0.45 LÊN 0.75: Cho phép data sinh thêm chiếm tới 75% tập Train
MAX_SYN         = 6000           # Tăng trần số lượng câu sinh ra (để bù đắp đủ 150-200 mẫu cho từng class nhỏ)

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print(device)
USE_AMP = device.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
print("AMP:", USE_AMP, AMP_DTYPE)

cuda
AMP: True torch.bfloat16


In [ ]:
EPOCHS, PATIENCE = 3, 12          # schedule thật sự chạy hết; early-stop không cắt giữa chừng
WARMUP_FRAC = 0.08                 # ~1 epoch warmup (trước là 10% của 50 epoch = 5 epoch)
NOISE_HI, NOISE_LO = 0.6, 0.3      # 70% epoch đầu nhiễu vừa, 30% cuối nhiễu nhẹ

In [ ]:
PLACES = ['canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports']
KINDS  = ['none','near','far','north','south','east','west']
HEADS = {'goal': PLACES, 'via': ['none']+PLACES,
         'gkind': KINDS, 'ganchor': ['none']+PLACES,     # goal_ref.kind / .anchor
         'vkind': KINDS, 'vanchor': ['none']+PLACES,     # via_ref.kind / .anchor
         'urgent': ['no','yes'], 'fragile': ['no','yes']}
HEAD_W = {'goal':2.0,'via':2.0,'gkind':1.0,'ganchor':1.0,'vkind':1.0,'vanchor':1.0,'urgent':1.2,'fragile':1.0}
H2I = {h:{l:i for i,l in enumerate(ls)} for h,ls in HEADS.items()}

def mission_to_labels(m):
    g, v = m.get('goal_ref') or {}, m.get('via_ref') or {}
    raw = dict(goal=m['goal'], via=m['via'] or 'none', gkind=g.get('kind') or 'none', ganchor=g.get('anchor') or 'none',
               vkind=v.get('kind') or 'none', vanchor=v.get('anchor') or 'none',
               urgent='yes' if m['urgent'] else 'no', fragile='yes' if m['fragile'] else 'no')
    return {h: H2I[h][x] for h, x in raw.items()}

def load_missions(path):
    seen, out = set(), []          # 1 scene = 10 robot cùng 1 câu -> giữ câu duy nhất
    for sc in json.load(open(path, encoding='utf-8')):
        m = sc['mission']
        if m['text'] in seen: continue
        seen.add(m['text']); out.append({'text': m['text'], 'y': mission_to_labels(m)})
    return out

base_items, val_items = load_missions(TRAIN_SCENES), load_missions(VAL_SCENES)
print("train gốc:", len(base_items), "| val:", len(val_items))
print("câu trùng giữa train và val:", len({i['text'] for i in base_items} & {i['text'] for i in val_items}))

train gốc: 2000 | val: 300
câu trùng giữa train và val: 0


In [ ]:
# ---------- BÁO CÁO PHÂN PHỐI CLASS + LỖI ----------
def dist_report(items, title='TRAIN'):
    print(f"\n===== PHÂN PHỐI CLASS — {title} (n={len(items)}) =====")
    for h, labels in HEADS.items():
        c = collections.Counter(it['y'][h] for it in items)
        real = [c.get(i, 0) for i, l in enumerate(labels) if l not in ('none', 'no')]
        real_pos = [x for x in real if x > 0]
        ratio = (max(real_pos) / min(real_pos)) if real_pos else 0
        print(f"\n[{h}]  max/min (bỏ none, bỏ class 0 mẫu) = {ratio:.1f}x" + ("  ⚠️ có class 0 mẫu" if 0 in real else ""))
        for i, l in enumerate(labels):
            n = c.get(i, 0); print(f"  {l:8s} {n:5d} {100*n/len(items):5.1f}% " + '#' * int(40 * n / len(items)))

dist_report(base_items, 'TRAIN gốc')


===== PHÂN PHỐI CLASS — TRAIN gốc (n=2000) =====

[goal]  max/min (bỏ none, bỏ class 0 mẫu) = 1.2x
  canteen    210  10.5% ####
  clinic     190   9.5% ###
  dorm       217  10.8% ####
  gate       187   9.3% ###
  lab        195   9.8% ###
  lecture    193   9.7% ###
  library    211  10.6% ####
  office     195   9.8% ###
  parking    180   9.0% ###
  sports     222  11.1% ####

[via]  max/min (bỏ none, bỏ class 0 mẫu) = 2.0x
  none      1370  68.5% ###########################
  canteen     40   2.0% 
  clinic      55   2.8% #
  dorm        64   3.2% #
  gate        51   2.5% #
  lab         79   4.0% #
  lecture     71   3.5% #
  library     66   3.3% #
  office      71   3.5% #
  parking     62   3.1% #
  sports      71   3.5% #

[gkind]  max/min (bỏ none, bỏ class 0 mẫu) = 3.4x
  none      1760  88.0% ###################################
  near        63   3.1% #
  far         75   3.8% #
  north       32   1.6% 
  south       24   1.2% 
  east        24   1.2% 
  west        22  

In [ ]:
# ---------- Ngân hàng đồng nghĩa ----------
def strip_accents(t):
    t = t.replace('đ','d').replace('Đ','D')
    return ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
def fold(s): return strip_accents(s).lower()

TRAIN_ALIASES = {
 'canteen': ['căn tin','canteen','nhà ăn','khu ăn uống','bếp ăn'],
 'clinic':  ['trạm y tế','phòng khám của trường','phòng khám','trạm xá','phòng sơ cứu','khu y tế'],
 'dorm':    ['ký túc xá','ktx','nhà ở của sinh viên','khu nội trú','phòng ở sinh viên'],
 'gate':    ['cổng trường','cổng vào','cổng a','chốt bảo vệ cổng','chốt bảo vệ','lối vào trường','cổng chính'],
 'lab':     ['phòng thí nghiệm','phòng thực hành','lab hóa','xưởng thực hành','khu thí nghiệm'],
 'lecture': ['giảng đường','tòa giảng đường','hội trường học','phòng học lớn','lớp học'],
 'library': ['thư viện','thư viện trường','chỗ trả sách','nơi mượn giáo trình','phòng đọc','khu đọc sách'],
 'office':  ['phòng hành chính','văn phòng khoa','phòng đào tạo','phòng một cửa','tòa hiệu bộ','khu hành chính'],
 'parking': ['bãi xe','nhà xe','khu gửi xe','bãi đỗ xe','hầm xe','chỗ để xe'],
 'sports':  ['sân tập','nhà thi đấu','sân bóng','khu thể thao','sân thể thao','nhà thể chất'],
}
EXTRA_ALIASES = {
 'canteen': ['nhà ăn sinh viên','quán cơm trong trường','chỗ ăn trưa','nơi bán cơm','căng tin trường','nhà bếp tập thể','nơi dùng bữa'],
 'clinic':  ['phòng y tế','nơi sơ cứu','chỗ khám bệnh','phòng khám sức khỏe','khu chăm sóc sức khỏe','phòng cấp cứu','trạm y tế trường'],
 'dorm':    ['ký túc','nhà ở sinh viên','khu ký túc xá','tòa nhà ở sinh viên','chỗ ở sinh viên','dãy nhà nội trú','khu lưu trú'],
 'gate':    ['cổng ra vào','cổng chính của trường','chốt gác','phòng bảo vệ','trạm gác','cổng đón khách','lối ra vào trường'],
 'lab':     ['phòng lab','xưởng thí nghiệm','phòng nghiên cứu','khu lab','phòng thực tập','xưởng thực tập','nơi thực hành thí nghiệm'],
 'lecture': ['phòng giảng','giảng đường lớn','tòa nhà học','khu giảng đường','phòng học','dãy lớp học','nơi giảng bài'],
 'library': ['thư viện sách','khu thư viện','phòng thư viện','nơi trả sách','chỗ mượn sách','phòng đọc sách','nơi mượn tài liệu'],
 'office':  ['văn phòng trường','phòng công tác sinh viên','nơi làm thủ tục','phòng văn thư','khu văn phòng','phòng tiếp nhận hồ sơ','tòa nhà hành chính'],
 'parking': ['chỗ đỗ xe','bãi giữ xe','bãi để xe','khu để xe','nhà để xe','hầm để xe','điểm gửi xe','bãi đậu xe'],
 'sports':  ['sân vận động','nhà tập','phòng tập thể thao','khu luyện tập','sân chơi thể thao','nhà đa năng','khu thể dục'],
}
# Cụm mới do mình tự viết thêm (không lấy từ val/test)
NEW_ALIASES = {
 'canteen': ['quán ăn sinh viên','khu bán đồ ăn','nhà ăn trường','khu ẩm thực'],
 'clinic':  ['trung tâm y tế trường','khu khám bệnh','điểm y tế','phòng chăm sóc sức khỏe'],
 'dorm':    ['nhà nội trú','nhà ở nội trú','ký túc xá sinh viên','tòa ký túc'],
 'gate':    ['cổng bảo vệ','cổng trường học','điểm bảo vệ ở cổng'],
 'lab':     ['phòng thí nghiệm trung tâm','khu nghiên cứu thực hành','phòng thí nghiệm hóa'],
 'lecture': ['tòa lớp học','giảng đường chính','phòng học tập trung'],
 'library': ['trung tâm học liệu','thư viện chính','nơi đọc sách','quầy mượn trả sách'],
 'office':  ['văn phòng hành chính','quầy tiếp nhận hồ sơ','phòng hành chính tổng hợp'],
 'parking': ['bãi xe máy','điểm đỗ xe máy','nơi đỗ xe'],
 'sports':  ['sân thi đấu','nhà luyện tập','khu thể chất','sân bóng đá'],
}
# Chỉ dùng để ĐO "câu có đồng nghĩa chưa thấy" trên val. Không đưa vào train (trừ khi TRAIN_ON_ALL).
VAL_ONLY_ALIASES = {
 'canteen': ['nơi phục vụ bữa trưa','quầy ăn','phòng ăn tập thể'],
 'clinic':  ['nơi khám sức khỏe','phòng bác sĩ'],
 'dorm':    ['dãy phòng nội trú','khu ở của sinh viên'],
 'gate':    ['chốt cổng','nơi khách đến trường'],
 'lab':     ['nơi làm thí nghiệm','phòng thực nghiệm'],
 'lecture': ['dãy phòng học','nơi lên lớp'],
 'library': ['phòng tài liệu','nơi mượn sách tham khảo'],
 'office':  ['phòng giáo vụ','nơi nộp hồ sơ'],
 'parking': ['khu đậu xe','nơi giữ xe'],
 'sports':  ['sân luyện tập','khu vận động'],
}
BANK = {c: list(TRAIN_ALIASES[c]) + list(EXTRA_ALIASES[c]) + list(NEW_ALIASES[c])
           + (VAL_ONLY_ALIASES[c] if TRAIN_ON_ALL else []) for c in PLACES}

# --- kiểm tra an toàn ---
ALIAS2CLS = {}
for c, al in BANK.items():
    for a in al:
        assert ALIAS2CLS.get(fold(a), c) == c, f"cụm trùng giữa 2 lớp: {a}"
        ALIAS2CLS[fold(a)] = c
if not TRAIN_ON_ALL:
    leak = {fold(a) for al in VAL_ONLY_ALIASES.values() for a in al} & set(ALIAS2CLS)
    assert not leak, f"alias của val lọt vào train: {leak}"
for a, ca in ALIAS2CLS.items():          # alias lớp A nằm lọt trong alias lớp B
    for b, cb in ALIAS2CLS.items():
        if ca != cb and re.search(r'(?<!\w)' + re.escape(a) + r'(?!\w)', b):
            print(f"⚠️ '{a}'({ca}) nằm trong '{b}'({cb})")

ALIAS_RE = re.compile(r'(?<!\w)(?:' + '|'.join(re.escape(a) for a in sorted(ALIAS2CLS, key=len, reverse=True)) + r')(?!\w)')
VI = re.compile('[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]')

def swap_synonyms(text, rng=random, p=SYN_P):
    t = unicodedata.normalize('NFC', text); f = ''.join(strip_accents(c) for c in t).lower()
    if len(t) != len(f): return text
    accented = bool(VI.search(t.lower())); out, last = [], 0
    for m in ALIAS_RE.finditer(f):
        s, e = m.span(); cls = ALIAS2CLS[m.group()]
        if cls == 'library' and f[max(0, s-4):s] == 'the ': continue
        if rng.random() > p: continue
        new = rng.choice(BANK[cls])
        if not accented: new = strip_accents(new)
        if t[s].isupper(): new = new[0].upper() + new[1:]
        out.append(t[last:s]); out.append(new); last = e
    out.append(t[last:]); return ''.join(out)

WORD_RE = re.compile(r'[^\W\d_]{3,}')
def strip_some(text, rng=random, q=0.5):
    return re.sub(r'\w+', lambda m: strip_accents(m.group()) if rng.random() < q else m.group(), text)

def one_typo(text, rng=random):
    ws = list(WORD_RE.finditer(text))
    if not ws: return text
    for _ in range(5):
        m = rng.choice(ws); w = m.group(); i = rng.randrange(len(w)); op = rng.choice(['swap','drop','dup','tone'])
        if op == 'swap' and i < len(w)-1: w2 = w[:i] + w[i+1] + w[i] + w[i+2:]
        elif op == 'drop': w2 = w[:i] + w[i+1:]
        elif op == 'dup':  w2 = w[:i] + w[i] + w[i:]
        else:              w2 = w[:i] + strip_accents(w[i]) + w[i+1:]
        if w2 != w: return text[:m.start()] + w2 + text[m.end():]
    return text

_TRAP_CL = ["Không cần ghé {t}", "Đừng nhầm với {t} nhé", "Tuyệt đối tránh {t}", "Bỏ qua {t}", "Không phải {t}"]
def _insert_trap(text, t, rng):
    clause = rng.choice(_TRAP_CL).format(t=t) + '.'
    parts = re.split(r'(?<=[.!?])\s+', text.strip())      # chỉ chèn ở RANH GIỚI câu, không chèn giữa chừng
    pos = rng.randrange(len(parts) + 1)
    if pos > 0 and parts[pos-1][-1:] not in '.!?': parts[pos-1] += '.'
    parts.insert(pos, clause)
    return ' '.join(parts)

def augment(text, y=None, rng=random, noise_scale=1.0):
    if noise_scale == 0: return text
    text = swap_synonyms(text, rng, p=SYN_P * noise_scale)
    if y is not None:
        used_places = {HEADS[k][y[k]] for k in HEADS if HEADS[k][y[k]] in PLACES}
        free_places = [p for p in PLACES if p not in used_places]
        if free_places and rng.random() <= (0.3 * noise_scale):
            text = _insert_trap(text, rng.choice(BANK[rng.choice(free_places)]), rng)
    r = rng.random()
    if r < 0.35:   text = strip_accents(text)       # không nhân noise_scale: val có ~25% câu không dấu thật
    elif r < 0.50: text = strip_some(text, rng)
    if rng.random() < (0.3 * noise_scale): text = one_typo(text, rng)
    if rng.random() < (0.2 * noise_scale): text = text.lower()
    return text

In [ ]:
# ---------- Nhiễu tổng quát mạnh hơn (không phụ thuộc nội dung test) ----------
_ROWS = ['qwertyuiop', 'asdfghjkl', 'zxcvbnm']
_NEIGH = {}
for _row in _ROWS:
    for _c, _ch in enumerate(_row):
        _NEIGH[_ch] = ([_row[_c - 1]] if _c > 0 else []) + ([_row[_c + 1]] if _c < len(_row) - 1 else [])
_WORD2 = re.compile(r'[^\W\d_]{2,}')

def _typo_word(w, rng):
    if len(w) < 2: return w
    i = rng.randrange(len(w))
    op = rng.choices(['swap', 'drop', 'dup', 'neigh', 'tone'], [0.35, 0.2, 0.15, 0.15, 0.15])[0]
    if op == 'swap' and i < len(w) - 1: return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    if op == 'drop' and len(w) > 2:     return w[:i] + w[i + 1:]
    if op == 'dup':                     return w[:i] + w[i] + w[i:]
    if op == 'neigh':
        c = strip_accents(w[i]).lower()
        if c in _NEIGH and _NEIGH[c]: return w[:i] + rng.choice(_NEIGH[c]) + w[i + 1:]
    if op == 'tone':                    return w[:i] + strip_accents(w[i]) + w[i + 1:]
    return w

def multi_typo(text, rng=random, rate=0.08, max_per_sent=4):
    ws = list(_WORD2.finditer(text))
    if not ws: return text
    k = min(max_per_sent, sum(rng.random() < rate for _ in ws))
    if k == 0: return text
    for idx in sorted(rng.sample(range(len(ws)), min(k, len(ws))), reverse=True):
        m = ws[idx]; text = text[:m.start()] + _typo_word(m.group(), rng) + text[m.end():]
    return text

def merge_or_split(text, rng=random, p=0.08):
    toks = text.split(' '); out, i = [], 0
    while i < len(toks):
        t = toks[i]
        if rng.random() < p:
            if rng.random() < 0.5 and i + 1 < len(toks) and t.isalpha() and toks[i + 1].isalpha():
                out.append(t + toks[i + 1]); i += 2; continue
            elif len(t) >= 4 and t.isalpha():
                j = rng.randrange(1, len(t) - 1); out.append(t[:j] + ' ' + t[j:]); i += 1; continue
        out.append(t); i += 1
    return ' '.join(out)

def has_accent(t): return bool(VI.search(t.lower()))

def augment_v2(text, y=None, rng=random, noise_scale=1.0):
    t = augment(text, y, rng=rng, noise_scale=noise_scale)           # đồng nghĩa + bẫy + bỏ dấu + 1 typo (bản gốc)
    if rng.random() < 0.5 * noise_scale: t = multi_typo(t, rng, rate=rng.choice([0.03, 0.05, 0.08]))
    if rng.random() < 0.15 * noise_scale: t = merge_or_split(t, rng)
    return t

_r = random.Random(1)
print(augment_v2("Robot ơi, giao hộp giấy đến thư viện giúp mình. Đừng nhầm với phòng lab nhé. Càng nhanh càng tốt.", None, _r, 1.0))

Robot ơi, giao hộp giấy đến chỗ tra sách igúp mình. Đừng nhầm vớiphòng nghiên cứu nhé. Càng nhanh càng ốt.


In [ ]:
# ---------- Kiểm tra ràng buộc cấu trúc trên NHÃN THẬT (train gốc + val) ----------
KINDS_NF = ('near', 'far')
def check_rules(items):
    v = collections.Counter()
    for it in items:
        y = {h: HEADS[h][it['y'][h]] for h in HEADS}
        if y['via'] != 'none' and y['via'] == y['goal']: v['goal_ne_via'] += 1
        for k, a, own in (('gkind', 'ganchor', y['goal']), ('vkind', 'vanchor', y['via'])):
            if (y[k] in KINDS_NF) != (y[a] != 'none'): v['anchor_iff'] += 1
            if y[a] != 'none' and y[a] == own: v['anchor_ne_own'] += 1
        if y['via'] == 'none' and (y['vkind'] != 'none' or y['vanchor'] != 'none'): v['via_none_clears'] += 1
    return v

_viol = check_rules(base_items + val_items)
RULE_NAMES = ['goal_ne_via', 'anchor_iff', 'anchor_ne_own', 'via_none_clears']
RULES = {r: _viol[r] == 0 for r in RULE_NAMES}
print(f"Số vi phạm trong {len(base_items) + len(val_items)} câu thật:", {r: _viol[r] for r in RULE_NAMES})
print("Ràng buộc được BẬT (0 vi phạm):", [r for r, on in RULES.items() if on])
if not all(RULES.values()): print("⚠️ Ràng buộc bị tắt vì nhãn thật có vi phạm:", [r for r, on in RULES.items() if not on])

Số vi phạm trong 2300 câu thật: {'goal_ne_via': 0, 'anchor_iff': 0, 'anchor_ne_own': 0, 'via_none_clears': 0}
Ràng buộc được BẬT (0 vi phạm): ['goal_ne_via', 'anchor_iff', 'anchor_ne_own', 'via_none_clears']


In [ ]:
KIND_LIST = ['near','far','north','south','east','west']

def _np(place, kind, anchor, rng):
    s = rng.choice(BANK[place])
    if kind: s += ' ' + rng.choice(REF[kind]).format(A=rng.choice(BANK[anchor]) if anchor else '')
    return s

def _wchoice(rng, idx, weights):
    return rng.choices(idx, weights=weights)[0] if sum(weights) > 0 else rng.choice(idx)

def generate_balanced(base, seed=123, verbose=True):
    rng = random.Random(seed)
    cnt = {h: collections.Counter(it['y'][h] for it in base) for h in HEADS}
    N = n0 = len(base); seen = {it['text'] for it in base}; skip = ('none', 'no')
    target = {}
    for h, labels in HEADS.items():
        if h in ('urgent', 'fragile'): continue
        big = max(cnt[h].get(i, 0) for i, l in enumerate(labels) if l not in skip)
        goal_n = max(BALANCE_FRAC * big, MIN_PER_CLASS.get(h, 0))
        target[h] = {i: goal_n for i, l in enumerate(labels) if l not in skip}
    def deficit(h, i):
        if h in ('urgent', 'fragile'): return (YES_RATIO * N - cnt[h].get(1, 0)) if i == 1 else 0
        return max(target[h].get(i, 0) - cnt[h].get(i, 0), 0)
    def total_def(): return sum(max(deficit(h, i), 0) for h, L in HEADS.items() for i in range(len(L)))
    def pick(h, allowed=None):
        L = HEADS[h]; idx = [i for i in range(len(L)) if L[i] not in skip and (allowed is None or L[i] in allowed)]
        return L[_wchoice(rng, idx, [max(deficit(h, i), 0) for i in idx])]
    def has(hs): return rng.random() < (0.75 if sum(max(deficit(h, i), 0) for h in hs for i in range(len(HEADS[h]))) > 0 else 0.3)
    def ref(kh, ah, own):
        if not has([kh, ah]): return None, None
        k = pick(kh, set(KIND_LIST))
        return k, (pick(ah, set(PLACES) - {own}) if k in ('near', 'far') else None)
    out, tries = [], 0
    cap = min(MAX_SYN, int(SYN_FRACTION * n0 / (1 - SYN_FRACTION)))
    while total_def() > 5 and len(out) < cap and tries < cap * 20:
        tries += 1
        goal = pick('goal')
        via  = pick('via', set(PLACES) - {goal}) if has(['via']) else None
        gk, ga = ref('gkind', 'ganchor', goal)
        vk, va = ref('vkind', 'vanchor', via) if via else (None, None)
        urg = rng.random() < (0.5 if deficit('urgent', 1) > 0 else 0.2)     # urgent / fragile độc lập nhau
        fra = rng.random() < (0.5 if deficit('fragile', 1) > 0 else 0.2)
        lab = dict(goal=goal, via=via, gkind=gk, ganchor=ga, vkind=vk, vanchor=va, urgent=urg, fragile=fra)
        txt = render(lab, rng)
        if txt in seen: continue
        seen.add(txt)
        y = mission_to_labels({'goal': goal, 'via': via, 'goal_ref': {'kind': gk, 'anchor': ga} if gk else None,
                               'via_ref': {'kind': vk, 'anchor': va} if vk else None, 'urgent': urg, 'fragile': fra})
        out.append({'text': txt, 'y': y, 'syn': True}); N += 1
        for h, i in y.items(): cnt[h][i] += 1
    if verbose: print(f"Đã sinh {len(out)} câu (trần {cap}); thiếu hụt còn lại = {total_def():.0f}")
    return out

# ---------- Template câu (theo cách nói thật trong train) ----------
ITEMS = ['gói hàng','hộp giấy','tập tài liệu','túi đồ giặt','quả bóng','bộ vợt','mũ bảo hiểm','hộp thuốc','phong bì công văn',
         'biển tên khách','chìa khóa xe','micro không dây','thùng nước uống','nguyên liệu nấu ăn','bưu kiện','tập bài kiểm tra',
         'túi sơ cứu','laptop','hộp cơm','chồng giáo trình',
         'sách tham khảo','hóa chất','dụng cụ tập luyện','thùng thực phẩm','sổ ghi khách','hộp khẩu trang','bộ đàm',
         'máy chiếu','chăn gối','khay cơm','tập giấy tờ','đơn hàng cần trả','đơn hàng','túi đồ']
WHO = ['Sinh viên','Giảng viên','Bác bảo vệ','Đội bóng','Huấn luyện viên','Cô thủ thư','Anh kỹ thuật viên','Khách mời']
assert not any(ALIAS_RE.search(fold(x)) for x in ITEMS + WHO), "tên hàng/người trùng alias địa điểm"
OPENERS = ['', '', '', 'Robot ơi, ', 'Nhờ bạn nhé: ', 'Chào robot, ', 'Yêu cầu mới: ', 'Nhắn robot: ', 'Xin chào! ',
           '[Đơn #{n}] ', 'Yêu cầu mới: [Đơn #{n}] ']
VERBS   = ['Giao','Đưa','Chuyển','Mang','Gửi','Vận chuyển','Đem']
TO      = ['đến','tới','qua']
PICK    = ['lấy','nhận','lấy giúp mình','nhận giúp mình']
CLOSE   = ['', '', 'Cảm ơn nhiều.', 'Cảm ơn!', 'Thanks robot.', 'Cảm ơn bạn nhé.']
URG_YES = ['Đang cần gấp lắm.','Không được chậm trễ.','Ưu tiên số một, đi ngay.','Càng nhanh càng tốt.','Hỏa tốc!','Gấp nhé!',
           'Việc này khẩn cấp.','Khẩn cấp nhé.','Cần ngay trong {m} phút.']
URG_NO  = ['Không gấp đâu.','Cứ từ từ, không vội.','Chiều nay giao cũng được.','Không cần vội.']
FRA_YES = ['Nhẹ tay vì đồ rất dễ vỡ.','Hàng dễ vỡ, đi cẩn thận.','Bên trong là đồ thủy tinh.','Đồ gốm bên trong, tránh va chạm.',
           'Chú ý: hàng dễ hỏng khi va đập.','Kiện hàng mỏng manh, đi nhẹ thôi.']
FRA_NO  = ['Hàng chắc chắn, không dễ vỡ.','Đồ không vỡ được đâu.','Rơi cũng chẳng sao.']
TRAP    = ['Hôm qua đã giao ở {t} rồi.','Người nhận đã rời {t} rồi.','Đừng nhầm với {t} nhé.','Không cần ghé {t}.','Bỏ qua {t}, không phải ở đó.']
REF = {'near':  ['ở gần {A}','nằm gần {A}','gần {A} hơn','sát {A}','cạnh {A}'],
       'far':   ['ở xa {A}','nằm xa {A}','xa {A} hơn','cách xa {A}'],
       'north': ['nằm phía bắc','phía bắc','ở phía bắc','ở phía trên bản đồ','phía bắc bản đồ','ở màn trên'],
       'south': ['nằm phía nam','phía nam','ở phía nam','ở phía dưới bản đồ','phía nam bản đồ','ở màn dưới'],
       'east':  ['nằm phía đông','phía đông','ở phía đông','ở bên phải bản đồ'],
       'west':  ['nằm phía tây','phía tây','ở phía tây','ở bên trái bản đồ']}   # "bên trái" mình suy ra từ "bên phải", bạn kiểm lại

def render(lab, rng):
    G = _np(lab['goal'], lab['gkind'], lab['ganchor'], rng)
    V = _np(lab['via'], lab['vkind'], lab['vanchor'], rng) if lab['via'] else None
    i1, i2, v, to = rng.choice(ITEMS), rng.choice(ITEMS), rng.choice(VERBS), rng.choice(TO)
    gs = rng.choice([                                   # câu nói về đích
        f"{v} {i2} {to} {G}.", f"{v} {i2} {to} {G} giúp mình.", f"Nhờ robot {v.lower()} {i2} {to} {G}.",
        f"Điểm giao: {G}. Hàng cần giao là {i2}.", f"Có đơn giao {i2} ở {G}.", f"Nơi nhận hàng là {G}, mang {i2} qua giúp mình.",
        f"{rng.choice(WHO)} ở {G} đang chờ {i2}."])
    if V:
        core = rng.choice([
            f"Trước tiên qua {V} {rng.choice(PICK)} {i1}, sau đó {v.lower()} {i2} {to} {G}.",
            f"Ghé {V} {rng.choice(PICK)} {i1} trước, rồi {v.lower()} {i2} {to} {G} giúp mình.",
            f"Trước khi {v.lower()} {i2} {to} {G}, nhớ ghé {V} {rng.choice(PICK)} {i1}.",
            f"Lấy {i1} ở {V} xong thì {v.lower()} {i2} {to} {G}.",
            f"Tạt qua {V} {rng.choice(PICK)} {i1}, tiếp theo {v.lower()} {i2} {to} {G}.",
            f"{gs} Nhớ ghé {V} lấy {i1} trước.",
            f"{gs} Nhưng phải tạt qua {V} trước đã."])
    else:
        core = gs
    pre, post = [], []
    def add(s): (pre if rng.random() < 0.3 else post).append(s)
    if lab['urgent']: add(rng.choice(URG_YES).replace('{m}', str(rng.choice([2, 5, 10, 15]))))
    elif rng.random() < 0.25: add(rng.choice(URG_NO))
    if lab['fragile']: add(rng.choice(FRA_YES))
    elif rng.random() < 0.25: add(rng.choice(FRA_NO))
    used = {lab[k] for k in ('goal', 'via', 'ganchor', 'vanchor') if lab[k]}
    free = [p for p in PLACES if p not in used]
    if free and rng.random() < 0.3: add(rng.choice(TRAP).format(t=rng.choice(BANK[rng.choice(free)])))
    rng.shuffle(pre); rng.shuffle(post)
    op = rng.choice(OPENERS).replace('{n}', str(rng.randint(1000, 9999)))
    body = ' '.join(pre + [core] + post + [rng.choice(CLOSE)])
    if op.rstrip().endswith(',') or (op.rstrip().endswith(':') and rng.random() < 0.5): body = body[0].lower() + body[1:]
    txt = re.sub(r'\s+', ' ', op + body).strip()
    return txt if op.rstrip().endswith((',', ':')) else txt[0].upper() + txt[1:]

syn_items = generate_balanced(base_items)
for it in random.Random(3).sample(syn_items, 6):
    print('\n', it['text']); print('   ', {h: HEADS[h][i] for h, i in it['y'].items() if HEADS[h][i] not in ('none', 'no')})

Đã sinh 6000 câu (trần 6000); thiếu hụt còn lại = 1241

 Robot ơi, có đơn giao nguyên liệu nấu ăn ở bãi đậu xe nằm xa khu giảng đường. Nhẹ tay vì đồ rất dễ vỡ. Gấp nhé! Cảm ơn bạn nhé.
    {'goal': 'parking', 'gkind': 'far', 'ganchor': 'lecture', 'urgent': 'yes', 'fragile': 'yes'}

 Nhờ bạn nhé: Nhẹ tay vì đồ rất dễ vỡ. Vận chuyển bưu kiện qua nơi dùng bữa ở phía đông giúp mình. Cứ từ từ, không vội. Cảm ơn bạn nhé.
    {'goal': 'canteen', 'gkind': 'east', 'fragile': 'yes'}

 Yêu cầu mới: [Đơn #5902] Khẩn cấp nhé. Có đơn giao hộp thuốc ở tòa nhà ở sinh viên ở phía tây. Đồ không vỡ được đâu. Hôm qua đã giao ở chốt bảo vệ cổng rồi.
    {'goal': 'dorm', 'gkind': 'west', 'urgent': 'yes'}

 Trước tiên qua lab hóa gần khu thể dục hơn lấy giúp mình hộp khẩu trang, sau đó mang thùng nước uống qua khu hành chính ở phía dưới bản đồ. Hôm qua đã giao ở nhà ở sinh viên rồi. Hàng dễ vỡ, đi cẩn thận. Cảm ơn bạn nhé.
    {'goal': 'office', 'via': 'lab', 'gkind': 'south', 'vkind': 'near', 'vanchor': 'sp

In [ ]:
# ---------- Sinh dữ liệu NGẮN + PHỦ ĐỊNH ----------
NEG_N        = 1500          # số mission gốc; mỗi mission = 1 câu có dấu + 1 câu không dấu (đúng 1:1)
P_GREF, P_VIA, P_VREF, P_NEG_REF, P_OPENER = 0.35, 0.40, 0.35, 0.25, 0.30
P_MINI       = 0.60          # tỉ lệ câu RẤT NGẮN (1 mệnh đề đích [+ via ngắn] + tối đa 1 phủ định, gần như không có cờ)
P_LOWER      = 0.30          # xác suất viết thường toàn câu
SEP_W        = [('. ', 0.50), (', ', 0.20), (' ', 0.17), ('; ', 0.05), (' - ', 0.04), ('! ', 0.04)]   # dấu nối giữa các mệnh đề
NEG_COUNT_W  = [(0, 0.10), (1, 0.50), (2, 0.32), (3, 0.08)]                                          # số cụm phủ định / mission (câu thường)

REF2 = {k: list(v) for k, v in REF.items()}
if TRAIN_ON_ALL:             # cách nói chỉ thấy ở val -> chỉ đưa vào khi gộp val
    REF2['north'] += ['ở mạn trên', 'mạn trên']; REF2['south'] += ['ở mạn dưới', 'mạn dưới']   # "mạn dưới" mình suy ra

NEG_T = ['Đừng nhầm với {t} nhé', 'Không cần ghé {t}', 'Bỏ qua {t}, không phải ở đó', '{T} không phải điểm nhận',
         '{T} không phải điểm giao', 'Không phải {t}', 'Tránh {t} giúp mình', 'Khỏi qua {t}', 'Không giao ở {t}',
         'Đừng đến {t}', 'Không lấy hàng ở {t}', 'Nhầm rồi, không phải {t}', 'Hôm qua đã giao ở {t} rồi',
         'Người nhận không ở {t}', 'Đừng ghé {t}, không có hàng ở đó', '{T} thì khỏi', 'Không đến {t} đâu',
         'Không qua {t}', 'Bỏ qua {t}', 'Đừng đưa tới {t}', 'Không nhận hàng ở {t}', 'Người nhận đã rời {t} rồi']
GOAL_T = ['{v} {i} {to} {G}', '{v} {i} {to} {G} giúp mình', 'Điểm giao: {G}', 'Điểm nhận là {G}', 'Giao ở {G}',
          'Nơi nhận hàng là {G}', 'Có đơn giao {i} ở {G}', '{WHO} ở {G} đang chờ {i}', 'Hàng {i} giao tới {G}',
          'Nhờ robot {vl} {i} {to} {G}', 'Cần {i} ở {G}', 'FIELD']
VIA_T  = ['Ghé {V} trước', 'Ghé {V} lấy {i} trước', 'Qua {V} nhận {i} trước', 'Nhớ tạt qua {V} trước', 'Lấy {i} ở {V} trước',
          'Trước tiên qua {V}', 'Phải ghé {V} trước đã', 'Tạt qua {V} {pick} {i} trước']
MERGED_T = ['Ghé {V} trước rồi {vl} {i} {to} {G}', 'Trước tiên qua {V} {pick} {i2}, sau đó {vl} {i} {to} {G}',
            'Lấy {i2} ở {V} xong thì {vl} {i} {to} {G}', 'Trước khi {vl} {i} {to} {G}, ghé {V} {pick} {i2}']

MINI_GOAL = ['{v} {i} {to} {G}', 'Điểm giao: {G}', 'Điểm nhận là {G}', 'Giao ở {G}', 'Nhờ robot {vl} {i} {to} {G}']
MINI_VIA  = ['Ghé {V} trước', 'Qua {V} trước', 'Trước tiên qua {V}', 'Phải ghé {V} trước đã']
MINI_MERGED = ['Ghé {V} trước rồi {vl} {i} {to} {G}', 'Qua {V} trước, sau đó {vl} {i} {to} {G}']

class Balancer:              # luôn chọn phương án đang được dùng ÍT NHẤT -> các class/từ đồng nghĩa/mẫu câu đều nhau
    def __init__(self, rng): self.rng, self.c = rng, collections.Counter()
    def pick(self, key, options):
        options = list(options); m = min(self.c[(key, o)] for o in options)
        o = self.rng.choice([o for o in options if self.c[(key, o)] == m]); self.c[(key, o)] += 1; return o

def generate_neg(n, seed=777):
    rng = random.Random(seed); B = Balancer(rng); out, seen = [], {it['text'] for it in base_items + syn_items + val_items}
    wch = lambda ws: rng.choices([w for w, _ in ws], [p for _, p in ws])[0]
    def noun(cls, kind=None, anchor=None):
        s = B.pick(('alias', cls), BANK[cls])
        if kind: s += ' ' + B.pick(('ref', kind), REF2[kind]).format(A=B.pick(('alias', anchor), BANK[anchor]) if anchor else '')
        return s
    tries = 0
    while len(out) < n and tries < n * 30:
        tries += 1
        goal = B.pick('goal', PLACES)
        gk = B.pick('gkind', KIND_LIST) if rng.random() < P_GREF else None
        ga = B.pick('ganchor', [p for p in PLACES if p != goal]) if gk in ('near', 'far') else None
        via = B.pick('via', [p for p in PLACES if p != goal]) if rng.random() < P_VIA else None
        vk = B.pick('vkind', KIND_LIST) if via and rng.random() < P_VREF else None
        va = B.pick('vanchor', [p for p in PLACES if p != via]) if vk in ('near', 'far') else None
        used = {x for x in (goal, ga, via, va) if x}
        mini = rng.random() < P_MINI
        negs = []
        for _ in range(wch([(0, 0.15), (1, 0.85)]) if mini else wch(NEG_COUNT_W)):
            cand = [p for p in PLACES if p not in used and p not in [c for c, *_ in negs]]
            if len(cand) < 2: break
            c = B.pick('neg', cand); k = a = None
            if rng.random() < P_NEG_REF:                       # bẫy khó: cụm bị phủ định kèm hướng/gần/xa, vẫn KHÔNG gán nhãn
                k = B.pick('nkind', KIND_LIST)
                if k in ('near', 'far'):
                    free = [p for p in cand if p != c]; a = B.pick('nanchor', free) if free else None
                    if a is None: k = None
            negs.append((c, k, a))
        i1, i2, v = B.pick('item', ITEMS), B.pick('item', ITEMS), B.pick('verb', VERBS)
        fmt = dict(v=v, vl=v.lower(), i=i1, i2=i2, to=B.pick('to', TO), pick=rng.choice(PICK), WHO=rng.choice(WHO))
        G = noun(goal, gk, ga); fmt['G'] = G
        cl = []
        gt = B.pick('goal_t_mini' if mini else 'goal_t', MINI_GOAL if mini else GOAL_T)
        if via and rng.random() < (0.5 if mini else 0.3):       # mệnh đề gộp: ghé via rồi giao tới goal
            cl.append(B.pick('merged_t', MINI_MERGED if mini else MERGED_T).format(V=noun(via, vk, va), **fmt))
        else:
            if gt == 'FIELD': cl += rng.sample([f"Điểm giao: {G}", f"Hàng: {i1}"], 2)
            else: cl.append(gt.format(**fmt))
            if via: cl.append(B.pick('via_t_mini' if mini else 'via_t', MINI_VIA if mini else VIA_T).format(V=noun(via, vk, va), **fmt))
        for c, k, a in negs:
            t = (rng.choice(TRAIN_ALIASES[c]) if (not k and rng.random() < globals().get('P_NEG_CORE', 0.0)) else noun(c, k, a)); cl.append(B.pick('neg_t', NEG_T).format(t=t, T=t[0].upper() + t[1:]))
        pf = 0.08 if mini else 0.45                             # xác suất có cờ urgent / fragile
        urg, fra = rng.random() < pf, rng.random() < pf
        if urg: cl.append(rng.choice(URG_YES).replace('{m}', str(rng.choice([2, 5, 10, 15]))).rstrip('.!'))
        elif not mini and rng.random() < 0.15: cl.append(rng.choice(URG_NO).rstrip('.'))
        if fra: cl.append(rng.choice(FRA_YES).rstrip('.'))
        elif not mini and rng.random() < 0.15: cl.append(rng.choice(FRA_NO).rstrip('.'))
        rng.shuffle(cl)
        sep = wch(SEP_W); parts = [cl[0][0].upper() + cl[0][1:]]
        for c in cl[1:]:
            s_ = sep if rng.random() > 0.1 else wch(SEP_W)       # đôi khi đổi dấu nối giữa chừng
            parts += [s_, c[0].upper() + c[1:] if s_ in ('. ', '! ') else c[0].lower() + c[1:]]
        txt = ''.join(parts)
        if rng.random() < 0.9 or sep in ('. ', '! '): txt += '.'
        if not mini and rng.random() < P_OPENER:
            op = rng.choice([o for o in OPENERS if o]).replace('{n}', str(rng.randint(1000, 9999)))
            txt = op + (txt[0].lower() + txt[1:] if op.rstrip().endswith(',') else txt)
        if not mini and rng.random() < 0.15: txt += ' ' + rng.choice([c for c in CLOSE if c])
        txt = re.sub(r'\s+', ' ', txt).strip()
        if rng.random() < P_LOWER: txt = txt.lower()
        na = strip_accents(txt)
        if na == txt or txt in seen: continue
        seen.add(txt)
        y = mission_to_labels({'goal': goal, 'via': via, 'goal_ref': {'kind': gk, 'anchor': ga} if gk else None,
                               'via_ref': {'kind': vk, 'anchor': va} if vk else None, 'urgent': urg, 'fragile': fra})
        out.append({'text': txt, 'text_na': na, 'y': y, 'neg_cls': [c for c, *_ in negs], 'syn': True})
    print(f"Đã sinh {len(out)} mission ngắn = {len(out)} câu có dấu + {len(out)} câu không dấu"); return out, B

neg_items, _B = generate_neg(NEG_N)

# --- kiểm tra tự động ---
bad = 0
for it in neg_items:
    f = fold(it['text'])
    for h in ('goal', 'via'):
        l = HEADS[h][it['y'][h]]
        if l != 'none' and not any(fold(a) in f for a in BANK[l]): bad += 1
    for c in it['neg_cls']:
        if not any(fold(a) in f for a in BANK[c]): bad += 1
        if any(HEADS[h][it['y'][h]] == c for h in ('goal', 'via', 'ganchor', 'vanchor')): bad += 1   # cụm bị phủ định tuyệt đối không được có nhãn
print("câu lỗi nhãn/phủ định:", bad, "/", len(neg_items))
print("có phủ định:", sum(bool(it['neg_cls']) for it in neg_items), "| có via:", sum(it['y']['via'] > 0 for it in neg_items),
      "| có goal_ref:", sum(it['y']['gkind'] > 0 for it in neg_items), "| có via_ref:", sum(it['y']['vkind'] > 0 for it in neg_items))
al = [v for (key, o), v in _B.c.items() if isinstance(key, tuple) and key[0] == 'alias']
print("số lần dùng mỗi từ đồng nghĩa: min", min(al), "| max", max(al))
for it in random.Random(5).sample(neg_items, 10):
    print('\n  CÓ DẤU   :', it['text']); print('  KHÔNG DẤU:', it['text_na'])
    print('  nhãn     :', {h: HEADS[h][i] for h, i in it['y'].items() if HEADS[h][i] not in ('none', 'no')}, '| phủ định:', it['neg_cls'])

Đã sinh 1500 mission ngắn = 1500 câu có dấu + 1500 câu không dấu
câu lỗi nhãn/phủ định: 0 / 1500
có phủ định: 1298 | có via: 590 | có goal_ref: 511 | có via_ref: 190
số lần dùng mỗi từ đồng nghĩa: min 23 | max 27

  CÓ DẤU   : Điểm nhận là sân thi đấu, không lấy hàng ở nơi bán cơm.
  KHÔNG DẤU: Diem nhan la san thi dau, khong lay hang o noi ban com.
  nhãn     : {'goal': 'sports'} | phủ định: ['canteen']

  CÓ DẤU   : giao ở giảng đường lớn qua khu văn phòng phía bắc bản đồ trước. đừng đến khu chăm sóc sức khỏe gần lối ra vào trường hơn.
  KHÔNG DẤU: giao o giang duong lon qua khu van phong phia bac ban do truoc. dung den khu cham soc suc khoe gan loi ra vao truong hon.
  nhãn     : {'goal': 'lecture', 'via': 'office', 'vkind': 'north'} | phủ định: ['clinic']

  CÓ DẤU   : Đừng đến phòng nghiên cứu. Huấn luyện viên ở cổng ra vào đang chờ hộp khẩu trang. Không đến giảng đường chính ở màn trên đâu. Qua phòng đọc nằm xa khu bán đồ ăn nhận hộp khẩu trang trước.
  KHÔNG DẤU: Dung den phong 

In [ ]:
# ---------- Ghép dữ liệu train ----------
train_items = base_items + syn_items + neg_items
if TRAIN_ON_ALL: train_items = train_items + val_items
n_real = sum(not it.get('syn') for it in train_items)
print(f"train: {len(train_items)} câu | thật={n_real} | sinh thêm={len(train_items) - n_real}")
dist_report(train_items, 'TRAIN cuối')

train: 9500 câu | thật=2000 | sinh thêm=7500

===== PHÂN PHỐI CLASS — TRAIN cuối (n=9500) =====

[goal]  max/min (bỏ none, bỏ class 0 mẫu) = 1.1x
  canteen    961  10.1% ####
  clinic     930   9.8% ###
  dorm       946  10.0% ###
  gate       902   9.5% ###
  lab        936   9.9% ###
  lecture    955  10.1% ####
  library    948  10.0% ###
  office     975  10.3% ####
  parking    991  10.4% ####
  sports     956  10.1% ####

[via]  max/min (bỏ none, bỏ class 0 mẫu) = 1.0x
  none      5650  59.5% #######################
  canteen    391   4.1% #
  clinic     378   4.0% #
  dorm       383   4.0% #
  gate       384   4.0% #
  lab        380   4.0% #
  lecture    389   4.1% #
  library    389   4.1% #
  office     393   4.1% #
  parking    381   4.0% #
  sports     382   4.0% #

[gkind]  max/min (bỏ none, bỏ class 0 mẫu) = 1.1x
  none      4440  46.7% ##################
  near       836   8.8% ###
  far        860   9.1% ###
  north      849   8.9% ###
  south      811   8.5% ###
  east

In [ ]:
# ---------- Giải mã có ràng buộc + độ đo (thuần numpy) ----------
CORE_HEADS = ('goal', 'via', 'urgent', 'fragile')
NP_ = len(PLACES); NF_IDX = [H2I['gkind']['near'], H2I['gkind']['far']]

def decode(P, rules=None):
    # P: dict head -> xác suất (N, C). Chọn tổ hợp nhãn HỢP LỆ có tổng log-xác suất lớn nhất.
    rules = RULES if rules is None else rules
    L = {h: np.log(np.clip(np.asarray(P[h], dtype=np.float64), 1e-9, 1.0)) for h in HEADS}
    N = len(L['goal']); out = {h: np.zeros(N, dtype=int) for h in HEADS}
    out['urgent'], out['fragile'] = L['urgent'].argmax(1), L['fragile'].argmax(1)

    def best_ref(lk, la, own):
        best = (-1e18, 0, 0)
        for k in range(len(KINDS)):
            if rules['anchor_iff']:
                if k in NF_IDX:
                    cands = [a for a in range(1, NP_ + 1) if not (rules['anchor_ne_own'] and a == own)]
                    a = max(cands, key=lambda j: la[j])
                else: a = 0
            else: a = int(la.argmax())
            s = lk[k] + la[a]
            if s > best[0]: best = (s, k, a)
        return best

    for i in range(N):
        G = [best_ref(L['gkind'][i], L['ganchor'][i], g + 1) for g in range(NP_)]
        V = [(L['vkind'][i][0] + L['vanchor'][i][0], 0, 0) if rules['via_none_clears']
             else best_ref(L['vkind'][i], L['vanchor'][i], 0)]
        V += [best_ref(L['vkind'][i], L['vanchor'][i], v) for v in range(1, NP_ + 1)]
        best = (-1e18, 0, 0)
        for g in range(NP_):
            for v in range(NP_ + 1):
                if rules['goal_ne_via'] and v == g + 1: continue
                s = L['goal'][i][g] + L['via'][i][v] + G[g][0] + V[v][0]
                if s > best[0]: best = (s, g, v)
        _, g, v = best
        out['goal'][i], out['via'][i] = g, v
        out['gkind'][i], out['ganchor'][i] = G[g][1], G[g][2]
        out['vkind'][i], out['vanchor'][i] = V[v][1], V[v][2]
    return out

def argmax_decode(P): return {h: np.asarray(P[h]).argmax(1) for h in HEADS}

def gold_of(items): return {h: np.array([it['y'][h] for it in items]) for h in HEADS}

def exact_masks(pred, gold):
    core = np.all([pred[h] == gold[h] for h in CORE_HEADS], 0)
    full = np.all([pred[h] == gold[h] for h in HEADS], 0)
    return core, full

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - m) / d, (c + m) / d)

def violations(pred):
    # đếm số câu có dự đoán vô lý về cấu trúc (dù chưa chắc sai nhãn)
    N = len(pred['goal']); v = collections.Counter()
    for i in range(N):
        g, vi = HEADS['goal'][pred['goal'][i]], HEADS['via'][pred['via'][i]]
        if vi != 'none' and vi == g: v['goal=via'] += 1
        for k, a, own in (('gkind', 'ganchor', g), ('vkind', 'vanchor', vi)):
            kk, aa = HEADS[k][pred[k][i]], HEADS[a][pred[a][i]]
            if (kk in KINDS_NF) != (aa != 'none'): v['anchor/kind lệch'] += 1
            if aa != 'none' and aa == own: v['anchor=chính nó'] += 1
        if vi == 'none' and (HEADS['vkind'][pred['vkind'][i]] != 'none' or HEADS['vanchor'][pred['vanchor'][i]] != 'none'): v['via_ref khi không có via'] += 1
    return v

# ---------- Các góc nhìn của val (nhiễu cố định seed, tạo 1 lần) ----------
_rv = random.Random(2024); _vt = [it['text'] for it in val_items]
VAL_VIEWS = {
    'gốc':           _vt,
    'bỏ dấu':        [strip_accents(t) for t in _vt],
    'typo':          [multi_typo(t, _rv, 0.08) for t in _vt],
    'bỏ dấu+typo':   [multi_typo(strip_accents(t), _rv, 0.10) for t in _vt],
    'nặng':          [merge_or_split(multi_typo(strip_accents(t), _rv, 0.15, 6), _rv, 0.08) for t in _vt],
}
SEL_VIEWS = ['gốc', 'bỏ dấu', 'typo', 'bỏ dấu+typo']      # 'nặng' chỉ để báo cáo, không dùng chọn checkpoint
GOLD_VAL = gold_of(val_items)
print({k: len(v) for k, v in VAL_VIEWS.items()})
print("Ví dụ view 'nặng':", VAL_VIEWS['nặng'][0])

{'gốc': 300, 'bỏ dấu': 300, 'typo': 300, 'bỏ dấu+typo': 300, 'nặng': 300}
Ví dụ view 'nặng': Robot oi, khong can hge day phong joc. Bep truong o cang tin man tren dang cgo hop giay. Dung nham voi phong lab nhe. Do khong vo douc dau. Dang can gap lam.


In [ ]:
# ---------- Tokenizer BamiBERT (nhận chữ thô, KHÔNG dùng pyvi) ----------
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
print("vocab:", len(tok), "| special:", tok.cls_token, tok.sep_token, tok.pad_token)
print("Ví dụ:", tok.tokenize("Giao hang oi, khong can ghe phong thi nghiem")[:20])

# kiểm tra có câu nào bị CẮT khi MAX_LEN không (cắt mất đuôi = mất cờ urgent/fragile)
def tok_len(texts): return np.array([len(tok(t, truncation=False)['input_ids']) for t in texts])
for name in ('gốc', 'bỏ dấu', 'nặng'):
    L_ = tok_len(VAL_VIEWS[name]); print(f"val[{name:7s}] token: max={L_.max()} p99={int(np.percentile(L_, 99))} | bị cắt (> {MAX_LEN}): {(L_ > MAX_LEN).sum()}")
L_ = tok_len([it['text'] for it in train_items[:1500]]); print(f"train mẫu: max={L_.max()} | bị cắt: {(L_ > MAX_LEN).sum()}")

vocab: 20481 | special: <s> </s> <pad>
Ví dụ: ['Giao', 'Ġhang', 'Ġo', 'i', ',', 'Ġkh', 'ong', 'Ġcan', 'Ġghe', 'Ġphong', 'Ġthi', 'Ġnghi', 'em']
val[gốc    ] token: max=94 p99=78 | bị cắt (> 192): 0
val[bỏ dấu ] token: max=94 p99=80 | bị cắt (> 192): 0
val[nặng   ] token: max=98 p99=86 | bị cắt (> 192): 0
train mẫu: max=72 | bị cắt: 0


In [ ]:
# ---------- Dataset ----------
class TrainDS(Dataset):
    def __init__(self, items):
        self.items = items; self.index = []; self.noise_scale = 1.0
        for j, it in enumerate(items): self.index += [j] * (1 if it.get('syn') else REAL_REPEAT)
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        it = self.items[self.index[i]]; t = it['text']
        if random.random() > P_CLEAN:
            for _ in range(3):
                t2 = augment_v2(it['text'], it['y'], random, self.noise_scale)
                if t2 != it['text']: t = t2; break
        b = strip_accents(t) if (has_accent(t) and random.random() < P_PAIR) else None
        return {'a': t, 'b': b, 'y': it['y']}

def collate(batch):
    enc = tok([x['a'] for x in batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    y = {h: torch.tensor([x['y'][h] for x in batch]) for h in HEADS}
    pidx = [i for i, x in enumerate(batch) if x['b'] is not None]
    if len(pidx) >= 2:
        eb = tok([batch[i]['b'] for i in pidx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        return enc['input_ids'], enc['attention_mask'], y, torch.tensor(pidx), eb['input_ids'], eb['attention_mask']
    return enc['input_ids'], enc['attention_mask'], y, None, None, None

_ds = TrainDS(train_items); print("mẫu / epoch:", len(_ds), "| bước / epoch:", math.ceil(len(_ds) / BATCH_SIZE))
for k in range(4): print(' ', _ds[k * 997 % len(_ds)]['a'])

mẫu / epoch: 11500 | bước / epoch: 360
  Không phải hu nội trú. xin chào! đội bó ng ở phòng tập thể thao đang chờ thùng hàng.
  Nhờ robot chuyển quả bóng qua nhà thể chất. Không phải nơi làm thủ tục. Nhớ ghé xưởng thực hành phía đông lấy linh kiện điện tử trước. Hàng chắc chắn, không dễ vỡ.
  Giao hộp giấy đến khu đọc sách giúp mình. Càng nhanh càng tốt. Chú ý: hàng dễ hỏng khi va đập. Cảm ơn! Đừng hnầm với sân thi đấu nhé.
  Yêu cầu mới: Có đơn giao túi sơ cứu ở phòng khám của trường. Người nhận đã rời căn tin rồi. Cảm ơn!


In [ ]:
# ---------- Model: BamiBERT + 8 head (CLS + attention pooling) ----------
class MultiHeadNLU(nn.Module):
    def __init__(self, name, extra_pools=3):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        hidden = self.enc.config.hidden_size
        self.extra_pools = extra_pools
        self.drop = nn.Dropout(0.2)
        self.pool_q = nn.ModuleDict({
            head: nn.Sequential(nn.Linear(hidden, hidden // 2), nn.Tanh(), nn.Linear(hidden // 2, 1))
            for head in HEADS
        })
        self.heads = nn.ModuleDict({head: nn.Linear(hidden * 2, len(labels)) for head, labels in HEADS.items()})
        self.extra_pool_q = nn.ModuleDict({head: nn.Linear(hidden, extra_pools, bias=False) for head in HEADS})
        extra_size = hidden * (extra_pools + 1)
        self.extra_norm = nn.ModuleDict({head: nn.LayerNorm(extra_size) for head in HEADS})
        self.extra_heads = nn.ModuleDict({head: nn.Linear(extra_size, len(labels)) for head, labels in HEADS.items()})
        for head in HEADS:
            nn.init.zeros_(self.pool_q[head][2].weight)
            nn.init.zeros_(self.pool_q[head][2].bias)
            nn.init.normal_(self.extra_pool_q[head].weight, mean=0.0, std=0.02)
            nn.init.zeros_(self.extra_heads[head].weight)
            nn.init.zeros_(self.extra_heads[head].bias)

    def forward(self, ids, mask, return_emb=False):
        hidden = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        cls = hidden[:, 0, :]; logits = {}; token_mask = mask.unsqueeze(-1) == 0
        for head in HEADS:
            weight = self.pool_q[head](hidden).squeeze(-1).float().masked_fill(mask == 0, -1e4).softmax(-1).to(hidden.dtype)
            pooled = (weight.unsqueeze(-1) * hidden).sum(1)
            old_logits = self.heads[head](self.drop(torch.cat([cls, pooled], -1)))

            extra_score = self.extra_pool_q[head](hidden).float().masked_fill(token_mask, -1e4)
            extra_weight = extra_score.softmax(dim=1).to(hidden.dtype)
            extra_pool = torch.einsum('btk,bth->bkh', extra_weight, hidden).flatten(1)
            extra_feature = self.extra_norm[head](torch.cat([cls, extra_pool], -1))
            logits[head] = old_logits + self.extra_heads[head](self.drop(extra_feature))

        if not return_emb: return logits
        mask3 = mask.unsqueeze(-1).to(hidden.dtype)
        mean = (hidden * mask3).sum(1) / mask3.sum(1).clamp(min=1)
        emb = F.normalize(F.normalize(cls.float(), dim=-1) + F.normalize(mean.float(), dim=-1), dim=-1)
        return logits, emb

ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
def compute_loss(lg, y): return sum(HEAD_W[h] * ce(lg[h].float(), y[h].to(device)) for h in HEADS)

class EMA:
    def __init__(self, model, decay):
        self.decay, self.step = decay, 0
        self.shadow = {n: p.detach().clone().float() for n, p in model.named_parameters()}
        self.backup = None
    @torch.no_grad()
    def update(self, model):
        self.step += 1; d = min(self.decay, (1 + self.step) / (10 + self.step))
        for n, p in model.named_parameters(): self.shadow[n].mul_(d).add_(p.detach().float(), alpha=1 - d)
    @torch.no_grad()
    def apply(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters()}
        for n, p in model.named_parameters(): p.copy_(self.shadow[n].to(p.dtype))
    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters(): p.copy_(self.backup[n])
        self.backup = None

@torch.no_grad()
def predict_probs(model, texts, bs=64):
    model.eval(); texts = list(texts)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i])); chunks = {h: [] for h in HEADS}
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            lg = model(enc['input_ids'], enc['attention_mask'])
        for h in HEADS: chunks[h].append(lg[h].float().softmax(-1).cpu().numpy())
    inv = np.empty(len(order), dtype=int); inv[np.array(order)] = np.arange(len(order))
    return {h: np.concatenate(chunks[h])[inv] for h in HEADS}

_BRACKET = re.compile(r'\[[^\]]*\]')
def tta_views(t):
    base = re.sub(r'\s+', ' ', _BRACKET.sub(' ', t)).strip() or t      # bỏ cụm [Đơn #1234]
    return [t, base, strip_accents(base)]
TTA_W = (2.0, 1.0, 1.0)
def predict_probs_tta(model, texts):
    acc = None
    for vi, w in enumerate(TTA_W):
        P = predict_probs(model, [tta_views(t)[vi] for t in texts])
        acc = {h: w * P[h] for h in HEADS} if acc is None else {h: acc[h] + w * P[h] for h in HEADS}
    return {h: v / sum(TTA_W) for h, v in acc.items()}

def eval_views(model, names=None, constrained=True, tta=False):
    res = {}
    for name, texts in VAL_VIEWS.items():
        if names is not None and name not in names: continue
        P = predict_probs_tta(model, texts) if tta else predict_probs(model, texts)
        pred = decode(P) if constrained else argmax_decode(P)
        core, full = exact_masks(pred, GOLD_VAL)
        res[name] = dict(core=float(core.mean()), full=float(full.mean()), ok_full=full, ok_core=core, P=P, pred=pred)
    return res

def sel_score(res, names=SEL_VIEWS): return float(np.mean([0.5 * (res[n]['core'] + res[n]['full']) for n in names]))

In [ ]:
# ---------- TRAIN ----------
CKPT       = "/content/bamibert_nlu_ft711.pt"
DRIVE_CKPT = "/content/drive/MyDrive/bamibert_nlu_ft711.pt"

def save_ckpt(model):
    torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, CKPT)
    try: shutil.copy(CKPT, DRIVE_CKPT); print('    -> đã lưu local & Drive')
    except Exception as e: print(f'    -> đã lưu local (Drive lỗi: {e})')

def run_training(items, epochs, select=True):
    set_seed(SEED)
    model = MultiHeadNLU(MODEL_NAME).to(device)
    print(f"Tham số: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
    ds = TrainDS(items); loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    enc_p  = [p for k, p in model.named_parameters() if k.startswith('enc.')]
    head_p = [p for k, p in model.named_parameters() if not k.startswith('enc.')]
    opt = torch.optim.AdamW([{'params': enc_p, 'lr': LR_ENC}, {'params': head_p, 'lr': LR_HEAD}], weight_decay=WD)
    total = len(loader) * epochs; sched = get_linear_schedule_with_warmup(opt, int(WARMUP_FRAC * total), total)
    use_scaler = USE_AMP and AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
    ema = EMA(model, EMA_DECAY); best, best_ep, bad = -1.0, 0, 0
    for ep in range(1, epochs + 1):
        ds.noise_scale = NOISE_HI if ep <= 0.7 * epochs else NOISE_LO
        model.train(); run = run_al = 0.0
        for ids, mask, y, pidx, ib, mb in tqdm(loader, desc=f"ep{ep}", leave=False):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                lg_a, ea = model(ids.to(device), mask.to(device), return_emb=True)
                loss = compute_loss(lg_a, y); al = torch.zeros((), device=device)
                if pidx is not None:
                    lg_b, eb = model(ib.to(device), mb.to(device), return_emb=True)
                    loss = loss + compute_loss(lg_b, {h: y[h][pidx] for h in HEADS})      # câu bỏ dấu: cùng nhãn
                    al = (1 - (ea[pidx.to(device)] * eb).sum(-1)).mean()                  # kéo embedding có dấu <-> không dấu lại gần nhau
                    loss = loss + ALIGN_W * al
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if use_scaler:
                sb = scaler.get_scale(); scaler.step(opt); scaler.update()
                if scaler.get_scale() >= sb: sched.step()
            else:
                opt.step(); sched.step()
            ema.update(model); run += loss.item(); run_al += float(al)
        msg = f"ep{ep:02d} loss={run / len(loader):.3f} align={run_al / len(loader):.3f}"
        if select:
            ema.apply(model); res = eval_views(model, SEL_VIEWS); sc = sel_score(res)
            msg += f" | score={sc:.4f} | " + ' '.join(f"{n}:{res[n]['full']:.3f}" for n in SEL_VIEWS)
            if sc > best + 1e-4:
                best, best_ep, bad = sc, ep, 0; save_ckpt(model); msg += '  *'
            else: bad += 1
            ema.restore(model); print(msg)
            if bad >= PATIENCE: print(f"Dừng sớm: {PATIENCE} epoch không cải thiện. Epoch tốt nhất = {best_ep}"); break
        else:
            print(msg)
    if not select:
        ema.apply(model); save_ckpt(model); best_ep = epochs
    else:
        model.load_state_dict(torch.load(CKPT, map_location=device)['state'])
    model.eval(); return model, best_ep

# if TRAIN_ON_ALL:
#     assert FIXED_EPOCHS, "Đặt FIXED_EPOCHS = epoch tốt nhất của lần chạy đầu (TRAIN_ON_ALL=False) rồi chạy lại từ đầu notebook."
#     print("⚠️ Val nằm trong train -> số liệu val bên dưới KHÔNG còn ý nghĩa đánh giá.")
#     model, BEST_EP = run_training(train_items, FIXED_EPOCHS, select=False)
# else:
#     model, BEST_EP = run_training(train_items, EPOCHS, select=True)
#     print(f"\n==> Epoch tốt nhất: {BEST_EP}  (dùng làm FIXED_EPOCHS khi train lần cuối với TRAIN_ON_ALL=True)")

In [ ]:
# ---------- PHASE 2: FINETUNE LÀM MƯỢT (COOL-DOWN) ----------
FT2_CKPT = "/content/drive/MyDrive/bamibert_nlu_ft711222.pt"
DRIVE_FT2_CKPT = "/content/drive/MyDrive/bamibert_nlu_ft711.pt"

# Ép Learning Rate xuống mức cực nhỏ để hội tụ (Convergence)
LR_ENC_FT = 8e-6    # Giảm 6 lần so với Phase 1
LR_HEAD_FT = 4e-4   # Giảm 10 lần so với Phase 1
FT_EPOCHS = 3
FT_PATIENCE = 60

set_seed(SEED)
model = MultiHeadNLU(MODEL_NAME).to(device)
# Nạp lại bộ não tốt nhất từ Phase 1
model.load_state_dict(torch.load(CKPT, map_location=device)['state'])
print(f"Đã nạp trọng số từ: {CKPT}")

ds = TrainDS(train_items)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)

enc_p  = [p for k, p in model.named_parameters() if k.startswith('enc.')]
head_p = [p for k, p in model.named_parameters() if not k.startswith('enc.')]
opt = torch.optim.AdamW([{'params': enc_p, 'lr': LR_ENC_FT}, {'params': head_p, 'lr': LR_HEAD_FT}], weight_decay=WD)

total = len(loader) * FT_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

use_scaler = USE_AMP and AMP_DTYPE == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
ema = EMA(model, EMA_DECAY)

# Lấy điểm xuất phát
best_ft = sel_score(eval_views(model, SEL_VIEWS))
print(f"Điểm khởi điểm Phase 2: {best_ft:.4f}\n")

bad = 0
# for ep in range(1, FT_EPOCHS + 1):
#     # Cố định nhiễu ở mức thấp để model tinh chỉnh chi tiết
#     ds.noise_scale = 0.3
#     model.train(); run = run_al = 0.0

#     for ids, mask, y, pidx, ib, mb in tqdm(loader, desc=f"FT{ep}", leave=False):
#         opt.zero_grad(set_to_none=True)
#         with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
#             lg_a, ea = model(ids.to(device), mask.to(device), return_emb=True)
#             loss = compute_loss(lg_a, y); al = torch.zeros((), device=device)
#             if pidx is not None:
#                 lg_b, eb = model(ib.to(device), mb.to(device), return_emb=True)
#                 loss = loss + compute_loss(lg_b, {h: y[h][pidx] for h in HEADS})
#                 al = (1 - (ea[pidx.to(device)] * eb).sum(-1)).mean()
#                 loss = loss + ALIGN_W * al

#         scaler.scale(loss).backward(); scaler.unscale_(opt)
#         nn.utils.clip_grad_norm_(model.parameters(), 1.0)

#         if use_scaler:
#             sb = scaler.get_scale(); scaler.step(opt); scaler.update()
#             if scaler.get_scale() >= sb: sched.step()
#         else:
#             opt.step(); sched.step()

#         ema.update(model); run += loss.item(); run_al += float(al)

#     ema.apply(model)
#     res = eval_views(model, SEL_VIEWS)
#     sc = sel_score(res)

#     msg = f"FT{ep:02d} loss={run / len(loader):.3f} align={run_al / len(loader):.3f} | score={sc:.4f} | " + ' '.join(f"{n}:{res[n]['full']:.3f}" for n in SEL_VIEWS)

#     if sc > best_ft + 1e-4:
#         best_ft, bad = sc, 0
#         torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, FT2_CKPT)
#         try: shutil.copy(FT2_CKPT, DRIVE_FT2_CKPT)
#         except: pass
#         msg += '  * (Kỷ lục mới)'
#     else:
#         bad += 1

#     ema.restore(model); print(msg)

#     if bad >= FT_PATIENCE:
#         print(f"Dừng sớm Phase 2. Đã đạt độ hội tụ tối đa.")
#         break

# print(f"\nHoàn tất. Checkpoint chốt sổ để nộp bài: {FT2_CKPT}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/bamibert_nlu_ft711.pt", map_location=device)['state'])

<All keys matched successfully>

In [ ]:
# ---------- ĐÁNH GIÁ ----------
res_c = eval_views(model, constrained=True)
res_a = eval_views(model, constrained=False)
N = len(val_items)
print(f"val n={N}  (core = goal+via+urgent+fragile | full = cả 8 head)   [CI = khoảng tin cậy 95% của full]\n")
print(f"{'góc nhìn':14s} | {'argmax core':>11s} {'full':>6s} | {'ràng buộc core':>14s} {'full':>6s} | {'CI full (ràng buộc)':>20s}")
for n in VAL_VIEWS:
    a, c = res_a[n], res_c[n]; lo, hi = wilson(int(c['ok_full'].sum()), N)
    print(f"{n:14s} | {a['core']:11.3f} {a['full']:6.3f} | {c['core']:14.3f} {c['full']:6.3f} | [{lo:.3f}, {hi:.3f}]")
print(f"\nĐiểm chọn checkpoint (4 góc nhìn): {sel_score(res_c):.4f}")
print("Lưu ý: với n=300, chênh lệch < ~1.5 điểm % giữa 2 lần chạy là NHIỄU, đừng kết luận.")

print("\n--- Dự đoán vô lý về cấu trúc (góc nhìn 'gốc' + 'bỏ dấu+typo') ---")
for nm in ('gốc', 'bỏ dấu+typo'):
    print(f"[{nm}] argmax: {dict(violations(res_a[nm]['pred'])) or 'không có'} | ràng buộc: {dict(violations(res_c[nm]['pred'])) or 'không có'}")

if not TRAIN_ON_ALL:
    UNSEEN = {fold(a) for al in VAL_ONLY_ALIASES.values() for a in al}
    has_unseen = np.array([any(a in fold(it['text']) for a in UNSEEN) for it in val_items])
    print("\n--- Câu chứa cách gọi địa điểm CHƯA TỪNG thấy ở train ---")
    for nm, mk in [('có từ lạ', has_unseen), ('chỉ từ đã thấy', ~has_unseen)]:
        if mk.sum(): print(f"{nm:16s}: core={res_c['gốc']['ok_core'][mk].mean():.3f} full={res_c['gốc']['ok_full'][mk].mean():.3f} (n={mk.sum()})")

print("\n--- Recall từng class (góc nhìn 'gốc', ràng buộc) ---")
pr = res_c['gốc']['pred']
for h, labels in HEADS.items():
    row = [f"{l}:{(pr[h][GOLD_VAL[h] == i] == i).mean():.2f}({(GOLD_VAL[h] == i).sum()})" for i, l in enumerate(labels) if (GOLD_VAL[h] == i).sum()]
    print(f"[{h}] " + ' '.join(row))

def show_wrong(name, k=15):
    r = res_c[name]; bad_idx = np.where(~r['ok_full'])[0]; print(f"\n=== SAI ở góc nhìn '{name}': {len(bad_idx)}/{N} ===")
    for i in bad_idx[:k]:
        g = {h: HEADS[h][GOLD_VAL[h][i]] for h in HEADS}; p = {h: HEADS[h][r['pred'][h][i]] for h in HEADS}
        diff = {h: (g[h], p[h]) for h in HEADS if g[h] != p[h]}
        print(' •', VAL_VIEWS[name][i][:160]); print('    khác nhau (đúng -> đoán):', diff)
for nm in ('gốc', 'bỏ dấu+typo'): show_wrong(nm)

val n=300  (core = goal+via+urgent+fragile | full = cả 8 head)   [CI = khoảng tin cậy 95% của full]

góc nhìn       | argmax core   full | ràng buộc core   full |  CI full (ràng buộc)
gốc            |       1.000  1.000 |          1.000  1.000 | [0.987, 1.000]
bỏ dấu         |       1.000  1.000 |          1.000  1.000 | [0.987, 1.000]
typo           |       0.987  0.987 |          0.987  0.987 | [0.966, 0.995]
bỏ dấu+typo    |       0.997  0.997 |          0.997  0.997 | [0.981, 0.999]
nặng           |       0.990  0.973 |          0.990  0.973 | [0.948, 0.986]

Điểm chọn checkpoint (4 góc nhìn): 0.9958
Lưu ý: với n=300, chênh lệch < ~1.5 điểm % giữa 2 lần chạy là NHIỄU, đừng kết luận.

--- Dự đoán vô lý về cấu trúc (góc nhìn 'gốc' + 'bỏ dấu+typo') ---
[gốc] argmax: không có | ràng buộc: không có
[bỏ dấu+typo] argmax: không có | ràng buộc: không có

--- Câu chứa cách gọi địa điểm CHƯA TỪNG thấy ở train ---
có từ lạ        : core=1.000 full=1.000 (n=188)
chỉ từ đã thấy  : core=1.000 fu

In [ ]:
res = eval_views(model, ['gốc', 'bỏ dấu+typo'], constrained=True)
UNSEEN = {fold(a) for al in VAL_ONLY_ALIASES.values() for a in al}
has_unseen = np.array([any(re.search(r'(?<!\w)' + re.escape(a) + r'(?!\w)', fold(it['text'])) for a in UNSEEN) for it in val_items])
has_man = np.array([bool(re.search(r'\bman (tren|duoi|phai|trai)\b', fold(it['text']))) for it in val_items])
for nm in res:
    pr = res[nm]['pred']
    print(f"\n[{nm}] số câu sai theo head:", {h: int((pr[h] != GOLD_VAL[h]).sum()) for h in HEADS})
    for tag, mk in [('alias lạ', has_unseen), ('alias đã thấy', ~has_unseen), ('có "mạn ..."', has_man)]:
        print(f"  {tag:14s} n={mk.sum():3d} core={res[nm]['ok_core'][mk].mean():.3f} full={res[nm]['ok_full'][mk].mean():.3f}")

# kiểm tra các họ câu của val có nằm trong 2000 câu train thật không (chỉ thêm vào generator nếu CÓ)
for f in ['chưa đi thẳng được', 'đích đến là', 'phải được', 'kỵ va đập', 'mỏng manh', 'không lo vỡ', 'mai giao cũng kịp', 'mạn trên', 'mạn phải']:
    print(f"{f:22s} train thật: {sum(fold(f) in fold(it['text']) for it in base_items)}")


[gốc] số câu sai theo head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 0}
  alias lạ       n=188 core=1.000 full=1.000
  alias đã thấy  n=112 core=1.000 full=1.000
  có "mạn ..."   n= 41 core=1.000 full=1.000

[bỏ dấu+typo] số câu sai theo head: {'goal': 0, 'via': 1, 'gkind': 0, 'ganchor': 0, 'vkind': 1, 'vanchor': 1, 'urgent': 0, 'fragile': 0}
  alias lạ       n=188 core=0.995 full=0.995
  alias đã thấy  n=112 core=1.000 full=1.000
  có "mạn ..."   n= 41 core=1.000 full=1.000
chưa đi thẳng được     train thật: 0
đích đến là            train thật: 0
phải được              train thật: 0
kỵ va đập              train thật: 0
mỏng manh              train thật: 0
không lo vỡ            train thật: 0
mai giao cũng kịp      train thật: 0
mạn trên               train thật: 0
mạn phải               train thật: 0


In [ ]:
# ---------- TTA có giúp không? (chỉ bật nếu KHÔNG làm tệ đi) ----------
res_t = eval_views(model, constrained=True, tta=True)
print(f"{'góc nhìn':14s} | không TTA full | có TTA full")
for n in VAL_VIEWS: print(f"{n:14s} | {res_c[n]['full']:14.3f} | {res_t[n]['full']:.3f}")
USE_TTA = sel_score(res_t) > sel_score(res_c) + 0.002
print("\n=> USE_TTA =", USE_TTA)

# ---------- Suy luận ----------
def parse_missions(texts, tta=None):
    tta = USE_TTA if tta is None else tta
    P = predict_probs_tta(model, texts) if tta else predict_probs(model, texts); pred = decode(P); out = []
    for i, t in enumerate(texts):
        p = {h: HEADS[h][int(pred[h][i])] for h in HEADS}
        ref = lambda k, a: None if p[k] == 'none' else {'kind': p[k], 'anchor': None if p[a] == 'none' else p[a]}
        out.append({'text': t, 'goal': p['goal'], 'goal_ref': ref('gkind', 'ganchor'),
                    'via': None if p['via'] == 'none' else p['via'],
                    'via_ref': ref('vkind', 'vanchor') if p['via'] != 'none' else None,
                    'urgent': p['urgent'] == 'yes', 'fragile': p['fragile'] == 'yes',
                    'min_conf': min(float(P[h][i].max()) for h in HEADS)})
    return out

for m in parse_missions(["Có đơn giao gói hàng đặt online ở tòa nhà ở của sinh viên ở gần phòng lab. Đang cần gấp lắm.",
                         "Robot oi, khong can ghe day phong hoc. Bep truong o cang tin man tren dang cho hop giay. Dung nham voi phong lab nhe. Do khong vo duoc dau. Dang can gap lam."]):
    print(json.dumps(m, ensure_ascii=False, indent=1))
print(f"\nCheckpoint: {CKPT}  (và bản sao trên Drive nếu mount được).  Trong pipeline: dùng parse_missions(...) và nhớ dùng decode() có ràng buộc.")

góc nhìn       | không TTA full | có TTA full
gốc            |          1.000 | 1.000
bỏ dấu         |          1.000 | 1.000
typo           |          0.987 | 0.987
bỏ dấu+typo    |          0.997 | 0.997
nặng           |          0.973 | 0.973

=> USE_TTA = False
{
 "text": "Có đơn giao gói hàng đặt online ở tòa nhà ở của sinh viên ở gần phòng lab. Đang cần gấp lắm.",
 "goal": "dorm",
 "goal_ref": {
  "kind": "near",
  "anchor": "lab"
 },
 "via": null,
 "via_ref": null,
 "urgent": true,
 "fragile": false,
 "min_conf": 0.9558059573173523
}
{
 "text": "Robot oi, khong can ghe day phong hoc. Bep truong o cang tin man tren dang cho hop giay. Dung nham voi phong lab nhe. Do khong vo duoc dau. Dang can gap lam.",
 "goal": "canteen",
 "goal_ref": {
  "kind": "north",
  "anchor": null
 },
 "via": null,
 "via_ref": null,
 "urgent": true,
 "fragile": false,
 "min_conf": 0.9560016989707947
}

Checkpoint: /content/bamibert_nlu_ft711.pt  (và bản sao trên Drive nếu mount được).  Trong pipeline: dù

#Finetune

In [ ]:
# ---------- PHASE 3 / BLOCK 1: nạp new_phrases.py, gộp vào generator ----------
# Cần upload new_phrases.py lên Colab (hoặc đổi đường dẫn sang Drive).
NEW_PHRASES_PATH = '/content/new_phrases.py'
assert os.path.exists(NEW_PHRASES_PATH), f"Không thấy {NEW_PHRASES_PATH} — upload file new_phrases.py vào /content/"

# Bật/tắt từng nhóm để so sánh (ablation). Chạy lại block 1 -> 2 -> 3 sau khi đổi.
GROUPS = dict(aliases=True, ref=True, neg=True, urgfra=True, verbs=True, to=True,
              pick=True, close=True, openers=True, roles=True)
P_CORE   = 0.35   # xác suất chọn alias GỐC của train khi sinh câu (chống loãng alias thật)
ROLE_N   = 500    # số câu "{vai trò} cần nhận {hàng}"
SEED_SYN, SEED_NEG, SEED_ROLE = 2026, 2027, 2028

NP = {}
exec(open(NEW_PHRASES_PATH, encoding='utf-8').read(), NP)
# ===== PATCH phase 3 (dán NGAY SAU dòng exec(open(NEW_PHRASES_PATH...), NP) trong Block 1) =====
# 1) bỏ alias/mẫu có hại   2) thêm alias kiểu 'nơi + hoạt động'   3) cụm bị phủ định ưu tiên alias quen   4) thêm 'mạn' khi gộp val
_DROP_ALIAS = {'không gian đọc sách', 'không gian thư viện', 'cổng trước', 'cổng sau', 'cổng xe'}   # chứa 'không'/'trước'/'sau' hoặc lẫn gate-parking
_NEW_FUNC = {
 'canteen': ['nơi ăn trưa của sinh viên', 'chỗ ăn cơm', 'nơi mua cơm', 'nơi phục vụ đồ ăn', 'chỗ bán thức ăn', 'nơi bán bữa sáng', 'nơi ăn cơm tập thể', 'khu phục vụ cơm trưa'],
 'clinic': ['nơi khám bệnh', 'nơi thăm khám sức khỏe', 'chỗ chữa bệnh', 'nơi băng bó vết thương', 'nơi gặp bác sĩ', 'chỗ gặp y tá', 'nơi lấy thuốc', 'nơi chăm sóc sinh viên ốm'],
 'dorm': ['nơi sinh viên ở', 'chỗ ngủ của sinh viên', 'nơi sinh viên nghỉ ngơi', 'khu sinh viên ở lại', 'chỗ ở của sinh viên nội trú', 'nơi sinh viên lưu trú', 'dãy nhà sinh viên ở', 'khu phòng ngủ sinh viên'],
 'gate': ['nơi ra vào trường', 'chỗ vào trường', 'nơi bảo vệ kiểm tra thẻ', 'nơi đón khách ở lối vào', 'chỗ bảo vệ trực', 'nơi kiểm soát người ra vào', 'chỗ đứng gác'],
 'lab': ['nơi làm thực hành', 'nơi làm việc với hóa chất', 'nơi chạy thí nghiệm', 'nơi tiến hành thí nghiệm', 'chỗ nghiên cứu mẫu', 'nơi thực hành kỹ thuật', 'nơi đo đạc thí nghiệm', 'chỗ làm thực nghiệm'],
 'lecture': ['nơi học bài trên lớp', 'nơi nghe giảng', 'chỗ học lý thuyết', 'nơi sinh viên học', 'chỗ ngồi học của lớp', 'nơi thầy cô giảng bài', 'nơi diễn ra tiết học', 'nơi học chính khóa'],
 'library': ['nơi đọc và mượn sách', 'chỗ mượn giáo trình', 'chỗ đọc sách yên tĩnh', 'nơi tra cứu tài liệu', 'chỗ tìm sách', 'nơi sinh viên đọc sách'],
 'office': ['nơi giải quyết thủ tục', 'nơi xin giấy xác nhận', 'chỗ nộp đơn', 'nơi lấy giấy tờ', 'nơi đóng dấu', 'chỗ làm giấy tờ', 'nơi tiếp nhận đơn từ', 'nơi xử lý hồ sơ'],
 'parking': ['nơi để xe máy', 'chỗ gửi xe đạp', 'nơi dựng xe', 'chỗ dựng xe', 'nơi xe máy đậu', 'nơi sinh viên gửi xe', 'khu xe đạp xe máy', 'nơi lấy xe'],
 'sports': ['nơi chơi thể thao', 'nơi tập luyện thể thao', 'chỗ đá bóng', 'nơi chạy bộ tập thể dục', 'nơi sinh viên tập thể thao', 'chỗ chơi bóng', 'nơi diễn ra trận đấu', 'nơi luyện tập đội tuyển']
}
for c in NP['NEW_ALIASES_2']:
    NP['NEW_ALIASES_2'][c] = [a for a in NP['NEW_ALIASES_2'][c] if a not in _DROP_ALIAS] + [a for a in _NEW_FUNC[c] if a not in NP['NEW_ALIASES_2'][c]]
NP['TO_SAFE']    = [x for x in NP['TO_SAFE'] if x not in ('đi qua', 'thẳng qua', 'qua tận')]   # đọc như điểm ghé (via), nhãn lại là goal
NP['URG_NO_NEW'] = [x for x in NP['URG_NO_NEW'] if x != 'Cứ xếp hàng theo thứ tự.']
P_NEG_CORE = 0.6     # xác suất cụm BỊ PHỦ ĐỊNH dùng alias gốc (quen); cần sửa 1 dòng trong cell 10, xem hướng dẫn
if TRAIN_ON_ALL:     # 'mạn' chỉ có ở val -> chỉ thêm khi train cuối, để số đo val trước đó không bị lạc quan
    for k, add in {'north': ['ở mạn trên', 'mạn trên'], 'south': ['ở mạn dưới', 'mạn dưới'],
                   'west':  ['ở mạn trái', 'mạn trái'],  'east':  ['ở mạn phải', 'mạn phải']}.items():
        NP['REF_NEW'][k] = list(NP['REF_NEW'][k]) + [x for x in add if x not in NP['REF_NEW'][k]]


def _add(lst, new):                      # gộp có khử trùng -> chạy lại block không bị nhân đôi
    seen = set(lst)
    for x in new:
        if x not in seen: lst.append(x); seen.add(x)

def _check_ph(name, items, allowed):
    for x in items:
        bad = set(re.findall(r'\{(\w+)\}', x)) - allowed
        assert not bad, f"{name}: placeholder lạ {bad} trong '{x}'"

# --- gộp từng nhóm ---
if GROUPS['aliases']:
    for c in PLACES: _add(EXTRA_ALIASES[c], NP['NEW_ALIASES_2'][c])
if GROUPS['ref']:
    for k in REF:
        _check_ph('REF_NEW', NP['REF_NEW'][k], {'A'} if k in ('near', 'far') else set())
        _add(REF[k], NP['REF_NEW'][k])
if GROUPS['neg']:
    _check_ph('NEG_T_NEW', NP['NEG_T_NEW'], {'t', 'T'}); _add(NEG_T, NP['NEG_T_NEW'])   # KHÔNG gộp vào TRAP
if GROUPS['urgfra']:
    _check_ph('URG_YES_NEW', NP['URG_YES_NEW'], {'m'})
    _add(URG_YES, NP['URG_YES_NEW']); _add(URG_NO, NP['URG_NO_NEW'])
    _add(FRA_YES, NP['FRA_YES_NEW']); _add(FRA_NO, NP['FRA_NO_NEW'])
if GROUPS['verbs']: _add(VERBS, NP['VERBS_NEW'])      # VERBS_POLITE bỏ: khung có "giúp mình" ở cuối sẽ bị đôi "giúp"
if GROUPS['to']:    _add(TO, [x for x in NP['TO_SAFE'] if 'ngay' not in fold(x)])   # bỏ 'ngay đến/đến ngay...': rò tín hiệu urgent (nhãn urgent=no mà câu có 'ngay')
                     # TO_CONTEXT (ra/lên/xuống/về...) bỏ: gượng với nhiều địa điểm
if GROUPS['pick']:  _add(PICK, NP['PICK_NEW'])
if GROUPS['close']: _add(CLOSE, NP['CLOSE_NEW'])
if GROUPS['openers']:
    _check_ph('OPENERS_NEW', NP['OPENERS_NEW'], {'n'}); _add(OPENERS, NP['OPENERS_NEW'])

# --- dựng lại BANK / ALIAS2CLS / ALIAS_RE (swap_synonyms & generator đọc các biến global này) ---
BANK = {c: list(dict.fromkeys(TRAIN_ALIASES[c] + EXTRA_ALIASES[c] + NEW_ALIASES[c]
                              + (VAL_ONLY_ALIASES[c] if TRAIN_ON_ALL else []))) for c in PLACES}
ALIAS2CLS = {}
for c, al in BANK.items():
    for a in al:
        assert ALIAS2CLS.get(fold(a), c) == c, f"cụm trùng giữa 2 lớp: {a}"
        ALIAS2CLS[fold(a)] = c
if not TRAIN_ON_ALL:
    leak = {fold(a) for al in VAL_ONLY_ALIASES.values() for a in al} & set(ALIAS2CLS)
    assert not leak, f"alias của val lọt vào train: {leak}"
nest = 0                                  # alias lớp A nằm lọt trong alias lớp B (so theo n-gram từ, nhanh hơn O(n^2) regex)
for b, cb in ALIAS2CLS.items():
    w = b.split()
    for i in range(len(w)):
        for j in range(i + 1, len(w) + 1):
            gram = ' '.join(w[i:j])
            if gram != b and ALIAS2CLS.get(gram, cb) != cb:
                nest += 1
                if nest <= 10: print(f"⚠️ '{gram}'({ALIAS2CLS[gram]}) nằm trong '{b}'({cb})")
print("cảnh báo lồng:", nest)
ALIAS_RE = re.compile(r'(?<!\w)(?:' + '|'.join(re.escape(a) for a in sorted(ALIAS2CLS, key=len, reverse=True)) + r')(?!\w)')

REF2 = {k: list(v) for k, v in REF.items()}
if TRAIN_ON_ALL:
    _add(REF2['north'], ['ở mạn trên', 'mạn trên']); _add(REF2['south'], ['ở mạn dưới', 'mạn dưới'])

# --- chống loãng alias thật: chọn alias gốc với xác suất P_CORE ---
CORE_SET = {c: set(TRAIN_ALIASES[c]) for c in PLACES}
def pick_alias(cls, rng, p_core=None):
    p = P_CORE if p_core is None else p_core
    return rng.choice(TRAIN_ALIASES[cls]) if rng.random() < p else rng.choice(BANK[cls])

def _np(place, kind, anchor, rng):                       # dùng bởi generate_balanced -> render
    s = pick_alias(place, rng)
    if kind: s += ' ' + rng.choice(REF[kind]).format(A=pick_alias(anchor, rng) if anchor else '')
    return s

_orig_bal_pick = Balancer.__dict__.get('_orig_pick', Balancer.pick)   # chạy lại block không bị bọc 2 lần
Balancer._orig_pick = _orig_bal_pick
def _biased_pick(self, key, options):                    # dùng bởi generate_neg
    if isinstance(key, tuple) and key[0] == 'alias' and self.rng.random() < P_CORE:
        core = [a for a in options if a in CORE_SET[key[1]]]
        if core: options = core
    return _orig_bal_pick(self, key, options)
Balancer.pick = _biased_pick

def swap_synonyms(text, rng=random, p=SYN_P):            # augmentation: đổi đồng nghĩa cũng chọn theo pick_alias
    t = unicodedata.normalize('NFC', text); f = ''.join(strip_accents(c) for c in t).lower()
    if len(t) != len(f): return text
    accented = bool(VI.search(t.lower())); out, last = [], 0
    for m in ALIAS_RE.finditer(f):
        s, e = m.span(); cls = ALIAS2CLS[m.group()]
        if cls == 'library' and f[max(0, s-4):s] == 'the ': continue
        if rng.random() > p: continue
        new = pick_alias(cls, rng)
        if not accented: new = strip_accents(new)
        if t[s].isupper(): new = new[0].upper() + new[1:]
        out.append(t[last:s]); out.append(new); last = e
    out.append(t[last:]); return ''.join(out)

print({c: len(BANK[c]) for c in PLACES})
print("REF:", {k: len(v) for k, v in REF.items()}, "| NEG_T:", len(NEG_T), "| VERBS:", len(VERBS), "| TO:", len(TO),
      "| PICK:", len(PICK), "| OPENERS:", len(OPENERS))

AssertionError: Không thấy /content/new_phrases.py — upload file new_phrases.py vào /content/

In [ ]:
# ===== BLOCK 1b: typo NHẮM VÀO CỤM CỜ urgent/fragile (chạy ngay sau Block 1, trước Block 2) =====
# Lý do: typo ngẫu nhiên hiếm khi trúng đúng chữ then chốt ("gấp", "mỏng manh", "vỡ"...), nên model ít thấy
# cảnh cụm cờ bị gõ sai ("mong msnh"). Hàm này gõ sai ĐÚNG 1 từ trong cụm cờ, nhãn giữ nguyên.
P_CUE_TYPO = 0.25

def _cue_alt(lst):
    pats = sorted({re.escape(fold(x).strip().rstrip('.!')).replace(r'\{m\}', r'\d+') for x in lst}, key=len, reverse=True)
    return re.compile(r'(?<!\w)(?:' + '|'.join(pats) + r')(?!\w)')
CUE_RX = _cue_alt(URG_YES + URG_NO + FRA_YES + FRA_NO)          # đọc các list ĐÃ gộp ở Block 1

def typo_cue(text, rng=random):
    t = unicodedata.normalize('NFC', text); f = ''.join(strip_accents(c) for c in t).lower()
    if len(t) != len(f): return text                                  # lệch độ dài (hiếm) -> bỏ qua cho an toàn
    spans = [m.span() for m in CUE_RX.finditer(f)]
    if not spans: return text
    s, e = rng.choice(spans); ws = list(_WORD2.finditer(t, s, e))
    if not ws: return text
    m = rng.choice(ws)
    return t[:m.start()] + _typo_word(m.group(), rng) + t[m.end():]

_augment_v2_base = globals().get('_augment_v2_base', augment_v2)      # chạy lại block không bị bọc 2 lần
def augment_v2(text, y=None, rng=random, noise_scale=1.0):
    t = _augment_v2_base(text, y, rng, noise_scale)
    return typo_cue(t, rng) if rng.random() < P_CUE_TYPO * noise_scale else t

_r = random.Random(5); _hit = 0; _smp = [it['text'] for it in random.Random(1).sample(train_items if 'train_items' in globals() else base_items, 300)]
print("Block 1b: câu có cụm cờ nhận diện được:", sum(bool(CUE_RX.search(fold(t))) for t in _smp), "/", len(_smp))
for t in _smp[:3]:
    print('  ', t[:110]); print('  ->', typo_cue(t, _r)[:110])


Block 1b: câu có cụm cờ nhận diện được: 230 / 300
   Xin chào! Nhờ robot gửi thùng thực phẩm qua trung tâm y tế trường nằm phía bắc. Nhưng phải tạt qua chỗ để xe t
  -> Xin chào! Nhờ robot gửi thùng thực phẩm qua trung tâm y tế trường nằm phía bắc. Nhưng phải tạt qua chỗ để xe t
   Bỏ qua chỗ ở sinh viên, không phải ở đó. Qua cổng a trước, sau đó vận chuyển nguyên liệu nấu ăn đến chỗ khám b
  -> Bỏ qua chỗ ở sinh viên, không phải ở đó. Qua cổng a trước, sau đó vận chuyển nguyên liệu nấu ăn đến chỗ khám b
   Robot ơi, chú ý: hàng dễ hỏng khi va đập. Cần túi đồ ở ký túc xá. Việc này khẩn cấp.
  -> Robot ơi, chú ý: hàng dễ hỏng khi va đập. Cần túi đồ ở ký túc xá. Việc này khẩn cấp.


In [ ]:
# ---------- PHASE 3 / BLOCK 2: câu "{vai trò} cần nhận {hàng}" + sinh lại toàn bộ dữ liệu ----------
# Kiểu câu có thật trong train (191/2000): đích chỉ suy ra từ vai trò, KHÔNG có tên địa điểm.
ROLES_OK = {}
for c, rs in NP['ROLES'].items():
    ok = [r for r in rs if not ALIAS_RE.search(fold(r))]
    if len(ok) < len(rs): print(f"  bỏ vai trò trùng alias ({c}):", [r for r in rs if r not in ok])
    ROLES_OK[c] = ok
# --- đối chiếu vai trò -> lớp với câu THẬT của train (chỉ base_items, không đụng val) ---
role_only = [it for it in base_items if not ALIAS_RE.search(fold(it['text']))]      # câu không có tên địa điểm nào
ver = unver = 0; dropped = []
for c in list(ROLES_OK):
    keep = []
    for r in ROLES_OK[c]:
        rx = re.compile(r'(?<!\w)' + re.escape(fold(r)) + r'(?!\w)')
        cnt = collections.Counter(HEADS['goal'][it['y']['goal']] for it in role_only if rx.search(fold(it['text'])))
        if cnt and cnt.most_common(1)[0][0] != c: dropped.append((r, c, dict(cnt))); continue   # train thật gán lớp khác -> bỏ
        ver += bool(cnt); unver += not cnt; keep.append(r)
    ROLES_OK[c] = keep
assert all(ROLES_OK[c] for c in PLACES), "có lớp không còn vai trò nào"
print(f"vai trò: {ver} khớp train thật | {unver} CHƯA có bằng chứng trong train thật | bỏ vì mâu thuẫn: {len(dropped)}")
for d in dropped: print("   bỏ:", d)
ROLE_TPL = NP['ROLE_TEMPLATES']; _check_ph('ROLE_TEMPLATES', ROLE_TPL, {'R', 'r', 'i'})

def generate_roles(n, seed=SEED_ROLE):
    rng = random.Random(seed); B = Balancer(rng); out = []; tries = 0
    seen = {it['text'] for it in base_items + syn_items + neg_items + val_items}
    while len(out) < n and tries < n * 30:
        tries += 1
        cls  = B.pick('role_cls', PLACES)
        role = B.pick(('role', cls), ROLES_OK[cls])
        item = B.pick('role_item', ITEMS)
        body = B.pick('role_tpl', ROLE_TPL).format(R=role[0].upper() + role[1:], r=role, i=item)
        urg, fra = rng.random() < 0.20, rng.random() < 0.20
        parts = [body]
        if urg: parts.append(rng.choice(URG_YES).replace('{m}', str(rng.choice([2, 5, 10, 15]))))
        elif rng.random() < 0.10: parts.append(rng.choice(URG_NO))
        if fra: parts.append(rng.choice(FRA_YES))
        elif rng.random() < 0.10: parts.append(rng.choice(FRA_NO))
        if rng.random() < 0.15:                                   # bẫy: nhắc 1 địa điểm khác nhưng bị phủ định
            other = rng.choice([p for p in PLACES if p != cls]); t = pick_alias(other, rng)
            parts.append(rng.choice(NEG_T).format(t=t, T=t[0].upper() + t[1:]) + '.')
        rng.shuffle(parts)
        txt = ' '.join(parts)
        op = rng.choice(OPENERS).replace('{n}', str(rng.randint(1000, 9999)))
        if op.rstrip().endswith(','): txt = txt[0].lower() + txt[1:]
        if rng.random() < 0.25: txt += ' ' + rng.choice(CLOSE)
        txt = re.sub(r'\s+', ' ', op + txt).strip()
        if rng.random() < 0.15: txt = txt.lower()
        if txt in seen: continue
        seen.add(txt)
        y = mission_to_labels({'goal': cls, 'via': None, 'goal_ref': None, 'via_ref': None, 'urgent': urg, 'fragile': fra})
        out.append({'text': txt, 'y': y, 'syn': True, 'role': True})
    print(f"Đã sinh {len(out)} câu vai trò"); return out

# --- sinh lại dữ liệu với bank/mẫu mới (seed khác lần trước để có câu mới) ---
set_seed(SEED)
syn_items  = generate_balanced(base_items, seed=SEED_SYN)
neg_items, _B = generate_neg(NEG_N, seed=SEED_NEG)
role_items = generate_roles(ROLE_N) if GROUPS['roles'] else []

# --- kiểm tra tự động (giống cell 11) ---
bad = 0
for it in neg_items:
    f = fold(it['text'])
    for h in ('goal', 'via'):
        l = HEADS[h][it['y'][h]]
        if l != 'none' and not any(fold(a) in f for a in BANK[l]): bad += 1
    for c in it['neg_cls']:
        if not any(fold(a) in f for a in BANK[c]): bad += 1
        if any(HEADS[h][it['y'][h]] == c for h in ('goal', 'via', 'ganchor', 'vanchor')): bad += 1
print("neg: câu lỗi nhãn/phủ định:", bad, "/", len(neg_items))
val_texts = {it['text'] for it in val_items}
new_all = syn_items + neg_items + role_items
print("câu sinh trùng nguyên văn với val:", sum(it['text'] in val_texts for it in new_all), "(phải = 0)")

# --- alias thật còn chiếm bao nhiêu % số lần alias xuất hiện trong câu sinh? ---
core_fold = {fold(a) for c in PLACES for a in TRAIN_ALIASES[c]}
hit = tot = 0
for it in new_all:
    for m in ALIAS_RE.finditer(fold(it['text'])): tot += 1; hit += m.group() in core_fold
print(f"alias GỐC chiếm {hit / max(tot, 1):.1%} số lần xuất hiện trong câu sinh (trước khi gộp ~31-41%, nếu không chống loãng ~7-9%)")

# --- ghép dữ liệu train ---
train_items = base_items + syn_items + neg_items + role_items
if TRAIN_ON_ALL: train_items = train_items + val_items
n_real = sum(not it.get('syn') for it in train_items)
print(f"train: {len(train_items)} câu | thật={n_real} | sinh thêm={len(train_items) - n_real} "
      f"(syn={len(syn_items)}, neg={len(neg_items)}, role={len(role_items)})")
dist_report(train_items, 'TRAIN (phase 3)')
for it in random.Random(9).sample(role_items, min(5, len(role_items))): print('  role:', it['text'])
for it in random.Random(9).sample(syn_items, 4): print('  syn :', it['text'])

NameError: name 'NP' is not defined

In [ ]:
# ---------- PHASE 3 / BLOCK 3: FINETUNE TIẾP từ ft2 trên dataset mới ----------
FT2_CKPT       = "/content/alo.pt"                 # model mới nhất (kết quả phase 2)
FT3_CKPT       = "/content/bamibert_nlu_ft4.pt"
DRIVE_FT3_CKPT = "/content/drive/MyDrive/bamibert_nlu_ft4.pt"
assert os.path.exists(FT2_CKPT), f"Không thấy {FT2_CKPT}"

LR_ENC_FT3, LR_HEAD_FT3 = 1.2e-6, 4e-5     # giữa phase 2 (8e-6) và phase 1 (3e-5): data mới cần học thêm cụm lạ
FT3_EPOCHS, FT3_PATIENCE = 20, 20
NOISE_HI_FT3, NOISE_LO_FT3 = 0.5, 0.3      # 60% epoch đầu nhiễu vừa, còn lại nhẹ (phase 2 dùng 0.3 cố định)

gc.collect(); torch.cuda.empty_cache()
set_seed(SEED)
model = MultiHeadNLU(MODEL_NAME).to(device)
model.load_state_dict(torch.load(FT2_CKPT, map_location=device)['state'])
print("Đã nạp:", FT2_CKPT)

ds = TrainDS(train_items)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
print(f"mẫu / epoch: {len(ds)} | bước / epoch: {len(loader)}")
enc_p  = [p for k, p in model.named_parameters() if k.startswith('enc.')]
head_p = [p for k, p in model.named_parameters() if not k.startswith('enc.')]
opt = torch.optim.AdamW([{'params': enc_p, 'lr': LR_ENC_FT3}, {'params': head_p, 'lr': LR_HEAD_FT3}], weight_decay=WD)
total = len(loader) * FT3_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
use_scaler = USE_AMP and AMP_DTYPE == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
ema = EMA(model, EMA_DECAY)

res0 = eval_views(model, SEL_VIEWS); base_ft3 = sel_score(res0)
print(f"Điểm xuất phát (ft2 trên val): {base_ft3:.4f} | " + ' '.join(f"{n}:{res0[n]['full']:.3f}" for n in SEL_VIEWS))
best_ft3, best_ep, bad = base_ft3, 0, 0

for ep in range(1, FT3_EPOCHS + 1):
    ds.noise_scale = NOISE_HI_FT3 if ep <= 0.6 * FT3_EPOCHS else NOISE_LO_FT3
    model.train(); run = run_al = 0.0
    for ids, mask, y, pidx, ib, mb in tqdm(loader, desc=f"FT3-{ep}", leave=False):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            lg_a, ea = model(ids.to(device), mask.to(device), return_emb=True)
            loss = compute_loss(lg_a, y); al = torch.zeros((), device=device)
            if pidx is not None:
                lg_b, eb = model(ib.to(device), mb.to(device), return_emb=True)
                loss = loss + compute_loss(lg_b, {h: y[h][pidx] for h in HEADS})
                al = (1 - (ea[pidx.to(device)] * eb).sum(-1)).mean()
                loss = loss + ALIGN_W * al
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if use_scaler:
            sb = scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale() >= sb: sched.step()
        else:
            opt.step(); sched.step()
        ema.update(model); run += loss.item(); run_al += float(al)
    ema.apply(model); res = eval_views(model, SEL_VIEWS); sc = sel_score(res)
    msg = (f"FT3-{ep:02d} loss={run / len(loader):.3f} align={run_al / len(loader):.3f} | score={sc:.4f} | "
           + ' '.join(f"{n}:{res[n]['full']:.3f}" for n in SEL_VIEWS))
    if sc > best_ft3 + 1e-4:                       # chỉ lưu khi THẮNG ft2 (điểm xuất phát) và các epoch trước
        best_ft3, best_ep, bad = sc, ep, 0
        torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, FT3_CKPT)
        try: shutil.copy(FT3_CKPT, DRIVE_FT3_CKPT)
        except Exception as e: print('  (Drive lỗi:', e, ')')
        msg += '  * (kỷ lục mới)'
    else: bad += 1
    ema.restore(model); print(msg)
    if bad >= FT3_PATIENCE: print("Dừng sớm."); break

if best_ep == 0:
    print(f"\n⚠️ Không epoch nào vượt ft2 ({base_ft3:.4f}) -> KHÔNG lưu ft3. Giữ nguyên {FT2_CKPT}.")
    print("   Thử: tắt bớt nhóm trong GROUPS (block 1), hạ LR_ENC_FT3, hoặc giảm ROLE_N / P_CORE rồi chạy lại block 1->3.")
else:
    print(f"\nHoàn tất. Tốt nhất ở epoch {best_ep}: {best_ft3:.4f} (ft2: {base_ft3:.4f}) -> {FT3_CKPT}")

In [ ]:
model.load_state_dict(torch.load("/content/drive/MyDrive/bamibert_nlu_ft62.pt", map_location=device)['state'])
# ---------- ĐÁNH GIÁ ----------
res_c = eval_views(model, constrained=True)
res_a = eval_views(model, constrained=False)
N = len(val_items)
print(f"val n={N}  (core = goal+via+urgent+fragile | full = cả 8 head)   [CI = khoảng tin cậy 95% của full]\n")
print(f"{'góc nhìn':14s} | {'argmax core':>11s} {'full':>6s} | {'ràng buộc core':>14s} {'full':>6s} | {'CI full (ràng buộc)':>20s}")
for n in VAL_VIEWS:
    a, c = res_a[n], res_c[n]; lo, hi = wilson(int(c['ok_full'].sum()), N)
    print(f"{n:14s} | {a['core']:11.3f} {a['full']:6.3f} | {c['core']:14.3f} {c['full']:6.3f} | [{lo:.3f}, {hi:.3f}]")
print(f"\nĐiểm chọn checkpoint (4 góc nhìn): {sel_score(res_c):.4f}")
print("Lưu ý: với n=300, chênh lệch < ~1.5 điểm % giữa 2 lần chạy là NHIỄU, đừng kết luận.")

print("\n--- Dự đoán vô lý về cấu trúc (góc nhìn 'gốc' + 'bỏ dấu+typo') ---")
for nm in ('gốc', 'bỏ dấu+typo'):
    print(f"[{nm}] argmax: {dict(violations(res_a[nm]['pred'])) or 'không có'} | ràng buộc: {dict(violations(res_c[nm]['pred'])) or 'không có'}")

if not TRAIN_ON_ALL:
    UNSEEN = {fold(a) for al in VAL_ONLY_ALIASES.values() for a in al}
    has_unseen = np.array([any(a in fold(it['text']) for a in UNSEEN) for it in val_items])
    print("\n--- Câu chứa cách gọi địa điểm CHƯA TỪNG thấy ở train ---")
    for nm, mk in [('có từ lạ', has_unseen), ('chỉ từ đã thấy', ~has_unseen)]:
        if mk.sum(): print(f"{nm:16s}: core={res_c['gốc']['ok_core'][mk].mean():.3f} full={res_c['gốc']['ok_full'][mk].mean():.3f} (n={mk.sum()})")

print("\n--- Recall từng class (góc nhìn 'gốc', ràng buộc) ---")
pr = res_c['gốc']['pred']
for h, labels in HEADS.items():
    row = [f"{l}:{(pr[h][GOLD_VAL[h] == i] == i).mean():.2f}({(GOLD_VAL[h] == i).sum()})" for i, l in enumerate(labels) if (GOLD_VAL[h] == i).sum()]
    print(f"[{h}] " + ' '.join(row))

def show_wrong(name, k=15):
    r = res_c[name]; bad_idx = np.where(~r['ok_full'])[0]; print(f"\n=== SAI ở góc nhìn '{name}': {len(bad_idx)}/{N} ===")
    for i in bad_idx[:k]:
        g = {h: HEADS[h][GOLD_VAL[h][i]] for h in HEADS}; p = {h: HEADS[h][r['pred'][h][i]] for h in HEADS}
        diff = {h: (g[h], p[h]) for h in HEADS if g[h] != p[h]}
        print(' •', VAL_VIEWS[name][i][:160]); print('    khác nhau (đúng -> đoán):', diff)
for nm in ('gốc', 'bỏ dấu+typo'): show_wrong(nm)

In [ ]:
# # ---------- PHASE 3 / BLOCK 4: so sánh ft2 vs ft3 trên 5 góc nhìn ----------
# def _load(path):
#     m = MultiHeadNLU(MODEL_NAME).to(device); m.load_state_dict(torch.load(path, map_location=device)['state']); m.eval(); return m

# rows = {}
# for tag, path in [('ft2', FT2_CKPT), ('ft3', FT3_CKPT)]:
#     if not os.path.exists(path): print(f"(bỏ qua {tag}: chưa có {path})"); continue
#     m = _load(path); rows[tag] = eval_views(m, constrained=True); del m; gc.collect(); torch.cuda.empty_cache()

# N = len(val_items)
# print(f"val n={N}  (full, giải mã có ràng buộc)   nhớ: chênh < ~1.5 điểm % với n=300 là NHIỄU\n")
# print(f"{'góc nhìn':14s} | " + ' | '.join(f"{t:>6s}" for t in rows))
# for n in VAL_VIEWS: print(f"{n:14s} | " + ' | '.join(f"{rows[t][n]['full']:6.3f}" for t in rows))
# for t in rows: print(f"điểm chọn ckpt {t}: {sel_score(rows[t]):.4f}")

# if 'ft3' in rows:
#     model = _load(FT3_CKPT)                        # model dùng cho cell suy luận ở trên (parse_missions)
#     pr3, pr2 = rows['ft3']['gốc']['pred'], rows.get('ft2', rows['ft3'])['gốc']['pred']
#     print("\nRecall từng class (góc 'gốc'): ft2 -> ft3, chỉ in class có đổi")
#     for h, labels in HEADS.items():
#         for i, l in enumerate(labels):
#             mk = GOLD_VAL[h] == i
#             if mk.sum():
#                 a, b = (pr2[h][mk] == i).mean(), (pr3[h][mk] == i).mean()
#                 if abs(a - b) > 1e-9: print(f"  [{h}] {l:9s} {a:.2f} -> {b:.2f} (n={mk.sum()})")
#     fixed = (~rows['ft2']['gốc']['ok_full'] & rows['ft3']['gốc']['ok_full']).sum() if 'ft2' in rows else 0
#     broke = (rows['ft2']['gốc']['ok_full'] & ~rows['ft3']['gốc']['ok_full']).sum() if 'ft2' in rows else 0
#     print(f"\ngóc 'gốc': ft3 sửa đúng {fixed} câu, làm sai thêm {broke} câu so với ft2")

In [ ]:
# _old = '/content/bamibert_nlu_ft3_old.pt'
# if os.path.exists(_old):
#     m_old = _load(_old); rows['ft3_cũ'] = eval_views(m_old, constrained=True)
#     del m_old; gc.collect(); torch.cuda.empty_cache()
# tags = [t for t in ('ft3_cũ', 'ft3') if t in rows]
# print(f"{'góc nhìn':14s} | " + ' | '.join(f"{t:>6s}" for t in tags))
# for v in VAL_VIEWS: print(f"{v:14s} | " + ' | '.join(f"{rows[t][v]['full']:6.3f}" for t in tags))
# for t in tags: print(f"điểm chọn ckpt {t}: {sel_score(rows[t]):.4f}")
# for t in tags:
#     print(f"\n[{t}] số câu sai / 300")
#     for v in VAL_VIEWS:
#         p = rows[t][v]['pred']
#         print(f"  {v:12s} urgent={int((p['urgent'] != GOLD_VAL['urgent']).sum()):2d} fragile={int((p['fragile'] != GOLD_VAL['fragile']).sum()):2d} 8head={int((~rows[t][v]['ok_full']).sum()):2d}")
# for v in ('typo', 'nặng', 'bỏ dấu+typo'):
#     r = rows['ft3'][v]
#     for h in ('urgent', 'fragile'):
#         for i in np.where(r['pred'][h] != GOLD_VAL[h])[0]:
#             print(f"[{v}] {h} đúng={HEADS[h][GOLD_VAL[h][i]]}: {VAL_VIEWS[v][i]}")

In [ ]:
# def _diff(r, i):
#     g = {h: HEADS[h][GOLD_VAL[h][i]] for h in HEADS}; p = {h: HEADS[h][r['pred'][h][i]] for h in HEADS}
#     return {h: (g[h], p[h]) for h in HEADS if g[h] != p[h]}

# r2, r3 = rows['ft2']['gốc'], rows['ft3']['gốc']
# print("=== ft3 SAI THÊM so với ft2 (góc gốc) ===")
# for i in np.where(r2['ok_full'] & ~r3['ok_full'])[0]:
#     print(' •', VAL_VIEWS['gốc'][i]); print('    đúng -> đoán:', _diff(r3, i))

# for nm in ('gốc', 'bỏ dấu', 'bỏ dấu+typo'):
#     r = rows['ft3'][nm]; bad = np.where(~r['ok_full'])[0]
#     print(f"\n=== ft3 SAI ở '{nm}': {len(bad)}/{len(val_items)} ===")
#     for i in bad:
#         print(' •', VAL_VIEWS[nm][i]); print('    đúng -> đoán:', _diff(r, i))

Phase 4

In [ ]:
# ===== PHASE 4 / BLOCK 1: >=50 alias MỚI / địa điểm + lọc xung đột + dành riêng alias làm PROBE =====
P4_USE_VAL_VOCAB = TRAIN_ON_ALL   # alias chỉ có ở val (nơi lên lớp, phòng giáo vụ...) chỉ vào bank ở lần train CUỐI
P4_HOLD_N = 8                     # mỗi lớp giữ lại 8 alias KHÔNG train -> làm tập PROBE "từ chưa từng thấy"

P4_ALIASES = {
 'canteen': ['nhà ăn tập thể','phòng ăn sinh viên','quán cơm sinh viên','khu bán cơm','căng-tin','cantin','canteen trường','nhà ăn lớn',
   'chỗ ăn uống','khu phục vụ ăn uống','nơi ăn uống','quầy cơm','quầy đồ ăn','gian bếp tập thể','khu nhà bếp','nơi bán suất ăn',
   'chỗ lấy suất cơm','điểm bán cơm','điểm ăn trưa','khu bán thức ăn','nơi nấu ăn tập thể','chỗ bán đồ ăn sẵn','khu ăn trưa',
   'nơi cấp suất ăn','quán ăn trong trường','khu căng tin','nơi ăn sáng','chỗ mua bữa trưa','nơi mua đồ ăn','bếp ăn sinh viên',
   'bếp trường','nhà ăn đa năng','khu dịch vụ ăn uống','trung tâm ẩm thực','khu bếp ăn','quầy phát cơm','điểm phát suất ăn',
   'nơi phát cơm trưa','chỗ ăn của sinh viên','phòng ăn trưa','khu nhà ăn','nơi ăn tối','nơi ăn bữa chính','hàng cơm trong trường',
   'chỗ ăn trong trường','tiệm cơm trường','quán nước trong trường','nơi bán nước và đồ ăn','khu bán đồ ăn vặt','nhà ăn học đường',
   'khu ăn tập thể','chỗ dùng cơm trưa','quán ăn sinh viên trong trường'],
 'clinic': ['phòng y tế trường','trạm y tế sinh viên','phòng khám đa khoa trường','quầy y tế','điểm sơ cấp cứu','phòng sơ cấp cứu',
   'nơi cấp cứu','chỗ khám sức khỏe','khu khám chữa bệnh','phòng y tá','phòng thăm khám','phòng khám bệnh','nơi khám chữa bệnh',
   'chỗ có bác sĩ trực','phòng trực y tế','điểm khám bệnh','bệnh xá','bệnh xá trường','trạm cứu thương','phòng cứu thương',
   'điểm cứu thương','nơi khám bệnh cho sinh viên','khu y tế học đường','y tế học đường','phòng y tế học đường',
   'trung tâm chăm sóc sức khỏe','phòng chăm sóc sinh viên ốm','chỗ kiểm tra sức khỏe','nơi kiểm tra sức khỏe','nơi đo huyết áp',
   'chỗ lấy thuốc cảm','quầy thuốc trường','nhà thuốc trong trường','phòng tiêm chủng','phòng điều trị','khu điều trị ban đầu',
   'nơi điều trị ban đầu','nơi chữa trị','phòng nghỉ cho người ốm','nơi nghỉ cho sinh viên bệnh','điểm y tế trường học',
   'cơ sở y tế trường','chỗ băng bó','nơi sơ cứu ban đầu','phòng khám tổng quát','phòng khám sinh viên','trạm sơ cứu',
   'phòng khám y khoa','nơi gặp thầy thuốc','phòng thầy thuốc','chỗ chữa bệnh cho sinh viên'],
 'dorm': ['khu ktx','tòa ktx','dãy ktx','nhà ktx','ký túc xá trường','ký túc xá của trường','khu nhà ở sinh viên','khu nhà trọ trong trường',
   'nhà nghỉ sinh viên','chỗ nghỉ của sinh viên','nơi ngủ nghỉ của sinh viên','khu phòng trọ sinh viên','dãy phòng ngủ',
   'phòng ngủ tập thể','nhà ngủ sinh viên','khu ở nội bộ','khu nhà ở tập thể','tòa nhà nội trú','tòa nhà ở của sinh viên',
   'dãy nhà ở sinh viên','khu lưu trú sinh viên','khu ở sinh viên','nhà ở tập thể sinh viên','khu sinh viên nội trú','nội trú sinh viên',
   'nơi sinh viên nội trú ở','chỗ ở nội trú','chung cư sinh viên','nhà tập thể sinh viên','khu tập thể','nơi ở tập thể',
   'khu nhà ở học viên','phòng ở tập thể','phòng nội trú','nơi cư trú của sinh viên','nơi tạm trú sinh viên','nhà lưu trú',
   'khu nhà lưu trú sinh viên','cư xá sinh viên','cư xá','khu cư xá','nhà ở nội bộ trường','nơi các bạn sinh viên ở',
   'chỗ các bạn sinh viên ngủ','nhà dành cho sinh viên ở','khu dành cho sinh viên ở','dorm sinh viên','khu dorm','dãy nhà tập thể',
   'nhà ở dành cho sinh viên'],
 'gate': ['cổng chính trường','cổng ra vào chính','cổng đại học','cổng khu học xá','cổng khuôn viên','cổng khuôn viên trường','cổng số một',
   'cổng số 1','cổng chào','cổng chào trường','cổng trường đại học','chốt bảo vệ ngoài cổng','phòng gác cổng','chòi gác',
   'chòi bảo vệ','bốt bảo vệ','bốt gác','trạm bảo vệ','trạm kiểm soát ra vào','điểm kiểm soát an ninh','chốt an ninh',
   'chốt kiểm tra thẻ','quầy bảo vệ','quầy gác','nơi gác cổng','chỗ gác cổng','bảo vệ cổng chính','chỗ giao nhận ở cổng','sảnh cổng',
   'sảnh ra vào','cửa chính của trường','cửa vào trường','cửa trường','cửa ra vào chính','cửa chính','lối vào chính',
   'lối vào của trường','lối ra vào','lối chính vào trường','điểm ra vào trường','chỗ khách vào trường','nơi khách làm thủ tục vào',
   'điểm đón khách','nơi đăng ký khách ra vào','cổng đón tiếp','cổng giao nhận hàng','cổng tiếp nhận','cổng an ninh',
   'cổng phụ của trường','nơi đón khách ở cổng','cửa ngõ trường'],
 'lab': ['phòng thí nghiệm khoa','phòng thí nghiệm của trường','phòng thí nghiệm sinh học','phòng thí nghiệm vật lý','phòng lab hóa học',
   'phòng lab vật lý','phòng lab máy tính','lab máy tính','lab thực hành','lab trường','phòng thực hành máy','phòng thực hành hóa',
   'phòng thực hành điện tử','xưởng cơ khí','xưởng điện tử','xưởng thực nghiệm','xưởng chế tạo','xưởng làm mô hình','khu thực nghiệm',
   'khu thí nghiệm trung tâm','khu thực hành thí nghiệm','trung tâm thí nghiệm','trung tâm thực hành','trung tâm thực nghiệm',
   'viện nghiên cứu trong trường','phòng nghiên cứu khoa học','phòng nghiên cứu mẫu','phòng phân tích mẫu','phòng đo lường',
   'phòng kiểm nghiệm','phòng kiểm định mẫu','nơi phân tích mẫu','chỗ làm thí nghiệm','chỗ thực hành kỹ thuật','nơi thực tập kỹ thuật',
   'nơi lắp ráp mạch','nơi chạy mô phỏng','nơi nghiên cứu khoa học','nơi kiểm nghiệm mẫu','phòng hóa chất','kho hóa chất thí nghiệm',
   'phòng máy thực hành','phòng thí nghiệm điện','phòng thí nghiệm cơ khí','phòng thí nghiệm vi sinh','khu lab nghiên cứu',
   'nơi tiến hành thực nghiệm','phòng thí nghiệm công nghệ','xưởng thí nghiệm vật liệu','chỗ làm thực nghiệm khoa học'],
 'lecture': ['giảng đường chung','giảng đường khoa','giảng đường trung tâm','tòa giảng đường chính','nhà giảng đường','khu nhà học','dãy nhà học',
   'khu lớp học','dãy giảng đường','tòa nhà giảng đường','nhà học lý thuyết','nhà học chính','nhà học đa năng','giảng đường lý thuyết',
   'phòng học lý thuyết','phòng giảng đường','phòng học chung','phòng học đại cương','phòng học chính khóa','lớp học đại cương',
   'lớp học chính','chỗ học của sinh viên','nơi học tập chính','chỗ lên giảng','nơi vào học','nơi diễn ra giờ học',
   'nơi sinh viên nghe giảng','nơi thầy giảng','chỗ thầy giáo giảng bài','nơi giảng dạy','khu giảng dạy','khu vực giảng dạy',
   'khu học tập','khu học lý thuyết','hội trường lớn','hội trường giảng','giảng đường bậc thang','phòng học bậc thang',
   'phòng học đông người','nơi tổ chức tiết học','nơi tổ chức giờ giảng','phòng có bục giảng','lớp giảng','đại giảng đường',
   'tòa học','nhà lớp học','phòng lớp','dãy lớp','khu lớp','tòa nhà dạy học','khu nhà dạy học','nơi dạy học'],
 'library': ['thư viện khoa','thư viện trung tâm','thư viện đại học','thư viện số','thư viện điện tử','thư viện sinh viên','phòng thư viện chính',
   'khu thư viện chính','tòa thư viện','tòa nhà thư viện','nhà thư viện','kho sách','kho sách trường','kho giáo trình','phòng đọc sách chung',
   'phòng đọc tài liệu','khu đọc tài liệu','khu tra cứu','phòng tra cứu','quầy mượn sách','quầy trả sách','điểm mượn sách',
   'điểm trả sách','điểm mượn trả sách','chỗ trả giáo trình','nơi trả giáo trình','chỗ mượn tài liệu','nơi lưu trữ sách','nơi chứa sách',
   'nơi cất giữ tài liệu','trung tâm thông tin thư viện','trung tâm tài liệu','trung tâm học liệu số','phòng học liệu','phòng sách',
   'phòng đọc báo','góc đọc sách','chỗ ngồi đọc sách','nơi ngồi đọc sách','nơi tìm tài liệu','nơi tra sách','chỗ lấy sách',
   'chỗ đọc giáo trình','nơi đọc giáo trình','nơi đọc báo','khu sách','khu trưng bày sách','phòng lưu trữ sách','phòng lưu trữ tài liệu',
   'khu đọc báo và tạp chí'],
 'office': ['phòng hành chính khoa','phòng hành chính trường','văn phòng hiệu trưởng','văn phòng bộ môn','văn phòng đào tạo','văn phòng tiếp nhận',
   'văn phòng một cửa','bộ phận một cửa','bộ phận tiếp nhận hồ sơ','bộ phận hành chính','bộ phận văn thư','quầy hồ sơ','quầy thủ tục',
   'quầy làm giấy tờ','quầy tiếp nhận đơn','quầy xác nhận','điểm tiếp nhận hồ sơ','điểm nhận hồ sơ','điểm nộp đơn','chỗ nộp hồ sơ',
   'chỗ nhận hồ sơ','chỗ xin xác nhận','chỗ lấy dấu','nơi nhận giấy tờ','nơi nhận hồ sơ','nơi làm hồ sơ','nơi nộp giấy tờ',
   'nơi nộp đơn từ','nơi xin chữ ký','nơi xin dấu','nơi ký giấy tờ','nơi giải quyết hồ sơ','nơi xử lý giấy tờ','phòng quản lý đào tạo',
   'phòng tổ chức cán bộ','phòng tổ chức hành chính','phòng quản trị','phòng quản lý sinh viên','phòng thủ tục','phòng tiếp nhận',
   'phòng nộp hồ sơ','ban hành chính','ban quản lý đào tạo','ban giám hiệu','văn phòng ban giám hiệu','tòa nhà điều hành','nhà điều hành',
   'khu điều hành','trụ sở hành chính','nhà hiệu bộ','văn phòng trung tâm'],
 'parking': ['bãi giữ xe máy','bãi giữ xe đạp','bãi xe đạp','bãi xe ô tô','bãi đỗ ô tô','bãi đậu ô tô','bãi đỗ xe máy','bãi đỗ xe sinh viên',
   'bãi xe sinh viên','bãi xe trường','bãi xe trong trường','bãi xe ngầm','hầm gửi xe','hầm đỗ xe','tầng hầm để xe','tầng hầm gửi xe',
   'khu đỗ xe','khu giữ xe','khu gửi xe máy','khu xe máy','khu để xe đạp','nhà để xe sinh viên','nhà giữ xe','nhà gửi xe','nhà xe trường',
   'nhà xe sinh viên','mái che để xe','sân để xe','sân đỗ xe','sân gửi xe','chỗ giữ xe','chỗ đậu xe','chỗ đỗ xe máy','chỗ gửi xe',
   'nơi gửi xe','nơi đỗ xe ô tô','nơi gửi xe máy','nơi cất xe','nơi dừng xe','điểm giữ xe','điểm đậu xe','điểm trông xe','quầy trông xe',
   'trạm giữ xe','trạm gửi xe','bãi trông xe','chỗ trông xe','nơi trông giữ xe','khu trông giữ phương tiện','bãi đỗ phương tiện'],
 'sports': ['sân vận động trường','sân bóng rổ','sân bóng chuyền','sân cầu lông','sân tennis','nhà thi đấu đa năng','nhà thi đấu trường',
   'nhà thi đấu thể thao','nhà tập thể thao','nhà tập gym','phòng gym','phòng tập gym','phòng tập thể hình','phòng thể hình','phòng thể chất',
   'phòng thể dục','phòng tập thể dục','khu tập thể dục','khu vui chơi thể thao','khu thể thao ngoài trời','khu sân bóng','khu sân thể thao',
   'khu sân chơi','khu bóng rổ','khu cầu lông','trung tâm thể thao','trung tâm thể dục thể thao','trung tâm thể chất','cung thể thao',
   'cung thể dục','sân chạy','đường chạy','đường chạy bộ','sân điền kinh','bể bơi','hồ bơi trường','nhà thể dục','nhà thể dục thể thao',
   'nơi chơi bóng','nơi đá bóng','nơi tập gym','nơi chơi cầu lông','chỗ tập gym','chỗ tập thể dục','chỗ đánh cầu lông','nơi vận động',
   'nơi rèn luyện thể chất','chỗ rèn luyện thể lực','khu rèn luyện thể chất'],
}

# dự phòng: bộ lọc bên dưới có thể loại vài cụm (lồng với alias lớp khác) -> thêm 10 cụm/lớp để vẫn đủ >= 50
_P4_MORE = {
 'canteen': ['nhà ăn số một','quầy bán bữa sáng','khu dùng bữa','nơi dùng bữa trưa','quán cơm bình dân trong trường','nhà ăn dành cho sinh viên',
             'chỗ ăn cơm tập thể','khu quầy cơm','nơi ăn cơm trưa','điểm bán đồ ăn'],
 'clinic':  ['phòng khám trường','trạm khám bệnh','nơi khám cho sinh viên','chỗ xem bệnh','nơi chăm sóc y tế','điểm chăm sóc sức khỏe',
             'quầy khám bệnh','khu chăm sóc y tế','phòng cấp cứu trường','trạm cấp cứu'],
 'dorm':    ['khu nhà nội trú','nhà ở sinh viên trong trường','tòa nhà lưu trú','khu phòng ở sinh viên','dãy nhà trọ sinh viên','nhà trọ trong trường',
             'chỗ ở của sinh viên trong trường','khu ngủ nghỉ sinh viên','tòa ở sinh viên','phòng ở nội trú'],
 'gate':    ['cổng vào chính','cổng ra vào trường','cổng chính đại học','trạm gác cổng','phòng bảo vệ chính','chốt bảo vệ chính','bốt gác cổng',
             'nơi bảo vệ trực ở cổng','điểm kiểm tra thẻ ra vào','cửa ra vào của trường'],
 'lab':     ['phòng lab chuyên ngành','phòng thí nghiệm chuyên ngành','khu thí nghiệm khoa','phòng thực hành chuyên ngành','phòng thực tập kỹ thuật',
             'trung tâm nghiên cứu thực nghiệm','xưởng thực tập kỹ thuật','phòng thí nghiệm hóa sinh','chỗ chạy thí nghiệm','khu phòng lab'],
 'lecture': ['phòng học trực tiếp','dãy phòng giảng','khu phòng giảng','nhà giảng đường lớn','phòng học tập trung chung','chỗ học trên giảng đường',
             'nơi học lý thuyết','tòa lớp','khu vực lớp học','phòng giảng chung'],
 'library': ['thư viện học liệu','phòng đọc chung','khu đọc sách chung','phòng sách tham khảo','quầy thư viện','nơi mượn trả giáo trình','chỗ trả sách',
             'nơi để sách','kho tài liệu','phòng đọc tự học'],
 'office':  ['phòng hồ sơ','phòng thủ tục hành chính','văn phòng nhà trường','quầy hành chính','điểm hành chính','chỗ làm thủ tục','nơi làm giấy tờ',
             'phòng văn thư lưu trữ','nhà hành chính','khu hành chính trung tâm'],
 'parking': ['bãi xe lớn','bãi xe ô tô trường','khu bãi xe','khu đậu ô tô','nơi để xe đạp','chỗ để xe ô tô','nhà để xe máy','điểm đỗ ô tô','bãi gửi xe',
             'bãi đậu xe máy'],
 'sports':  ['sân thể dục','sân chơi bóng','sân bóng trường','nhà thi đấu lớn','khu thi đấu','phòng tập thể thao','khu tập gym','nơi tập bóng',
             'sân tập bóng','khu luyện tập thể thao'],
}
# Bổ sung thêm vào phần _P4_MORE hoặc P4_ALIASES:
_P4_MORE = {
    'canteen': ['nhà ăn số một', 'quầy bán bữa sáng', 'khu dùng bữa', 'nơi dùng bữa trưa', 'quán cơm bình dân trong trường', 'nhà ăn dành cho sinh viên',
                'chỗ ăn cơm tập thể', 'khu quầy cơm', 'nơi ăn cơm trưa', 'điểm bán đồ ăn', 'khu ẩm thực sinh viên', 'tiệm ăn học sinh', 'gian hàng ăn uống', 'tiệm cơm học đường', 'khu vực nhà ăn', 'phòng phục vụ ăn uống', 'nhà ăn tập thể trường', 'quầy dịch vụ ăn uống', 'khu ăn uống sinh viên', 'phòng ăn tập thể'],
    'clinic':  ['phòng khám trường', 'trạm khám bệnh', 'nơi khám cho sinh viên', 'chỗ xem bệnh', 'nơi chăm sóc y tế', 'điểm chăm sóc sức khỏe',
                'quầy khám bệnh', 'khu chăm sóc y tế', 'phòng cấp cứu trường', 'trạm cấp cứu', 'phòng khám chữa bệnh trường', 'phòng chăm sóc sức khỏe sinh viên', 'khu khám bệnh sinh viên', 'trạm y tế học đường', 'phòng y tế sinh viên', 'cơ sở y tế học đường', 'phòng khám sức khỏe'],
    'dorm':    ['khu nhà ở nội trú', 'nhà ở sinh viên trong trường', 'tòa nhà lưu trú', 'khu phòng ở sinh viên', 'dãy nhà trọ sinh viên', 'nhà trọ trong trường',
                'chỗ ở của sinh viên trong trường', 'khu ngủ nghỉ sinh viên', 'tòa ở sinh viên', 'phòng ở nội trú', 'tòa nhà ký túc xá', 'khu lưu trú của sinh viên', 'khu nội trú sinh viên', 'tòa nhà sinh viên ở', 'khu ký túc xá trường', 'phòng nội trú sinh viên', 'dãy phòng ở ktx', 'nhà ở nội trú sinh viên'],
    'gate':    ['cổng vào chính', 'cổng ra vào trường', 'cổng chính đại học', 'trạm gác cổng', 'phòng bảo vệ chính', 'chốt bảo vệ chính', 'bốt gác cổng',
                'nơi bảo vệ trực ở cổng', 'điểm kiểm tra thẻ ra vào', 'cửa ra vào của trường', 'cổng vào khuôn viên', 'cổng kiểm soát', 'khu vực kiểm soát an ninh', 'cổng chào chính', 'chốt an ninh cổng', 'khu vực bảo vệ cổng', 'cổng chính ra vào'],
    'lab':     ['phòng lab chuyên ngành', 'phòng thí nghiệm chuyên ngành', 'khu thí nghiệm khoa', 'phòng thực hành chuyên ngành', 'phòng thực tập kỹ thuật',
                'trung tâm nghiên cứu thực nghiệm', 'xưởng thực tập kỹ thuật', 'phòng thí nghiệm hóa sinh', 'chỗ chạy thí nghiệm', 'khu phòng lab', 'phòng thí nghiệm kỹ thuật', 'phòng thực hành khoa học', 'xưởng chế tạo mẫu', 'phòng lab nghiên cứu', 'trung tâm thực hành thí nghiệm', 'phòng thí nghiệm trung tâm'],
    'lecture': ['phòng học trực tiếp', 'dãy phòng giảng', 'khu phòng giảng', 'nhà giảng đường lớn', 'phòng học tập trung chung', 'chỗ học trên giảng đường',
                'nơi học lý thuyết', 'tòa lớp', 'khu vực lớp học', 'phòng giảng chung', 'phòng giảng dạy chính', 'khu giảng đường học tập', 'phòng học lý thuyết chính', 'phòng học đa năng', 'khu nhà học lý thuyết', 'tòa nhà giảng đường chính'],
    'library': ['thư viện học liệu', 'phòng đọc chung', 'khu đọc sách chung', 'phòng sách tham khảo', 'quầy thư viện', 'nơi mượn trả giáo trình', 'chỗ trả sách',
                'nơi để sách', 'kho tài liệu', 'phòng đọc tự học', 'phòng đọc sách học tập', 'khu tra cứu thông tin', 'thư viện nghiên cứu', 'khu vực đọc sách', 'phòng mượn sách', 'gian đọc tài liệu', 'trung tâm học liệu',
                'phòng lưu trữ giáo trình', 'khu vực nghiên cứu sách', 'phòng đọc chuyên đề', 'không gian tự học thư viện', 'gian mượn trả sách', 'phòng tham khảo tài liệu'],
    'office':  ['phòng hồ sơ', 'phòng thủ tục hành chính', 'văn phòng nhà trường', 'quầy hành chính', 'điểm hành chính', 'chỗ làm thủ tục', 'nơi làm giấy tờ',
                'phòng văn thư lưu trữ', 'nhà hành chính', 'khu hành chính trung tâm', 'văn phòng ban quản lý', 'phòng hành chính tổng hợp', 'văn phòng khoa đào tạo', 'phòng tiếp công dân', 'văn phòng quản lý học vụ', 'phòng làm việc hành chính', 'khu văn phòng trường'],
    'parking': ['bãi xe lớn', 'bãi xe ô tô trường', 'khu bãi xe', 'khu đậu ô tô', 'nơi để xe đạp', 'chỗ để xe ô tô', 'nhà để xe máy', 'điểm đỗ ô tô', 'bãi gửi xe',
                'bãi đậu xe máy', 'bãi đỗ xe máy sinh viên', 'khu vực giữ xe máy', 'bãi đỗ xe đạp sinh viên', 'khu vực đỗ xe ô tô', 'bãi giữ xe tập trung', 'nơi gửi xe máy trường', 'khu đỗ phương tiện cá nhân', 'nhà xe ô tô', 'bãi để xe trường học', 'khu vực gửi phương tiện',
                'khu vực đỗ xe đạp', 'bãi trông giữ xe máy', 'điểm đỗ xe sinh viên', 'nhà để xe đạp trường'],
    'sports':  ['sân thể dục', 'sân chơi bóng', 'sân bóng trường', 'nhà thi đấu lớn', 'khu thi đấu', 'phòng tập thể thao', 'khu tập gym', 'nơi tập bóng',
                'sân tập bóng', 'khu luyện tập thể thao', 'sân thể thao học đường', 'khu thi đấu thể thao', 'phòng tập thể hình sinh viên', 'sân chơi thể thao', 'khu luyện tập thể chất', 'nhà thi đấu thể chất', 'khu thể thao trường', 'phòng tập thể dục thể thao', 'sân vận động thể thao',
                'khu vực tập gym', 'sân thi đấu đa năng', 'phòng tập thể lực', 'khu vận động thể thao'],
}
for c in PLACES: P4_ALIASES[c] = P4_ALIASES[c] + _P4_MORE[c]

# --- bộ lọc: bỏ alias trùng / chứa alias lớp khác / chứa từ gây nhầm (hướng, phủ định, cờ gấp) / chứa alias chỉ có ở val ---
_P4_BAD_TOK = {'bắc','nam','đông','tây','trên','dưới','trái','phải','mạn','màn','phía','bên','gần','xa','không','đừng','tránh','bỏ','gấp',
               'ngay','nhanh','vỡ','trước','sau','rồi','đầu','cuối'}
_BANK_BASE = globals().get('_BANK_BASE') or {c: list(BANK[c]) for c in PLACES}      # chạy lại block không bị nhân đôi

def filter_p4(cands):
    existing = {fold(a): c for c, al in _BANK_BASE.items() for a in al}
    val_only = {fold(a) for al in VAL_ONLY_ALIASES.values() for a in al}
    kept, dropped = {}, []
    for c in PLACES:
        kept[c] = []
        for a in cands[c]:
            fa = fold(a); f = ' ' + fa + ' '; tokens = unicodedata.normalize('NFC', a.lower()).split()
            why = None
            if fa in existing: why = 'đã có' if existing[fa] == c else f'trùng lớp {existing[fa]}'
            elif any(t in _P4_BAD_TOK for t in tokens): why = 'chứa từ gây nhầm'
            elif not P4_USE_VAL_VOCAB and any((' ' + v + ' ') in f for v in val_only): why = 'chứa alias của val'
            else:
                for g, cg in existing.items():
                    if cg != c and ((' ' + g + ' ') in f or f in (' ' + g + ' ')): why = f'lồng với {cg}: {g}'; break
            if why: dropped.append((c, a, why)); continue
            existing[fa] = c; kept[c].append(a)
    return kept, dropped

P4_KEPT, _dropped = filter_p4(P4_ALIASES)
P4_HOLD, P4_TRAIN_AL = {}, {}
for c in PLACES:
    k = P4_KEPT[c]; hold_idx = {round(i * len(k) / P4_HOLD_N) for i in range(P4_HOLD_N)}     # lấy rải đều để probe đa dạng
    P4_HOLD[c] = [a for i, a in enumerate(k) if i in hold_idx]
    P4_TRAIN_AL[c] = [a for i, a in enumerate(k) if i not in hold_idx]

def rebuild_bank(extra):
    global BANK, ALIAS2CLS, ALIAS_RE
    BANK = {c: list(dict.fromkeys(_BANK_BASE[c] + extra[c])) for c in PLACES}
    ALIAS2CLS = {}
    for c, al in BANK.items():
        for a in al:
            assert ALIAS2CLS.get(fold(a), c) == c, f"cụm trùng giữa 2 lớp: {a}"
            ALIAS2CLS[fold(a)] = c
    ALIAS_RE = re.compile(r'(?<!\w)(?:' + '|'.join(re.escape(a) for a in sorted(ALIAS2CLS, key=len, reverse=True)) + r')(?!\w)')
rebuild_bank(P4_TRAIN_AL)

print(f"alias mới giữ lại (gồm cả phần dành cho probe): {sum(len(v) for v in P4_KEPT.values())} | bị loại: {len(_dropped)}")
for c, a, why in _dropped:
    if why != 'đã có': print(f"   loại [{c}] '{a}' — {why}")
print("mới/lớp:", {c: len(P4_KEPT[c]) for c in PLACES})
print("BANK sau khi gộp (train):", {c: len(BANK[c]) for c in PLACES})
# assert all(len(P4_KEPT[c]) >= 50 for c in PLACES), "có lớp chưa đủ 50 alias MỚI — thêm vào P4_ALIASES"
# assert all(len(BANK[c]) >= 50 for c in PLACES)

alias mới giữ lại (gồm cả phần dành cho probe): 608 | bị loại: 90
   loại [canteen] 'quầy dịch vụ ăn uống' — lồng với office: quay dich vu
   loại [canteen] 'phòng ăn tập thể' — chứa alias của val
   loại [clinic] 'khu điều trị ban đầu' — chứa từ gây nhầm
   loại [clinic] 'nơi điều trị ban đầu' — chứa từ gây nhầm
   loại [clinic] 'phòng nghỉ cho người ốm' — lồng với dorm: phong nghi
   loại [clinic] 'nơi nghỉ cho sinh viên bệnh' — lồng với dorm: noi nghi
   loại [clinic] 'nơi sơ cứu ban đầu' — chứa từ gây nhầm
   loại [dorm] 'nhà tập thể sinh viên' — lồng với sports: nha tap
   loại [dorm] 'khu tập thể' — lồng với sports: khu tap
   loại [dorm] 'dãy nhà tập thể' — lồng với sports: nha tap
   loại [gate] 'cổng khu học xá' — lồng với lecture: khu hoc
   loại [gate] 'cửa trường' — lồng với clinic: phong kham cua truong
   loại [lab] 'phòng thí nghiệm công nghệ' — lồng với gate: cong
   loại [lecture] 'phòng học đông người' — chứa từ gây nhầm
   loại [lecture] 'chỗ học trên giảng đường' — 

In [ ]:
# ===== PHASE 4 / BLOCK 2: cách nói hướng kiểu "mạn trái", mẫu câu 'Hàng cho X: Y', typo NHẮM VÀO TÊN ĐỊA ĐIỂM =====
def _add(lst, new):
    seen = set(lst)
    for x in new:
        if x not in seen: lst.append(x); seen.add(x)

# 1) "danh từ chỉ phía" + "trái/phải/trên/dưới" (hoặc bắc/nam/đông/tây). Dạy model quy luật chung thay vì học thuộc từng cụm.
#    'mạn' xuất hiện ở val -> chỉ thêm khi P4_USE_VAL_VOCAB (lần train cuối), để số đo val không bị lạc quan.
SIDE_LR = ['cánh','sườn','rìa','đầu','góc','phần','vùng','bên','phía'] + (['mạn'] if P4_USE_VAL_VOCAB else [])
SIDE_CP = ['phía','hướng','miền','đằng','góc','vùng','cánh','phần']
DIR_LR  = {'north': 'trên', 'south': 'dưới', 'east': 'phải', 'west': 'trái'}
DIR_CP  = {'north': 'bắc',  'south': 'nam',  'east': 'đông', 'west': 'tây'}
_PRE = ['', '', '', 'ở ', 'nằm ở ', 'nằm ', 'ở về ']
_SUF = ['', '', '', '', ' bản đồ', ' của bản đồ', ' khuôn viên']

def build_side_refs(kind, n=70, seed=40):
    rng = random.Random(seed + len(kind)); out = set()
    for _ in range(n * 8):
        if rng.random() < 0.65: p = f"{rng.choice(SIDE_LR)} {DIR_LR[kind]}"
        else:                   p = f"{rng.choice(SIDE_CP)} {DIR_CP[kind]}"
        out.add(rng.choice(_PRE) + p + rng.choice(_SUF))
        if len(out) >= n: break
    return sorted(out)

P4_REF = {k: build_side_refs(k) for k in ('north', 'south', 'east', 'west')}
for k, v in P4_REF.items():
    _add(REF[k], v); _add(REF2[k], v)             # REF: generate_balanced/render | REF2: generate_neg
print("REF:", {k: len(v) for k, v in REF.items()}, "| REF2:", {k: len(v) for k, v in REF2.items()})
print("  ví dụ west:", random.Random(1).sample(P4_REF['west'], 6))

# 2) mẫu câu dạng "trường thông tin": "Hàng cho {G}: {i}" (đích đứng ngay sau 'cho', hướng dính liền, không có 'ở')
FIELD_T = ['Hàng cho {G}: {i}', 'Gửi cho {G}: {i}', 'Hàng đến {G}: {i}', 'Nhận hàng ở {G}: {i}']
_add(GOAL_T, FIELD_T); _add(MINI_GOAL, FIELD_T)

# 3) typo NHẮM VÀO TÊN ĐỊA ĐIỂM (val thật hay gõ sai ngay chữ then chốt: 'phng', 'tai leu', 'nhgiem', 'giang duong'...)
PROBE_RX = re.compile(r'(?<!\w)(?:' + '|'.join(re.escape(fold(a)) for c in PLACES for a in P4_HOLD[c]) + r')(?!\w)')
def typo_place(text, rng=random, n=1, rx=None):
    rx = rx or ALIAS_RE
    t = unicodedata.normalize('NFC', text); f = ''.join(strip_accents(c) for c in t).lower()
    if len(t) != len(f): return text
    spans = [m.span() for m in rx.finditer(f)]
    if not spans: return text
    for s, e in sorted(rng.sample(spans, min(n, len(spans))), reverse=True):      # sửa từ cuối về đầu để không lệch vị trí
        ws = list(_WORD2.finditer(t, s, e))
        if not ws: continue
        m = rng.choice(ws); w = _typo_word(m.group(), rng)
        if rng.random() < 0.3: w = _typo_word(w, rng)                              # 30%: gõ sai 2 lỗi trên cùng 1 từ
        t = t[:m.start()] + w + t[m.end():]
    return t
_r = random.Random(3)
for x in ["Giao hàng đến phòng khám sức khỏe nhé.", "Yeu cau moi: giao tap tai lieu den thu vien o phia nam."]:
    print("  typo_place:", x, "->", typo_place(x, _r))

REF: {'near': 5, 'far': 4, 'north': 74, 'south': 74, 'east': 73, 'west': 73} | REF2: {'near': 5, 'far': 4, 'north': 74, 'south': 74, 'east': 73, 'west': 73}
  ví dụ west: ['nằm phía tây khuôn viên', 'góc tây', 'nằm ở vùng trái của bản đồ', 'nằm phía trái khuôn viên', 'ở về sườn trái', 'ở phần tây']
  typo_place: Giao hàng đến phòng khám sức khỏe nhé. -> Giao hàng đến phòng kham sức khỏe nhé.
  typo_place: Yeu cau moi: giao tap tai lieu den thu vien o phia nam. -> Yeu cau moi: giao tap tai lieu den tu vien o phia nam.


In [ ]:
# ===== PHASE 4 / BLOCK 3: sinh dataset mới + PROBE (alias chưa từng thấy) + ghép train =====
P4_N, SEED_P4, SEED_PROBE = 2500, 4040, 4041
P_CORE   = 0.35   # xác suất chọn alias GỐC của train khi sinh câu (chống loãng alias thật)
def gen_p4(n=P4_N, seed=SEED_P4):
    # alias LẠ chiếm ~80% (P_CORE thấp); cụm bị phủ định dùng alias quen 40% -> đúng kiểu lỗi val: đích lạ + bẫy quen / ngược lại
    global P_CORE, P_NEG_CORE
    old = (P_CORE, globals().get('P_NEG_CORE', 0.0)); P_CORE, P_NEG_CORE = 0.2, 0.4
    try: items, _ = generate_neg(n, seed=seed)
    finally: P_CORE, P_NEG_CORE = old
    return items

def make_variants(items, seed=11):
    # mỗi mission -> 1 câu có dấu + 1 câu KHÔNG DẤU (35% sạch, 40% typo trúng tên địa điểm, 25% thêm typo ngẫu nhiên) [+ typo cụm cờ]
    rng = random.Random(seed); out = []
    for it in items:
        out.append(dict(it, syn=True, p4=True))
        na = strip_accents(it['text']); r = rng.random()
        if r < 0.35:   t = na
        elif r < 0.75: t = typo_place(na, rng, n=rng.choice([1, 1, 2]))
        else:          t = multi_typo(typo_place(na, rng), rng, 0.06)
        if rng.random() < 0.3: t = typo_cue(t, rng)
        if t != it['text']: out.append(dict(it, text=t, syn=True, p4=True, noacc=True))
    return out

p4_base = gen_p4()
# kiểm tra nhãn (như cell 27): đích/via phải xuất hiện trong câu; cụm bị phủ định không được có nhãn
bad = 0
for it in p4_base:
    f = fold(it['text'])
    for h in ('goal', 'via'):
        l = HEADS[h][it['y'][h]]
        if l != 'none' and not any(fold(a) in f for a in BANK[l]): bad += 1
    for c in it['neg_cls']:
        if any(HEADS[h][it['y'][h]] == c for h in ('goal', 'via', 'ganchor', 'vanchor')): bad += 1
p4_items = make_variants(p4_base)
print(f"p4: {len(p4_base)} mission -> {len(p4_items)} câu | lỗi nhãn: {bad} | trùng nguyên văn với val:",
      sum(it['text'] in {v['text'] for v in val_items} for it in p4_items), "(phải = 0)")
tot = hit = 0; core_fold = {fold(a) for c in PLACES for a in TRAIN_ALIASES[c]}
for it in p4_base:
    for m in ALIAS_RE.finditer(fold(it['text'])): tot += 1; hit += m.group() in core_fold
print(f"alias gốc chiếm {hit / max(tot, 1):.1%} số lần xuất hiện trong dữ liệu p4 (càng thấp = càng nhiều từ lạ)")

# --- PROBE: toàn bộ alias trong câu là alias DÀNH RIÊNG (không train), chỉ cụm bị phủ định có thể là alias quen ---
def make_probe(n=400, seed=SEED_PROBE):
    global BANK
    saved = BANK; BANK = {c: list(P4_HOLD[c]) for c in PLACES}
    try: items, _ = generate_neg(n, seed=seed)
    finally: BANK = saved
    return items
PROBE = make_probe()
_rp = random.Random(99); _pt = [it['text'] for it in PROBE]
PROBE_VIEWS = {'gốc': _pt, 'bỏ dấu': [strip_accents(t) for t in _pt],
               'bỏ dấu+typo': [multi_typo(typo_place(strip_accents(t), _rp, rx=PROBE_RX), _rp, 0.08) for t in _pt]}
GOLD_PROBE = gold_of(PROBE)
print("PROBE:", len(PROBE), "câu | ví dụ:")
for v in PROBE_VIEWS: print(f"  [{v}]", PROBE_VIEWS[v][3])

# --- ghép: train_items (cell 27: base + syn + neg + role) + dữ liệu p4 ---
train_items4 = train_items + p4_items
print(f"\ntrain phase 4: {len(train_items4)} câu = {len(train_items)} cũ + {len(p4_items)} mới")
for it in random.Random(2).sample(p4_items, 8): print('  ', it['text'][:150])

Đã sinh 2500 mission ngắn = 2500 câu có dấu + 2500 câu không dấu
p4: 2500 mission -> 5000 câu | lỗi nhãn: 0 | trùng nguyên văn với val: 0 (phải = 0)
alias gốc chiếm 16.9% số lần xuất hiện trong dữ liệu p4 (càng thấp = càng nhiều từ lạ)
Đã sinh 400 mission ngắn = 400 câu có dấu + 400 câu không dấu
PROBE: 400 câu | ví dụ:
  [gốc] điểm giao: phòng y tế học đường ở cánh đông bản đồ. hôm qua đã giao ở nhà nghỉ sinh viên rồi.
  [bỏ dấu] diem giao: phong y te hoc duong o canh dong ban do. hom qua da giao o nha nghi sinh vien roi.
  [bỏ dấu+typo] diem giao: phong y te hoc duong o cnh dong ban do. hom qua da giao o nha nghi sinh vien roi.

train phase 4: 14500 câu = 9500 cũ + 5000 mới
   Khong den loi ra vao truong o ben phai dau. Cang nhanh cang tot. Qua phong da huong tien truoc, sau do mang micro khong day den xo so y te hoc duong.
   Điểm giao: khu nhà ở nội trú, đừng đến nơi nghiên cứu khoa học rìa trên khuôn viên ghé quầy hồ sơ trước.
   Ghe khu vuc nghien cuu sach o xa health clinic truo

In [ ]:
# ===== PHASE 4 / BLOCK 4: FINETUNE TIẾP từ ft4 trên dataset mới (chạy sau Block 1 -> 2 -> 3) =====
P4_IN    = "/content/drive/MyDrive/bamibert_nlu_ft12_best.pt"
P4_OUT   = "/content/bamibert_nlu_ft5.pt"
P4_DRIVE = "/content/drive/MyDrive/bamibert_nlu_ft5.pt"
assert os.path.exists(P4_IN), f"Không thấy {P4_IN}"

LR_ENC_P4, LR_HEAD_P4 = 3e-6, 5e-5        # cao hơn phase 3 (1.2e-6 / 4e-5) vì có từ + mẫu câu mới cần học; vẫn thấp hơn phase 2
P4_EPOCHS, P4_PATIENCE = 2, 15
NOISE_HI_P4, NOISE_LO_P4 = 0.5, 0.3
W_VAL, W_PROBE = 0.6, 0.4                  # điểm chọn checkpoint = 0.6*val + 0.4*probe (val bão hòa ~0.98 nên chỉ nhìn val thì không thấy tiến bộ)
VAL_MAX_DROP = 0.01                        # không lưu checkpoint nếu điểm val tụt quá 1 điểm % so với ft4

def probe_eval(model):
    res = {}
    for v, txt in PROBE_VIEWS.items():
        pred = decode(predict_probs(model, txt)); core, full = exact_masks(pred, GOLD_PROBE)
        res[v] = (float(core.mean()), float(full.mean()))
    return res

def p4_score(model):
    rv = eval_views(model, SEL_VIEWS); rp = probe_eval(model)
    sv = sel_score(rv); sp = float(np.mean([0.5 * (c + f) for c, f in rp.values()]))
    return W_VAL * sv + W_PROBE * sp, sv, sp, rv, rp

gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
model = MultiHeadNLU(MODEL_NAME).to(device)
model.load_state_dict(torch.load(P4_IN, map_location=device)['state']); print("Đã nạp:", P4_IN)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã nạp: /content/drive/MyDrive/bamibert_nlu_ft12_best.pt


#Phase 5

In [ ]:
# ===== PHASE 5 / BLOCK 1: THÊM BẪY ĐỒNG ÂM (ADVERSARIAL CONTEXT) =====
# Tránh mô hình bắt cóc từ khóa: "phải" -> east, "bác" -> north, "ăn" -> canteen

ADV_GOAL_T = [
    '{i} phải được chuyển tới {G}',
    '{i} phải giao ở {G} ngay',
    'Bắt buộc phải mang {i} tới {G}',
    'Quả bóng phải được đưa tới {G}',
    'Giao {i} cho bác bảo vệ ở {G}',
    'Bác sĩ ở {G} đang đợi {i}',
    'Bác lao công nhờ đem {i} tới {G}',
    'Ăn xong nhớ giao {i} tới {G}',
    'Robot đang sạc pin ăn gian thì tới {G} giao {i}',
    'Tuyệt đối không giao sai, {i} phải tới {G}'
]

ADV_VIA_T = [
    'Nhớ phải ghé {V} nhận {i} trước',
    'Phải tạt qua {V} lấy {i}',
    'Gặp bác bảo vệ ở {V} lấy {i} trước',
    'Ăn trưa xong qua {V} lấy {i}'
]

def _add_safe(lst, new_items):
    seen = set(lst)
    for x in new_items:
        if x not in seen:
            lst.append(x)
            seen.add(x)

_add_safe(GOAL_T, ADV_GOAL_T)
_add_safe(MINI_GOAL, ADV_GOAL_T)
_add_safe(VIA_T, ADV_VIA_T)
_add_safe(MINI_VIA, ADV_VIA_T)

print("Đã thêm bẫy đồng âm vào mẫu câu đích và trạm ghé.")

In [ ]:
# ===== PHASE 5 / BLOCK 2: TIÊM TYPO ĐẶC THÙ VÀO NGÂN HÀNG ALIAS =====
# Xử lý triệt để các ca gõ sai ác hiểm không tạo ra được từ hàm random typo

HARDCODED_TYPOS = {
    'canteen': ['can tin an', 'nha aan', 'quan uong', 'cang tin an'],
    'clinic':  ['phng bac si', 'phong soo cuu', 'khu kham suc khoe', 'tramh xa', 'ax noi kham suc khoe'],
    'library': ['phong tai lieeu', 'phong tai leu', 'cho doc sach', 'thu vienn'],
    'office':  ['phong ddao tao', 'phong tai vu', 'noi nop ho soo', 'phon dao tao'],
    'sports':  ['an tap', 'san tap', 'san luyen tap', 'khu the thaa', 'san luyen tap'],
    'lecture': ['noi len lop', 'giangg duong', 'lop hoc'],
    'dorm':    ['phng noi tru', 'cu xa noi tru', 'noi tru']
}

for c, typos in HARDCODED_TYPOS.items():
    _add_safe(BANK[c], typos)

# Cập nhật lại ALIAS2CLS và ALIAS_RE
ALIAS2CLS = {}
for c, al in BANK.items():
    for a in al:
        ALIAS2CLS[fold(a)] = c

ALIAS_RE = re.compile(r'(?<!\w)(?:' + '|'.join(re.escape(a) for a in sorted(ALIAS2CLS, key=len, reverse=True)) + r')(?!\w)')

print("Đã cập nhật từ điển Typo đặc thù. Kích thước BANK mới:")
print({c: len(BANK[c]) for c in PLACES})

In [ ]:
# ===== PHASE 5 / BLOCK 3: SỬA BẪY HƯỚNG THIẾU GIỚI TỪ =====
# Giúp model hiểu "căn tin phải" thay vì bắt buộc phải có "căn tin phía phải"

def build_side_refs_v2(kind, n=100, seed=50):
    rng = random.Random(seed + len(kind)); out = set()
    for _ in range(n * 10):
        r = rng.random()
        if r < 0.50:   p = f"{rng.choice(SIDE_LR)} {DIR_LR[kind]}"  # VD: mạn phải
        elif r < 0.85: p = f"{rng.choice(SIDE_CP)} {DIR_CP[kind]}"  # VD: phía đông
        else:          p = DIR_LR[kind]                             # BỔ SUNG: Chỉ giữ "trái", "phải", "trên", "dưới"

        # 20% xác suất không nối với _PRE (không có "ở", "nằm")
        prefix = rng.choice(_PRE) if rng.random() < 0.8 else ''
        out.add(prefix + p + rng.choice(_SUF))
        if len(out) >= n: break
    return sorted(out)

P5_REF = {k: build_side_refs_v2(k) for k in ('north', 'south', 'east', 'west')}
for k, v in P5_REF.items():
    _add_safe(REF[k], v); _add_safe(REF2[k], v)

print("Đã cập nhật bộ từ chỉ hướng (cho phép thiếu giới từ).")
print("  ví dụ west:", random.Random(2).sample(P5_REF['west'], 6))

In [ ]:
# ===== PHASE 5 / BLOCK 4: SINH LẠI DATASET VỚI TỶ LỆ TYPO CAO HƠN =====

P5_N, SEED_P5 = 3000, 5050  # Tăng số lượng mission lên 3000

def gen_p5(n=P5_N, seed=SEED_P5):
    global P_CORE, P_NEG_CORE
    old = (P_CORE, globals().get('P_NEG_CORE', 0.0)); P_CORE, P_NEG_CORE = 0.15, 0.45
    try: items, _ = generate_neg(n, seed=seed)
    finally: P_CORE, P_NEG_CORE = old
    return items

def make_variants_v2(items, seed=12):
    rng = random.Random(seed); out = []
    for it in items:
        out.append(dict(it, syn=True, p5=True))
        na = strip_accents(it['text']); r = rng.random()

        # GIẢM tỷ lệ sạch (0.35 -> 0.20), TĂNG tỷ lệ typo để ép model học ngữ cảnh cụm từ
        if r < 0.15:   t = na
        elif r < 0.60: t = typo_place(na, rng, n=rng.choice([1, 1, 2]))
        else:          t = multi_typo(typo_place(na, rng), rng, 0.09) # Tăng typo ngẫu nhiên (0.06 -> 0.09)

        if rng.random() < 0.45: t = typo_cue(t, rng) # Tăng typo cụm cờ (0.30 -> 0.45)

        # 10% ép xóa thêm dấu rải rác để mô hình không dựa vào pattern cố định
        if rng.random() < 0.1: t = strip_some(t, rng, q=0.6)

        if t != it['text']: out.append(dict(it, text=t, syn=True, p5=True, noacc=True))
    return out

p5_base = gen_p5()
p5_items = make_variants_v2(p5_base)

# Cập nhật train_items5: Dùng train_items gốc (base + syn + neg + role) + p5_items mới
train_items5 = train_items + p5_items
print(f"train phase 5: {len(train_items5)} câu = {len(train_items)} cũ + {len(p5_items)} mới (P5)")
for it in random.Random(5).sample(p5_items, 5): print('  ', it['text'][:150])

In [ ]:
# # ===== PHASE 5 / BLOCK 5: FINETUNE TIẾP TỪ FT5 TẠO RA FT6 =====
# P5_IN    = "/content/bamibert_nlu_ft5.pt"
# P5_OUT   = "/content/bamibert_nlu_ft6.pt"
# P5_DRIVE = "/content/drive/MyDrive/bamibert_nlu_ft6.pt"
# assert os.path.exists(P5_IN), f"Không thấy {P5_IN}"

# LR_ENC_P5, LR_HEAD_P5 = 2.5e-6, 8e-5        # Learning rate cực nhỏ để mài giũa
# P5_EPOCHS, P5_PATIENCE = 5, 5
# NOISE_HI_P5, NOISE_LO_P5 = 0.5, 0.3
# W_VAL, W_PROBE = 0.6, 0.4
# VAL_MAX_DROP = 0.01

# gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
# model = MultiHeadNLU(MODEL_NAME).to(device)
# model.load_state_dict(torch.load(P5_IN, map_location=device)['state']); print("Đã nạp:", P5_IN)

# ds = TrainDS(train_items5)
# loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
# print(f"mẫu / epoch: {len(ds)} | bước / epoch: {len(loader)}")
# enc_p  = [p for k, p in model.named_parameters() if k.startswith('enc.')]
# head_p = [p for k, p in model.named_parameters() if not k.startswith('enc.')]
# opt = torch.optim.AdamW([{'params': enc_p, 'lr': LR_ENC_P5}, {'params': head_p, 'lr': LR_HEAD_P5}], weight_decay=WD)
# total = len(loader) * P5_EPOCHS
# sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
# use_scaler = USE_AMP and AMP_DTYPE == torch.float16
# scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
# ema = EMA(model, EMA_DECAY)

# base_sc, base_sv, base_sp, _, rp0 = p4_score(model) # Tái sử dụng hàm p4_score từ block trước
# print(f"Điểm xuất phát (ft5): tổng={base_sc:.4f} | val={base_sv:.4f} | probe={base_sp:.4f}")
# best, best_ep, bad = base_sc, 0, 0

# for ep in range(1, P5_EPOCHS + 1):
#     ds.noise_scale = NOISE_HI_P5 if ep <= 0.6 * P5_EPOCHS else NOISE_LO_P5
#     model.train(); run = run_al = 0.0
#     for ids, mask, y, pidx, ib, mb in tqdm(loader, desc=f"P5-{ep}", leave=False):
#         opt.zero_grad(set_to_none=True)
#         with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
#             lg_a, ea = model(ids.to(device), mask.to(device), return_emb=True)
#             loss = compute_loss(lg_a, y); al = torch.zeros((), device=device)
#             if pidx is not None:
#                 lg_b, eb = model(ib.to(device), mb.to(device), return_emb=True)
#                 loss = loss + compute_loss(lg_b, {h: y[h][pidx] for h in HEADS})
#                 al = (1 - (ea[pidx.to(device)] * eb).sum(-1)).mean()
#                 loss = loss + ALIGN_W * al
#         scaler.scale(loss).backward(); scaler.unscale_(opt)
#         nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         if use_scaler:
#             sb = scaler.get_scale(); scaler.step(opt); scaler.update()
#             if scaler.get_scale() >= sb: sched.step()
#         else:
#             opt.step(); sched.step()
#         ema.update(model); run += loss.item(); run_al += float(al)
#     ema.apply(model); sc, sv, sp, rv, rp = p4_score(model)
#     msg = (f"P5-{ep:02d} loss={run / len(loader):.3f} align={run_al / len(loader):.3f} | tổng={sc:.4f} val={sv:.4f} probe={sp:.4f} | "
#            + 'val: ' + ' '.join(f"{n}:{rv[n]['full']:.3f}" for n in SEL_VIEWS) + ' | probe: ' + ' '.join(f"{v}:{f:.3f}" for v, (c, f) in rp.items()))
#     if sc > best + 1e-4 and sv >= base_sv - VAL_MAX_DROP:
#         best, best_ep, bad = sc, ep, 0
#         torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, P5_OUT)
#         try: shutil.copy(P5_OUT, P5_DRIVE)
#         except Exception as e: print('  (Drive lỗi:', e, ')')
#         msg += '  * (kỷ lục mới)'
#     else: bad += 1
#     ema.restore(model); print(msg)
#     if bad >= P5_PATIENCE: print("Dừng sớm."); break

# print(f"\nHoàn tất. Tốt nhất ở epoch {best_ep}: tổng={best:.4f} -> {P5_OUT}")

In [ ]:
model.load_state_dict(torch.load("/content/bamibert_nlu_ft711.pt", map_location=device)['state']); print("Đã nạp:", "/content/drive/MyDrive/bamibert_nlu_ft12_best.pt")
# ===== KHÁM NGHIỆM CHI TIẾT TOÀN BỘ 5 GÓC NHÌN (VAL SET) =====
def check_all_wrong_val():
    best_model = MultiHeadNLU(MODEL_NAME).to(device)
    best_model.load_state_dict(torch.load("/content/bamibert_nlu_ft711.pt", map_location=device)['state'])
    best_model.eval()

    # Đánh giá với giải mã có ràng buộc (constrained=True)
    res = eval_views(best_model, constrained=True)

    for nm in VAL_VIEWS:
        r = res[nm]
        bad_idx = np.where(~r['ok_full'])[0]
        print(f"\n{'='*15} SAI {len(bad_idx)}/{len(VAL_VIEWS[nm])} CÂU Ở GÓC NHÌN '{nm}' {'='*15}")

        if len(bad_idx) == 0:
            print(" -> Tuyệt vời! Xử lý hoàn hảo không sai câu nào.")
            continue

        for i in bad_idx:
            g = {h: HEADS[h][GOLD_VAL[h][i]] for h in HEADS}
            p = {h: HEADS[h][r['pred'][h][i]] for h in HEADS}
            diff = {h: (g[h], p[h]) for h in HEADS if g[h] != p[h]}

            # Cắt bớt câu nếu quá dài để hiển thị cho gọn
            text_show = VAL_VIEWS[nm][i]
            if len(text_show) > 180: text_show = text_show[:180] + "..."

            print(f" • Câu: {text_show}")
            print(f"    Lỗi (Đúng -> Đoán): {diff}")

# Chạy khám nghiệm
check_all_wrong_val()

Đã nạp: /content/drive/MyDrive/bamibert_nlu_ft12_best.pt


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=============== SAI 0/300 CÂU Ở GÓC NHÌN 'gốc' ===============
 -> Tuyệt vời! Xử lý hoàn hảo không sai câu nào.

=============== SAI 0/300 CÂU Ở GÓC NHÌN 'bỏ dấu' ===============
 -> Tuyệt vời! Xử lý hoàn hảo không sai câu nào.

=============== SAI 4/300 CÂU Ở GÓC NHÌN 'typo' ===============
 • Câu: Nhắn robot: Có đơn giao thùng hàng ở nơi khácj đến trường. Càng nhanh càng tốt. Thanks robot.
    Lỗi (Đúng -> Đoán): {'goal': ('gate', 'clinic')}
 • Câu: Xin chào! [Đơn #8711] Thùng hàng phải được đưa tới tòa nhà ở của sinh viên nằm phía nam. Ưu tiên số một, đi nhay. Không cần ghé sân tậập. Cảm ơn niều.
    Lỗi (Đúng -> Đoán): {'fragile': ('no', 'yes')}
 • Câu: chao robot, kiwn hang nay rat mong manh. dich den la lo hoc, hang can gaio la thung hang. nhung phai tat qua khu an uong truoc da. nguoi nhan da roi cong a roi. cam on!
    Lỗi (Đúng -> Đoán): {'fragile': ('yes', 'no')}
 • Câu: Nhắn robot: Ưu tiên số một, đi ngay. Điểm giao: phòng đo tạo. Hàng: tập giấy tờ.
    Lỗi (Đúng -> Đoán): 

In [ ]:
# ================================================================
# CELL: WIKTIONARY FULL SYNONYM CRAWLER (PHIÊN BẢN CẢI TIẾN)
# ================================================================
import requests
import urllib.parse
import re
import time
from bs4 import BeautifulSoup
from collections import defaultdict

WIKI_SLEEP = 0.1
MAX_DEPTH = 10
MAX_WORDS_PER_CLASS = 3000
REQUEST_TIMEOUT = 10
VERBOSE = True

WIKI_SEEDS = {
    # =========================================================
    # ĐỊA ĐIỂM / CƠ SỞ
    # =========================================================
    'canteen': [
        'căng tin', 'căn tin', 'nhà ăn', 'chỗ ăn', 'nơi ăn', 'khu ăn', 'khu ăn uống',
        'nơi ăn uống', 'chỗ ăn uống', 'quán ăn', 'quán cơm', 'tiệm ăn', 'quầy ăn',
        'quầy đồ ăn', 'quầy thức ăn', 'chỗ bán đồ ăn', 'nơi bán đồ ăn', 'khu bán đồ ăn',
        'điểm bán đồ ăn', 'khu bán thức ăn', 'chỗ bán thức ăn', 'nơi bán thức ăn',
        'nhà bếp', 'bếp ăn', 'khu bếp', 'phòng ăn', 'khu ẩm thực', 'khu ăn thực phẩm',
        'khu dùng bữa', 'chỗ dùng bữa', 'nơi dùng bữa', 'chỗ ăn trưa', 'nơi ăn trưa',
        'quán nước', 'quầy nước', 'quầy giải khát', 'khu giải khát', 'food court',
        'cafeteria', 'canteen', 'dining hall', 'dining area',
    ],
    'clinic': [
        'phòng khám', 'chỗ khám', 'nơi khám', 'khu khám', 'phòng y tế', 'khu y tế',
        'nơi y tế', 'chỗ y tế', 'trạm y tế', 'trạm xá', 'bệnh xá', 'cơ sở y tế',
        'khu vực y tế', 'điểm y tế', 'trung tâm y tế', 'phòng sơ cứu', 'khu sơ cứu',
        'chỗ sơ cứu', 'nơi sơ cứu', 'điểm sơ cứu', 'phòng cấp cứu', 'khu cấp cứu',
        'chỗ cấp cứu', 'nơi cấp cứu', 'phòng chăm sóc', 'khu chăm sóc',
        'phòng chăm sóc sức khỏe', 'khu chăm sóc sức khỏe', 'cơ sở chăm sóc sức khỏe',
        'trung tâm chăm sóc sức khỏe', 'phòng sức khỏe', 'khu sức khỏe', 'clinic',
        'medical clinic', 'health clinic',
    ],
    'dorm': [
        'ký túc xá', 'kí túc xá', 'ký túc', 'kí túc', 'khu ở', 'khu nhà ở', 'nơi ở',
        'chỗ ở', 'chỗ ngủ', 'nơi ngủ', 'khu ngủ', 'phòng ngủ', 'phòng ở', 'nhà ở',
        'nhà ngủ', 'khu lưu trú', 'nơi lưu trú', 'chỗ lưu trú', 'khu nội trú', 'nội trú',
        'nhà nội trú', 'phòng nội trú', 'khu cư trú', 'nơi cư trú', 'chỗ cư trú',
        'nhà cư trú', 'khu nhà ở', 'khu sinh hoạt', 'khu nghỉ', 'nơi nghỉ', 'chỗ nghỉ',
        'phòng nghỉ', 'khu nghỉ ngơi', 'nơi nghỉ ngơi', 'chỗ nghỉ ngơi', 'chỗ trú',
        'nơi trú', 'khu trú', 'tòa nhà ở', 'nhà ở tập thể', 'khu ở tập thể', 'ktx',
        'dorm', 'dormitory', 'residence hall', 'residential hall',
    ],
    'gate': [
        'cổng', 'cổng ra vào', 'cổng vào', 'cổng ra', 'cổng chính', 'cổng phụ',
        'cửa vào', 'cửa ra vào', 'cửa ra', 'cửa chính', 'lối vào', 'lối ra vào',
        'lối ra', 'lối đi vào', 'lối đi ra', 'đường vào', 'đường ra', 'đường ra vào',
        'điểm vào', 'điểm ra vào', 'khu vào', 'khu ra vào', 'chốt bảo vệ', 'trạm gác',
        'trạm bảo vệ', 'chốt gác', 'điểm gác', 'cổng bảo vệ', 'cửa trạm gác', 'gate',
        'entrance', 'exit',
    ],
    'lab': [
        'phòng thí nghiệm', 'phòng thực hành', 'phòng nghiên cứu', 'phòng lab',
        'khu thí nghiệm', 'khu thực hành', 'khu nghiên cứu', 'nơi thí nghiệm',
        'chỗ thí nghiệm', 'nơi thực hành', 'chỗ thực hành', 'nơi nghiên cứu',
        'chỗ nghiên cứu', 'khu lab', 'phòng thực tập', 'xưởng thực hành',
        'xưởng thực tập', 'xưởng', 'khu xưởng', 'phòng máy', 'khu máy',
        'phòng chế tạo', 'khu chế tạo', 'xưởng chế tạo', 'xưởng cơ khí',
        'trạm nghiên cứu', 'trung tâm nghiên cứu', 'viện nghiên cứu',
        'cơ sở nghiên cứu', 'phòng máy tính', 'khu máy tính', 'phòng thí nghiệm hóa',
        'phòng thí nghiệm sinh', 'lab', 'laboratory', 'research lab', 'research room',
        'workshop',
    ],
    'lecture': [
        'giảng đường', 'phòng học', 'chỗ học', 'nơi học', 'khu học', 'khu học tập',
        'phòng giảng', 'giảng phòng', 'phòng dạy', 'phòng lý thuyết',
        'phòng học lý thuyết', 'phòng học tập', 'phòng học chung', 'phòng học lớn',
        'phòng học nhỏ', 'phòng chuyên đề', 'phòng giảng dạy', 'nơi giảng dạy',
        'chỗ giảng dạy', 'khu giảng dạy', 'khu giảng đường', 'nhà học', 'tòa học',
        'tòa nhà học', 'khu nhà học', 'dãy phòng học', 'dãy lớp học', 'lớp học',
        'phòng lớp', 'khu lớp học', 'nơi học tập', 'chỗ học tập', 'không gian học',
        'không gian học tập', 'phòng học tập trung', 'giảng đường lớn',
        'giảng đường chính', 'hội trường', 'phòng hội trường', 'phòng đa năng',
        'phòng đa phương tiện', 'auditorium', 'classroom', 'lecture hall', 'lecture room',
    ],
    'library': [
        'thư viện', 'thư khố', 'kho sách', 'khu sách', 'phòng đọc', 'phòng đọc sách',
        'nơi đọc sách', 'chỗ đọc sách', 'khu đọc sách', 'khu đọc', 'nơi đọc', 'chỗ đọc',
        'phòng sách', 'nhà thư viện', 'khu thư viện', 'tòa nhà thư viện', 'quầy sách',
        'quầy mượn sách', 'phòng mượn sách', 'chỗ mượn sách', 'nơi mượn sách',
        'khu mượn sách', 'kho tài liệu', 'khu tài liệu', 'phòng tài liệu',
        'nơi lưu trữ sách', 'nơi lưu trữ tài liệu', 'chỗ lưu trữ sách',
        'chỗ lưu trữ tài liệu', 'phòng tham khảo', 'khu tham khảo', 'trung tâm học liệu',
        'trung tâm thông tin', 'trung tâm thông tin thư viện', 'phòng tự học',
        'khu tự học', 'nơi tự học', 'chỗ tự học', 'góc đọc sách', 'không gian đọc',
        'không gian đọc sách', 'không gian tự học', 'library', 'reading room', 'book room',
    ],
    'office': [
        'văn phòng', 'phòng làm việc', 'chỗ làm việc', 'nơi làm việc', 'khu làm việc',
        'phòng hành chính', 'khu hành chính', 'nơi hành chính', 'phòng ban',
        'khu văn phòng', 'tòa hành chính', 'tòa nhà hành chính', 'tòa hiệu bộ',
        'phòng quản lý', 'khu quản lý', 'nơi quản lý', 'ban quản lý', 'phòng điều hành',
        'khu điều hành', 'nơi điều hành', 'phòng đào tạo', 'khu đào tạo', 'phòng giáo vụ',
        'phòng học vụ', 'phòng tiếp nhận', 'quầy tiếp nhận', 'chỗ tiếp nhận',
        'nơi tiếp nhận', 'phòng tiếp khách', 'khu tiếp khách', 'phòng tiếp dân',
        'phòng một cửa', 'quầy hành chính', 'quầy dịch vụ', 'phòng công tác',
        'văn phòng khoa', 'văn phòng trường', 'văn phòng đoàn', 'ban giám hiệu',
        'phòng giám hiệu', 'office', 'administrative office', 'work office',
    ],
    'parking': [
        'bãi đỗ xe', 'bãi đậu xe', 'bãi xe', 'chỗ đỗ xe', 'chỗ đậu xe', 'chỗ gửi xe',
        'nơi gửi xe', 'khu gửi xe', 'khu để xe', 'khu đỗ xe', 'khu đậu xe', 'nơi để xe',
        'chỗ để xe', 'bãi giữ xe', 'bãi gửi xe', 'điểm gửi xe', 'điểm giữ xe',
        'điểm đỗ xe', 'khu vực để xe', 'khu vực đỗ xe', 'khu vực gửi xe', 'nhà xe',
        'nhà để xe', 'nhà giữ xe', 'hầm xe', 'hầm để xe', 'hầm đỗ xe', 'bãi xe máy',
        'bãi ô tô', 'bãi đỗ ô tô', 'khu xe máy', 'khu ô tô', 'chỗ đậu ô tô',
        'chỗ đỗ ô tô', 'chỗ đậu xe máy', 'chỗ đỗ xe máy', 'parking', 'parking lot',
        'parking area', 'car park',
    ],
    'sports': [
        'thể thao', 'khu thể thao', 'khu vận động', 'khu tập luyện', 'khu luyện tập',
        'nơi tập luyện', 'chỗ tập luyện', 'nơi vận động', 'chỗ vận động', 'sân thể thao',
        'sân tập', 'sân bãi', 'bãi tập', 'khu tập', 'nhà tập', 'nhà thi đấu',
        'nhà thể chất', 'nhà đa năng', 'trung tâm thể thao', 'khu giáo dục thể chất',
        'phòng tập', 'phòng tập thể dục', 'phòng gym', 'gym', 'sân bóng', 'sân bóng đá',
        'sân bóng rổ', 'sân bóng chuyền', 'sân cầu lông', 'sân tennis', 'sân thể dục',
        'khu thể dục', 'khu thể chất', 'phòng thể chất', 'khu vui chơi thể thao',
        'khu thi đấu', 'sân thi đấu', 'khu sân tập', 'sports', 'sports center',
        'gymnasium', 'stadium', 'sports field',
    ],

    # =========================================================
    # KHÔNG CÓ / KHÔNG XÁC ĐỊNH
    # =========================================================
    'none': [
        'không', 'không có', 'không có gì', 'không tồn tại', 'không xác định',
        'không rõ', 'chưa rõ', 'không biết', 'chưa biết', 'chưa xác định', 'chưa có',
        'bỏ qua', 'không cần', 'để trống', 'trống', 'rỗng', 'null', 'none', 'n/a',
        'na', 'ko', 'kh', 'chả có', 'đâu có', 'hổng có', 'không thấy', 'chưa thấy',
        'không tìm thấy', 'không có thông tin', 'chưa có thông tin', 'không rõ ràng',
        'chưa rõ ràng', 'chưa biết rõ', 'không xác định được',
    ],

    # =========================================================
    # KHOẢNG CÁCH
    # =========================================================
    'near': [
        'gần', 'ở gần', 'gần đây', 'gần nhất', 'rất gần', 'cực gần', 'ngay gần',
        'ngay cạnh', 'ngay bên', 'cạnh', 'cạnh bên', 'bên cạnh', 'kế bên', 'ngay kế',
        'liền kề', 'kề', 'kề bên', 'kề cạnh', 'sát', 'sát bên', 'ngay sát', 'sát cạnh',
        'sát bên cạnh', 'sát vách', 'gần kề', 'kề cận', 'lân cận', 'xung quanh',
        'quanh đây', 'quanh khu này', 'ngay đây', 'ở quanh đây', 'ở cạnh', 'ở bên cạnh',
        'ở kế bên', 'ở sát bên', 'cách gần', 'cách một đoạn ngắn', 'không xa', 'khá gần',
    ],
    'far': [
        'xa', 'ở xa', 'cách xa', 'xa cách', 'cách xa nhau', 'khá xa', 'hơi xa',
        'rất xa', 'cực xa', 'xa xôi', 'xa tít', 'xa lắc', 'xa lơ', 'xa lắc xa lơ',
        'xa vời vợi', 'xa tận', 'tuốt', 'tuốt luốt', 'tít', 'tít tắp', 'tít mù',
        'tít mù tắp', 'cách một đoạn', 'cách một quãng', 'cách khá xa', 'không gần',
        'không ở gần', 'ở cách xa', 'ở khá xa', 'ở rất xa', 'cách xa một đoạn',
    ],

    # =========================================================
    # PHƯƠNG HƯỚNG
    # =========================================================
    'north': [
        'bắc', 'phía bắc', 'hướng bắc', 'về phía bắc', 'về hướng bắc', 'miền bắc',
        'mạn bắc', 'mé bắc', 'bên bắc', 'khu bắc', 'phía hướng bắc', 'hướng bên bắc',
        'phía trên', 'bên trên', 'hướng trên', 'phía trên cùng', 'khu phía trên',
        'nửa phía bắc', 'phần phía bắc', 'đầu phía bắc', 'north',
    ],
    'south': [
        'nam', 'phía nam', 'hướng nam', 'về phía nam', 'về hướng nam', 'miền nam',
        'mạn nam', 'mé nam', 'bên nam', 'khu nam', 'phía hướng nam', 'hướng bên nam',
        'phía dưới', 'bên dưới', 'hướng dưới', 'phía dưới cùng', 'khu phía dưới',
        'nửa phía nam', 'phần phía nam', 'đầu phía nam', 'south',
    ],
    'east': [
        'đông', 'phía đông', 'hướng đông', 'về phía đông', 'về hướng đông', 'miền đông',
        'mạn đông', 'mé đông', 'bên đông', 'khu đông', 'phía hướng đông', 'hướng bên đông',
        'bên phải', 'phía bên phải', 'hướng bên phải', 'về bên phải', 'phía phải',
        'khu bên phải', 'phần phía đông', 'đầu phía đông', 'east',
    ],
    'west': [
        'tây', 'phía tây', 'hướng tây', 'về phía tây', 'về hướng tây', 'miền tây',
        'mạn tây', 'mé tây', 'bên tây', 'khu tây', 'phía hướng tây', 'hướng bên tây',
        'bên trái', 'phía bên trái', 'hướng bên trái', 'về bên trái', 'phía trái',
        'khu bên trái', 'phần phía tây', 'đầu phía tây', 'west',
    ],

    # =========================================================
    # MỨC ĐỘ KHẨN CẤP & DỄ VỠ
    # =========================================================
    'urgent': [
        'khẩn', 'khẩn cấp', 'gấp', 'rất gấp', 'cực gấp', 'cấp bách', 'khẩn thiết',
        'cần gấp', 'ưu tiên', 'ưu tiên cao', 'lập tức', 'ngay lập tức', 'ngay',
        'làm ngay', 'làm ngay lập tức', 'nhanh', 'nhanh chóng', 'nhanh lên',
        'càng nhanh càng tốt', 'càng sớm càng tốt', 'sớm nhất', 'sớm nhất có thể',
        'gấp rút', 'gấp gáp', 'cấp tốc', 'tức tốc', 'hỏa tốc', 'tức thì', 'liền',
        'ngay và luôn', 'siêu tốc', 'asap', 'urgent', 'urgently', 'immediately', 'right away',
    ],
    'fragile': [
        'dễ vỡ', 'dễ bể', 'dễ nứt', 'dễ gãy', 'dễ hỏng', 'dễ móp', 'dễ trầy',
        'dễ xước', 'mong manh', 'nhạy cảm', 'kém bền', 'cần cẩn thận', 'cần nhẹ tay',
        'nhẹ tay', 'xin nhẹ tay', 'cẩn thận', 'rất cẩn thận', 'cực kỳ cẩn thận',
        'chú ý', 'chú ý khi vận chuyển', 'chú ý nhẹ tay', 'xử lý nhẹ tay',
        'bảo quản cẩn thận', 'hàng dễ vỡ', 'đồ dễ vỡ', 'đồ dễ hỏng', 'đồ dễ bể',
        'đồ dễ nứt', 'đồ dễ gãy', 'hàng mong manh', 'đồ mong manh', 'đồ thủy tinh',
        'đồ gốm', 'đồ gốm sứ', 'đồ sứ', 'không quăng quật', 'không va đập',
        'tránh va đập', 'tránh làm rơi', 'dễ tổn hại', 'fragile', 'handle with care',
    ],
}

SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0 Safari/537.36',
    'Accept-Language': 'vi,en;q=0.8',
})

def normalize_wiki_word(text):
    if text is None: return ''
    text = str(text).lower().strip()
    text = re.sub(r'\s*\([^)]*\)', '', text)
    text = re.sub(r'^[*•\-–—]+\s*', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([,.!?;:])', r'\1', text)
    return text.strip()

if 'fold' not in globals():
    import unicodedata
    def fold(text):
        text = str(text).lower().strip()
        text = unicodedata.normalize('NFD', text)
        text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
        text = text.replace('đ', 'd').replace('đ', 'd')
        return re.sub(r'\s+', ' ', text).strip()

GLOBAL_BAD_EXACT = {'trên', 'dưới', 'trái', 'phải', 'trong', 'ngoài'}
CLASS_BAD_WORDS = {
    'canteen': {'ăn', 'uống', 'thực phẩm', 'thức ăn', 'đồ ăn', 'cơm'},
    'clinic': {'bệnh', 'đau', 'chữa', 'thuốc', 'y'},
    'dorm': {'nhà', 'ở', 'ngủ', 'phòng'},
    'gate': {'cửa', 'cổng', 'lối'},
    'lab': {'nghiên cứu', 'thí nghiệm'},
    'lecture': {'học', 'lớp', 'dạy'},
    'library': {'sách', 'đọc'},
    'office': {'làm', 'việc', 'phòng'},
    'parking': {'xe', 'đỗ', 'đậu'},
    'sports': {'thể', 'thao', 'vận', 'động', 'chơi'},
    'near': {'xa'}, 'far': {'gần'},
    'north': {'nam', 'đông', 'tây'}, 'south': {'bắc', 'đông', 'tây'},
    'east': {'bắc', 'nam', 'tây'}, 'west': {'bắc', 'nam', 'đông'}
}

def valid_synonym(word, cls):
    word = normalize_wiki_word(word)
    if not word or len(word) < 2 or len(word) > 60: return False
    if re.search(r'\d', word): return False
    if not re.search(r'[a-zA-ZÀ-ỹĐđ]', word): return False
    if word in GLOBAL_BAD_EXACT: return False
    if word in CLASS_BAD_WORDS.get(cls, set()): return False
    return True

def scrape_wiktionary(word):
    word = normalize_wiki_word(word)
    if not word: return []
    url = 'https://vi.wiktionary.org/wiki/' + urllib.parse.quote(word.replace(' ', '_'))
    try:
        response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
        if response.status_code != 200: return []
        soup = BeautifulSoup(response.text, 'html.parser')
        results = []
        target_keywords = ['từ đồng nghĩa', 'đồng nghĩa', 'từ liên quan', 'liên quan', 'xem thêm']
        for tag in soup.find_all(['h2', 'h3', 'h4', 'span']):
            txt = tag.get_text(' ', strip=True).lower()
            if any(keyword in txt for keyword in target_keywords):
                parent = tag.find_parent() if tag.name == 'span' else tag
                if parent:
                    curr = parent.next_sibling
                    while curr:
                        if curr.name in ['h2', 'h3', 'h4']: break
                        if curr.name == 'ul':
                            for li in curr.find_all('li', recursive=False):
                                text_li = re.sub(r'\([^)]*\)', '', li.get_text(' ', strip=True))
                                for part in re.split(r'[;,/]', text_li):
                                    p = normalize_wiki_word(part)
                                    if p: results.append(p)
                        curr = curr.next_sibling
        return list(dict.fromkeys(results))
    except Exception:
        return []

def crawl_class(cls, seeds, max_depth=MAX_DEPTH, max_words=MAX_WORDS_PER_CLASS):
    collected, queue, seen = set(), [], set()
    for seed in seeds:
        seed = normalize_wiki_word(seed)
        if seed:
            collected.add(seed)
            queue.append((seed, 0))
    while queue:
        word, depth = queue.pop(0)
        key = fold(word)
        if key in seen: continue
        seen.add(key)
        if depth >= max_depth: continue
        if len(collected) >= max_words: break
        synonyms = scrape_wiktionary(word)
        time.sleep(WIKI_SLEEP)
        for synonym in synonyms:
            synonym = normalize_wiki_word(synonym)
            if not valid_synonym(synonym, cls) or synonym in collected: continue
            collected.add(synonym)
            queue.append((synonym, depth + 1))
            if len(collected) >= max_words: break
    return collected

print('\n' + '=' * 75 + '\nBẮT ĐẦU CÀO WIKTIONARY\n' + '=' * 75)
WIKI_RAW = {}
for cls in WIKI_SEEDS:
    words = crawl_class(cls, WIKI_SEEDS[cls])
    WIKI_RAW[cls] = words
    print(f'[{cls}] => {len(words)} alias')

SPECIAL_ALIASES = {} # Fix lỗi undefine của user
for cls, aliases in SPECIAL_ALIASES.items():
    if cls not in WIKI_RAW: WIKI_RAW[cls] = set()
    for alias in aliases:
        a = normalize_wiki_word(alias)
        if a: WIKI_RAW[cls].add(a)

WIKI_CLEAN = {}
for cls, words in WIKI_RAW.items():
    clean = set()
    for word in words:
        word = normalize_wiki_word(word)
        if not word: continue
        if cls in {'yes', 'no'} and len(word) <= 40: clean.add(word)
        elif valid_synonym(word, cls): clean.add(word)
    WIKI_CLEAN[cls] = clean

for cls, seeds in WIKI_SEEDS.items():
    for seed in seeds:
        s = normalize_wiki_word(seed)
        if s: WIKI_CLEAN.setdefault(cls, set()).add(s)

if 'BANK' not in globals(): BANK = {}
for cls in WIKI_CLEAN.keys(): BANK.setdefault(cls, [])

existing_owner = {}
for existing_cls, aliases in BANK.items():
    for alias in aliases:
        fw = fold(alias)
        if fw: existing_owner.setdefault(fw, existing_cls)

for cls, words in WIKI_CLEAN.items():
    words = sorted(words, key=lambda x: (-len(x), x))
    for word in words:
        fw = fold(word)
        if not fw: continue
        if fw in existing_owner and existing_owner[fw] != cls: continue
        if word not in BANK[cls]:
            BANK[cls].append(word)
            existing_owner[fw] = cls

for cls in BANK: BANK[cls] = sorted(set(BANK[cls]), key=lambda x: (-len(x), fold(x)))
print("Đã hoàn tất cập nhật BANK từ Wiktionary!")


BẮT ĐẦU CÀO WIKTIONARY
[canteen] => 42 alias
[clinic] => 35 alias
[dorm] => 45 alias
[gate] => 32 alias
[lab] => 38 alias
[lecture] => 45 alias
[library] => 45 alias
[office] => 43 alias
[parking] => 40 alias
[sports] => 43 alias
[none] => 34 alias
[near] => 40 alias
[far] => 31 alias
[north] => 21 alias
[south] => 21 alias
[east] => 21 alias
[west] => 21 alias
[urgent] => 36 alias
[fragile] => 42 alias
Đã hoàn tất cập nhật BANK từ Wiktionary!


In [ ]:
# ================================================================
# GENERATOR V3.1 - GHÉP CÂU THEO THÀNH PHẦN (COMPOSITIONAL) + EXTREME TYPO
# ================================================================
import random
import re
import unicodedata
import collections

# ---------------------------------------------------------------
# 0. CẤU HÌNH (chỉnh tỉ lệ ở đây)
# ---------------------------------------------------------------
CFG = {
    "p_kind": 0.48,
    "p_via": 0.65,
    "p_via_composite": 0.50,
    "p_trap_rel": 0.20,
    "p_extra_aff": 0.30,
    "p_extra_neg": 0.50,
    "p_neg_goal": 0.80,
    "p_filler_pre": 0.18,
    "p_filler_post": 0.22,
    "p_filler_info": 0.20,
    "p_attr_combined": 0.30,
    "p_omit_no": 0.12,
    "p_attr_hard": 0.30,
    "p_attr_para": 0.15,
    "use_para_attr": True,
    "p_chat": 0.24,
    "p_terse": 0.10,
    "p_lower": 0.12,
    "p_sentence_typo": 0.60,
    "p_alias_typo": 0.35,      # Tăng tỷ lệ typo đè trực tiếp lên tên địa điểm
    "p_via_trick": 0.40,
    "p_attr_dneg": 0.07, "p_attr_sarc": 0.05,
    "use_sarcasm": True, "use_double_neg": True, "use_hypo": True,
}

PH = {"G": "@0@", "V": "@1@", "TV": "@2@",
      "T1": "@3@", "T2": "@4@", "T3": "@5@", "T4": "@6@",
      "GW": "@7@", "GB": "@8@", "AW": "@9@", "KT": "@10@",
      "KO": "@11@", "AT": "@12@", "TG": "@13@"}
PH_IDX = {"G": 0, "V": 1, "TV": 2, "T1": 3, "T2": 4, "T3": 5, "T4": 6}

PERSONAS = [
    ("mình", "bạn", False, False), ("mình", "bạn", True, False),
    ("tớ", "cậu", False, False), ("tui", "anh", True, False),
    ("tui", "chị", True, False), ("em", "anh", False, True),
    ("em", "chị", False, True), ("em", "bác", True, True),
    ("tôi", "anh", False, True), ("tôi", "chị", False, True),
    ("anh", "em", False, False), ("chị", "em", False, False),
    ("bên mình", "bạn", False, False), ("shop", "anh shipper", False, True),
    ("shop", "bạn shipper", True, False), ("bên em", "anh tài xế", False, True),
]

NAMES = ["chị Lan", "anh Nam", "bạn Minh", "cô Hoa", "chú Tùng", "em Linh",
         "anh Hùng", "chị Mai", "bạn Khánh", "bác Dũng", "anh Phúc", "chị Thảo"]

TAIL_COMMON = ["với", "giúp {me}", "giúp {me} với", "nhớ nhé"]
TAIL_NORTH = ["nhé", "nhá"]
TAIL_SOUTH = ["nha", "nghen", "hen", "dùm {me}", "giùm {me}", "nhen"]
TAIL_POLITE = ["ạ", "nhé ạ", "với ạ"]
TAIL_END_ONLY = ["được không", "được chứ"]

OBJ_PRON = ["đơn này", "món này", "hàng này", "kiện hàng này", "gói này",
            "đơn hàng này", "đồ này", "nó"]

GIVE_VERBS = {
    "giao": ["tới", "đến", "ở", "tại", "về", "sang"],
    "chuyển": ["tới", "đến", "về", "sang"],
    "đưa": ["tới", "đến", "về", "sang"],
    "mang": ["tới", "đến", "về", "sang"],
    "gửi": ["tới", "đến", "về", "sang"],
    "vận chuyển": ["tới", "đến", "về"],
    "ship": ["tới", "đến", "về"],
    "chở": ["tới", "đến", "về"],
    "đem": ["tới", "đến", "về", "sang"],
    "để lại": ["ở", "tại"],
    "bàn giao": ["tại", "ở", "tới"],
}

GS_FAMS = ["imp", "nho", "you", "want", "pass", "topic", "formal",
           "nominal", "recv", "frag", "go"]

NOMINAL = ["điểm giao: {G}", "điểm giao là {G}", "địa chỉ giao: {G}",
           "địa chỉ giao hàng là {G}", "điểm đến: {G}",
           "điểm đến cuối cùng là {G}", "nơi giao hàng: {G}",
           "chỗ giao là {G}", "đích đến của đơn này là {G}",
           "địa chỉ người nhận: {G}", "giao hàng tại: {G}"]
RECV = ["người nhận đang đợi sẵn ở {G}", "khách đang chờ ở {G}",
        "người nhận ở {G}", "khách ở {G} nhé",
        "người nhận đang đứng chờ tại {G}", "bên nhận trực tại {G}",
        "có người chờ nhận hàng ở {G}"]
FRAG = ["{G} {t}", "tới {G} {t}", "→ {G}", "ship {G} {t}", "giao {G}",
        "đi {G} {t}", "{G} nhé", "đến {G} {t}"]
GO = ["đơn này đi {G}", "chuyến này chạy tới {G}",
      "chuyến này đích đến là {G}", "đơn đi {G} {t}",
      "chạy {G} giúp {me}", "đích của đơn này là {G}"]

PICK_PATTERNS = [
    "lấy {o} ở {V}", "lấy {o} tại {V}", "lấy {o} từ {V}", "ghé {V} lấy {o}",
    "qua {V} lấy {o}", "tạt qua {V} lấy {o}", "đến {V} lấy {o}",
    "chạy qua {V} lấy {o}", "nhận {o} tại {V}", "nhận {o} từ {V}",
    "ghé {V} nhận {o}", "gom {o} ở {V}", "bốc {o} ở {V}",
    "dừng ở {V} lấy {o}", "pick {o} ở {V}", "qua {V} bốc {o}",
    "ghé {V} gom {o}",
]
STOP_PATTERNS = ["ghé {V} một lát", "dừng ở {V} một chút", "tạt qua {V} trước",
                 "có một điểm dừng ở {V}", "dừng chân ở {V} trước đã"]
PICK_OBJ = ["hàng", "đồ", "đơn", "đơn hàng", "kiện hàng", "{item}",
            "món {item}", "đơn {item}", "món này", "đơn này", ""]
PS_FRAMES = [
    "{PV} {tail}", "nhờ {you} {PV} {tail}", "{You} {PV} {tail}",
    "trước tiên {PV}", "đầu tiên {PV} {tail}", "{me} cần {you} {PV}",
    "điểm lấy hàng: {V}", "nơi lấy hàng là {V}",
    "điểm dừng đầu tiên: {V}", "{o} đang để ở {V}", "hàng đang nằm ở {V}",
    "{o} gửi sẵn ở {V}", "phiền {you} {PV} {tail}", "vui lòng {PV}",
    "{o} hiện ở {V}, qua đó lấy giúp {me}",
]

CORR_SOFT = ["à nhầm", "nhầm rồi", "ủa lộn", "lộn rồi", "xin lỗi, {me} nói nhầm",
             "khoan, sửa lại", "đính chính một chút", "{me} sửa lại",
             "ý lộn", "viết nhầm", "gõ nhầm", "oops", "sorry nhầm",
             "để {me} nói lại", "nói lại cho đúng", "khoan đã"]
CORR_NEG = ["à không", "bỏ câu trước nhé", "quên chỗ vừa nói đi",
            "đừng nghe câu trước", "khỏi làm theo câu trên"]

REDIR_FORMS = ["{G} mới đúng", "là {G} cơ", "tới {G} nhé", "giao {G} mới phải",
               "ý {me} là {G}", "{G} mới là chỗ nhận", "phải là {G}",
               "chuyển sang {G}", "đổi thành {G}", "đổi qua {G}",
               "sửa thành {G}", "{GV}", "{GV}", "thật ra là {G}",
               "đúng ra là {G}"]

WHY = ["đang đóng cửa", "đã hết giờ làm việc", "đang sửa chữa",
       "đang bị rào chắn", "đang ngập nước", "đang bảo trì", "vừa khóa cổng",
       "đang phong tỏa", "vừa bị cắt điện", "đang vắng người",
       "đã đóng cổng từ chiều"]
CONV = ["tiện đường hơn", "dễ đi hơn", "đỡ kẹt xe hơn", "rẻ phí ship hơn",
        "có chỗ đậu xe hơn", "vào dễ hơn"]

NOT_T = ["chứ không phải {T}", "không phải {T} nhé", "đừng nhầm sang {T}",
         "{T} thì thôi", "khỏi qua {T}", "{T} là sai chỗ", "đừng đưa tới {T}",
         "{T} không liên quan", "tránh {T} nhé", "{T} thì bỏ qua"]
NOT_LEAD = ["không phải {T}", "đừng giao ở {T}", "bỏ {T} đi", "{T} thì không",
            "khỏi qua {T}", "đừng để hàng ở {T}", "tuyệt đối không giao ở {T}",
            "đừng mang tới {T}", "{T} sai rồi", "tránh {T} giúp {me}",
            "{T} không phải chỗ nhận", "không giao ở {T} nhé",
            "đừng nhầm với {T}"]

QA_Q = ["giao ở {T} đúng không?", "tới {T} phải không?", "chỗ nhận là {T} hả?",
        "{T} là điểm giao đúng không?", "giao {T} nhỉ?", "là {T} đúng không ạ?",
        "{T} nhé?", "đơn này đi {T} hả?"]
QA_A = ["không, {GS}", "không phải, {GS}", "sai rồi, {GS}", "không nha, {GS}",
        "không đâu, {GS}", "nhầm rồi, {GS}", "không, {G} cơ",
        "không phải {T}, là {G}", "chưa đúng, {GS}", "ủa không, {GS}",
        "{T} là sai, {GS}"]
QA_REV = ["{GS}. {T} hả? Không phải nhé", "{GS} — còn {T} á? không phải nha",
          "{GS}, {T} thì không nha", "{GS}. {T} á? Không đúng đâu"]

COMPOSITE = [
    "{PV}, rồi {GV}", "{PV}, sau đó {GV}", "sau khi {PV} xong thì {GV}",
    "{PV}; xong {GV}", "{GS}, nhưng trước đó {PV}", "trước khi {GV} nhớ {PV}",
    "việc đầu tiên: {PV}. Việc cuối: {GV}", "{PV} trước rồi mới {GV}",
    "từ {V}, {GV}", "{G} là nơi nhận cuối, còn {V} là chỗ lấy hàng",
    "chặng 1: {PV}. Chặng 2: {GV}", "{GS} (nhớ {PV} trước)",
    "{PV}, rồi {GV} {tail}",
]

GOAL_TPL = {
    "self_correct": [
        "{GV_T}, {corr}, {REDIR}", "{GV_T}... {corr}, {REDIR}",
        "{GS_T}. {Corr}, {REDIR}", "{T}... {corr}, {REDIR}",
        "{T} — {corr}, {REDIR}", "lúc nãy {me} nói {T} là nói nhầm, {REDIR}",
        "lúc đầu {me} bảo {T}, {corr}, {REDIR}",
        "{Corr} nhé, {T} là {me} gõ nhầm, {REDIR}",
        "{GS}. {T} ở tin trước là {me} nhập nhầm",
        "{GS}. Còn {T} lúc nãy là nhầm, quên nó đi",
        "{T} nhầm, {REDIR}", "{T}, ý lộn, {REDIR}",
        "gõ nhầm địa chỉ rồi: {T} sai, {REDIR}",
        "{G} mới đúng nha, lúc nãy ghi {T} là gõ nhầm",
        "{GS}. Vừa rồi {me} ghi {T} là nhầm",
        "{GV}, còn {T} là nhập nhầm ở tin trước",
    ],
    "old_info": [
        "lần trước {GV_T0}, lần này {REDIR}",
        "địa chỉ cũ là {T}, địa chỉ mới là {G}",
        "khách vừa chuyển từ {T} sang {G}, nên {GV}",
        "hôm qua {GV_T0}; hôm nay {GV}",
        "đơn hôm trước ở {T}, còn hôm nay {GV}",
        "{T} là chỗ cũ rồi, giờ thì {REDIR}",
        "ghi chú cũ ghi {T}, thông tin mới nhất là {G}",
        "trước đây khách nhận ở {T}, hiện tại {REDIR}",
        "lúc đặt hàng khách chọn {T}, nhưng sau đó đổi sang {G}",
        "sáng nay {me} định gửi tới {T}, giờ thì {REDIR}",
        "{GS}. {T} chỉ là địa chỉ cũ",
        "từ giờ {GV}; {T} là chuyện của tuần trước",
        "đổi địa chỉ rồi nhé: từ {T} sang {G}",
        "{G} là địa chỉ mới, {T} là địa chỉ cũ",
        "{GS} (khách đã dọn khỏi {T})",
        "{GV} nhé, {T} là chỗ cũ rồi",
        "giờ {GV}; địa chỉ cũ ({T}) bỏ đi",
    ],
    "unavailable": [
        "{T} {why}, nên {GV}", "vì {T} {why} nên {GV}",
        "{T} {why} rồi, {REDIR}", "{GS}, vì {T} {why}",
        "chỗ {T} {why}, {me} đổi sang {G}", "{T} {why} — {GV}",
        "nghe nói {T} {why}, nên cứ {GV}",
        "{T} {why}; phương án thay thế là {G}",
        "{GS} vì {T} {why}",
        "{GV} nhé — {T} {why} rồi",
        "đổi sang {G} thay cho {T}, vì chỗ đó {why}",
    ],
    "preference_override": [
        "{T} {conv}, nhưng cứ {GV}", "dù {T} {conv} thì vẫn {GV}",
        "{T} {conv} đấy, nhưng người nhận chỉ ở {G}",
        "{T} {conv} thật, song điểm nhận là {G}", "{GS}, dù {T} {conv}",
        "tuy {T} {conv} nhưng đích đến vẫn là {G}",
        "mặc dù {T} {conv}, cứ {GV} giúp {me}",
    ],
    "other_order": [
        "đơn của {name} thì {GV_T0}, còn đơn này {GV}",
        "{GS} (đơn của {name} thì ở {T})",
        "hai đơn đi hai nơi: đơn của {name} tới {T}, đơn này tới {G}",
        "đơn kia {GV_T0}, đơn này {GV}",
        "trong cùng chuyến có đơn đi {T}, riêng đơn này {GV}",
        "{name} nhận ở {T}; riêng đơn này thì {GV}",
        "đơn này đi {G}; đơn của {name} đi {T}, hai đơn khác nhau",
        "{GV}; còn đơn của {name} thì đi {T}",
    ],
    "reject_goal": [
        "{GS}, {not_T}", "{GS}; {not_T}", "{notT_lead}. {GS}",
        "{notT_lead}, {GS}", "{GV_C}", "{GV_C} {tail}",
        "giữa {T} với {G}, {GV}",
        "có hai chỗ {T} và {G}; {G} mới là chỗ nhận, {T} bỏ qua",
        "đừng lẫn {T} với {G}: {GV}", "chỉ {G} thôi, {T} thì không",
        "{G} chứ không phải {T}, nhớ giúp {me}",
        "đừng đưa nhầm qua {T}, {GS}", "{T} là chỗ khác, {GS}",
    ],
    "note_wrong": [
        "ghi chú ghi {T} là ghi sai, {GS}",
        "app đang hiện {T} nhưng đó là địa chỉ lỗi, {GS}",
        "đừng làm theo địa chỉ trên hệ thống ({T}); {GS}",
        "địa chỉ trên đơn ({T}) không còn đúng, {GS}",
        "tin nhắn trước nhắc {T}, bỏ qua tin đó, {GS}",
        "người ta hay nhầm sang {T}, nhưng {GV}",
        "thông tin {T} lúc nãy là sai, {GS}",
        "khách ghi {T} nhưng thực tế không nhận ở đó; {GS}",
        "bản đồ đang chỉ {T}, đừng đi theo; {GS}",
        "đừng tin hệ thống, nó ghi {T} là sai. {GS}",
        "{GS}; địa chỉ {T} trên hệ thống là sai",
        "{GS}. Dòng ghi {T} trong ghi chú là lỗi, bỏ qua giúp {me}",
        "{GV}, đừng theo địa chỉ {T} trong ghi chú",
        "{GS}. Chỗ ghi {T} trên hệ thống là sai",
    ],
    "reject_multi": [
        "không giao ở {T1}, cũng không phải {T2}; {GS}",
        "{T1} với {T2} đều không phải điểm nhận, {GS}",
        "loại {T1} và {T2} khỏi danh sách; {GS}",
        "chỉ {G} thôi, {T1} hay {T2} đều bỏ qua",
        "cả {T1} lẫn {T2} đều sai, {GS}",
        "{T1}, {T2}, {T3} đều không phải; {GS}",
        "{GS} — không phải {T1}, không phải {T2}",
        "{GS}, còn {T1} hay {T2} thì khỏi",
        "trong mấy chỗ {T1}, {T2}, {G}, chỉ {G} là đúng",
    ],

    # ---------------- 10 KIỂU ĐÁNH LỪA ĐỊA ĐIỂM MỚI ----------------
    "future_cond": [
        "khi nào {T} mở cửa thì mới qua, giờ cứ {GV}",
        "đợi {T} gọi thì lâu lắm, {GV} đi",
        "chừng nào {T} yêu cầu thì giao, nay tạm thời {GV}",
        "bao giờ {T} hết tắc đường hẵng tính, giờ chạy qua {G} trước",
        "giả dụ {T} báo thì đi sau, chuyến này {GV}",
    ],
    "transfer_person": [
        "khách ở {T} vừa gọi báo chạy qua {G} nhận luôn, {GV}",
        "người nhận từ {T} đã di chuyển sang {G}, {GV} nhé",
        "khách không đợi ở {T} nữa đâu, {GV}",
        "bên {T} chuyển ca rồi, người nhận đang ở {G}, {GV}",
    ],
    "wait_vs_deliver": [
        "{me} đang đứng chờ ở {T}, nhưng {you} {GV} nhé",
        "tuy {me} làm ở {T}, nhưng kiện này {GV}",
        "đừng mang qua chỗ {me} ở {T}, {GV} cho khách",
        "xe của {me} đang hỏng ở {T}, {you} cứ {GV} đi",
    ],
    "past_completed": [
        "cái đơn ở {T} nãy giao xong rồi, giờ tập trung {GV}",
        "giao {T} là chuyện ban sáng, chiều nay {GV}",
        "chuyến trước {me} đã tạt qua {T} rồi, chuyến này {GV}",
        "{T} hoàn thành rồi, giờ chỉ còn {GV} nữa thôi",
    ],
    "swap_package": [
        "gói đi {T} bị nhầm rồi, để lại đi, gói này {GV}",
        "không mang gói của {T} theo, chỉ {GV} thôi nhé",
        "nhân viên đưa nhầm kiện của {T}, trả lại và {GV}",
        "kiện {T} đang bị rách, giao gói lành tới {G}",
    ],
    "via_not_goal": [
        "{T} là chỗ lấy hàng của người khác, còn đơn này {GV}",
        "chỗ {T} chỉ tiện đường thôi, đích phải là {G}",
        "đừng tưởng cứ đi ngang {T} là giao, đích đến là {G}",
        "ghé {T} chỉ để đổ xăng thôi, {GV}",
    ],
    "out_of_bounds": [
        "{T} không cho shipper vào đâu, {GV}",
        "xe không vào được {T}, nên {GV}",
        "bảo vệ ở {T} đuổi rồi, vòng qua {G} giùm",
        "{T} nay cấm xe máy, chốt lại là {GV}",
    ],
    "canceled_order": [
        "đơn đi {T} khách hủy rồi, chạy thẳng {GV}",
        "hủy chuyến {T} nha, {GV} thôi",
        "bom hàng ở {T} rồi, mang qua {G} đi",
        "đơn {T} khách không nghe máy, tập trung đơn {G}",
    ],
    "ignore_note": [
        "kệ cái dòng ghi chú {T} đi, {GS}",
        "đừng quan tâm tin nhắn nhắc {T}, {GS}",
        "bỏ qua tờ giấy note dán trên hộp ghi {T}, {GS}",
        "hệ thống lỗi hiển thị {T}, cứ lơ đi, {GS}",
    ],
    "misdirected": [
        "bảo vệ hay chỉ nhầm ra {T}, thực ra {GS}",
        "người ta bảo {T} là sai đấy, {GV}",
        "đứa nào xúi đi {T} là lừa đó, {GS}",
        "đừng nghe mấy người đi đường chỉ ra {T}, {GS}",
    ]
}

AFF_W = {"plain": 3.0, "self_correct": 2.0, "old_info": 1.5, "unavailable": 1.5,
         "preference_override": 1.5, "other_order": 1.5,
         "future_cond": 1.5, "transfer_person": 1.5, "wait_vs_deliver": 1.5,
         "past_completed": 1.5, "swap_package": 1.5, "via_not_goal": 1.5,
         "out_of_bounds": 1.5, "canceled_order": 1.5}
NEG_W = {"reject_goal": 3.0, "qa_reject": 1.5, "note_wrong": 1.5,
         "reject_multi": 1.5, "ignore_note": 1.5, "misdirected": 1.5}

EXTRA_TPL = {
    "route_pass": [
        "đường đi sẽ ngang {T}, cứ chạy thẳng", "chạy ngang {T} là thấy",
        "{T} chỉ là mốc để định hướng", "lấy {T} làm mốc rồi đi tiếp",
        "đi ngang qua {T} sẽ thấy", "{T} nằm trên đường đi thôi",
    ],
    "route_no_stop": [
        "đi ngang {T} thôi, đừng dừng", "chạy qua {T} nhưng không cần ghé",
        "{T} chỉ để định hướng, không dừng ở đó",
        "không cần ghé {T}, chỉ đi ngang", "{T} chỉ là điểm đi ngang, không dừng",
        "lướt qua {T} thôi nhé, không vào",
    ],
    "no_stop": [
        "không cần ghé đâu hết, đi thẳng", "đi thẳng, đừng ghé {T}",
        "khỏi qua {T} lấy gì, hàng có sẵn trên xe rồi",
        "hàng có sẵn rồi, không cần qua {T} lấy",
        "không cần lấy hàng ở {T}, đi thẳng",
        "không có điểm dừng trung gian nào, kể cả {T}",
        "đi một mạch, không ghé {T}",
        "không phải qua {T} đâu, hàng {you} đã cầm rồi",
    ],
    "reject_via": [
        "lấy hàng ở {V}, đừng lấy ở {TV}", "{PS}, không phải {TV}",
        "nhận hàng tại {V} chứ không phải {TV}", "{TV} khỏi ghé, {PS}",
        "không lấy hàng ở {TV} nữa, {PS}",
        "điểm lấy hàng là {V}; {TV} là chỗ cũ, bỏ qua",
        "đừng ghé {TV}; {PV}", "{TV} thì thôi, qua {V} lấy hàng",
        "hàng không nằm ở {TV} mà ở {V}",
    ],
}

KIND_TYPES = {"pos_correct", "pos_neg_kind"}
ANCHOR_TYPES = {"pos_neg_anchor", "anchor_reject"}

REF_FIRST = ["cái đầu", "cái thứ nhất", "chỗ đầu tiên", "chỗ nhắc trước", "cái nói trước"]
REF_SECOND = ["cái thứ hai", "cái sau", "chỗ thứ hai", "chỗ nhắc sau", "cái nói sau",
              "chỗ vừa nhắc sau cùng"]
COREF_LEAD = ["{a} và {b} đều ở gần đây", "{a} với {b} đều tiện đường",
              "có hai chỗ: {a} và {b}", "{a}, {b}: cả hai đều giao được",
              "nhắc hai chỗ {a} và {b}", "hai chỗ {a} và {b} đều nhận hàng"]
PERSONS = ["cô lao công", "bác bảo vệ", "chị lễ tân", "anh trực ban", "bạn cùng phòng",
           "thầy giám thị", "cô quản lý", "chú bảo vệ", "em trực quầy"]
TIME_MOD = ["giờ nghỉ trưa", "sau giờ học", "vào buổi tối", "lúc tan ca"]

POS_NEG_KIND_ANCHOR = ["{GS}, chứ không {KO} {AT}", "{GS}, đâu có {KO} {AT}",
                       "{GS}; chỗ đó không {KO} {AT} đâu", "{GS} — nhớ là không {KO} {AT}"]
POS_NEG_KIND_PLAIN = ["{GS}, chứ không phải {GB} {KO}", "{GS} (không phải {KO})",
                      "{GS}; {KO} thì là chỗ khác", "đừng nhầm sang {GB} {KO}: {GS}"]

GOAL_TPL.update({
    "temporal": [
        "hôm qua đã {GV_T0} rồi, hôm nay {REDIR}",
        "hôm qua {GV_T0} xong rồi; hôm nay {GV}",
        "đơn hôm qua đã tới {T} rồi nhé, đơn hôm nay thì {GV}",
        "hôm nay thì {GV}, ngày mai mới tính tới {T}",
        "tuần trước {GV_T0}, tuần này {REDIR}",
        "lần sau {GV_T0}, còn lần này {REDIR}",
        "sau này có thể {GV_T0}, còn bây giờ {GV}",
        "đã {GV_T0} từ chiều qua, giờ {REDIR}",
        "chiều nay {GV_T0} xong rồi, đơn mới này {GV}",
        "{T} là chuyện của hôm kia, hôm nay {REDIR}",
        "{GS}. Chuyện giao ở {T} là của hôm qua",
        "hôm kia {GV_T0}, hôm qua cũng vậy, nhưng hôm nay {GV}",
    ],
    "hypo": [
        "nếu {G} đóng cửa thì mới qua {T}, nhưng {GB} đang mở nên cứ {GV}",
        "{G} mở thì giao ở đó; còn nếu {GB} đóng mới tính tới {T}. Hiện {GB} đang mở",
        "phương án dự phòng là {T}, nhưng chưa cần dùng vì {GB} vẫn nhận hàng bình thường, {GV}",
        "chỉ khi {GB} đóng cửa mới chuyển sang {T}, còn bây giờ {GV}",
        "nếu {T} còn chỗ thì giao ở {T}, nhưng {T} hết chỗ rồi nên {REDIR}",
        "giả sử {T} nhận được thì đã giao ở đó, tiếc là {T} đang đóng nên {REDIR}",
        "kế hoạch A là {T}, nhưng kế hoạch A bị hủy; chốt kế hoạch B: {G}",
        "nếu {T} có người nhận thì {me} đã chọn {T}, nhưng vắng người nên {REDIR}",
    ],
    "pickup_goal": [
        "khách sẽ tự qua lấy ở {G}", "để khách ghé lấy tại {G}",
        "gửi lại ở {G} cho khách tự lấy",
        "người nhận tự đến lấy ở {G}, {you} chỉ cần đưa tới đó",
        "{o} để ở {G}, người nhận sẽ qua lấy",
        "bên nhận tự lấy tại {G}", "{name} tự qua {G} lấy",
    ],
    "pos_correct": [
        "{GV_W}, {corr}, {REDIR}", "{GV_W}... {corr}, {REDIR}",
        "{GS}. {GW} là {me} nói nhầm", "{GW}? {Corr}, {G} mới đúng",
        "{GS}; lúc nãy {me} ghi {GW} là gõ nhầm",
        "vừa rồi {me} bảo {GW}, {corr}: {REDIR}", "{GW} — {corr}, {REDIR}",
    ],
    "pos_neg_anchor": [
        "{GS}, chứ không phải chỗ {KT} {AW}", "{GS} (không {KT} {AW} nhé)",
        "{GS}; {KT} {AW} là chỗ khác", "đừng nhầm với chỗ {KT} {AW}: {GS}",
        "{GS}, {me} không nói chỗ {KT} {AW}",
    ],
    "anchor_reject": [
        "{GS}, chứ không phải {AT}", "{GS}; {AT} chỉ là mốc, không phải chỗ nhận",
        "đừng nhầm {AT} với chỗ nhận: {GS}", "{AT} chỉ để định vị thôi, {GV}",
        "{AT} là mốc thôi nhé, {GS}", "đích không phải {AT} đâu, {GS}",
    ],
    "anchor_trap": [
        "{GS}, không phải {TG}", "{GS}; còn {TG} là chỗ khác",
        "đừng nhầm với {TG}: {GS}", "{TG} thì thôi, {GS}",
    ],
    "double_neg": [
        "không phải là không được ghé {T}, nhưng đích thật sự là {G}",
        "{T} thì không phải không đi được, mà đích đến là {G}",
        "chẳng phải {me} không muốn giao ở {T}, chỉ là đích cuối vẫn là {G}",
        "đâu phải {T} không nhận hàng, nhưng đơn này giao {G}",
        "không thể nói là không cần tới {G}, còn {T} thì khỏi",
        "đâu phải không cần giao tới {G}, đừng nhầm sang {T}",
        "không phải {T} không tiện, song điểm nhận là {G}",
        "chẳng phải {T} không đúng đường, mà là phải {GV}",
    ],
    "sarcasm": [
        "giỏi thì cứ mang ra {T} mà vứt, {me} đang đợi ở {G} cơ mà",
        "ừ cứ giao ở {T} đi, rồi {me} đứng ở {G} chờ hoài",
        "{T} á? thôi đi, {GS}", "chắc là {T} nhỉ? đùa thôi, {GS}",
        "ừ thì cứ ra {T} đi, còn {me} chờ ở {G} nè",
        "tuyệt, lại đưa nhầm ra {T}, {REDIR} mới đúng chứ",
    ],
})

AFF_W.update({"temporal": 1.8, "hypo": 1.2, "coref": 1.3, "pickup_goal": 1.0,
              "recipient": 1.0, "pos_correct": 1.4})
NEG_W.update({"double_neg": 1.0, "sarcasm": 0.8, "coref_neg": 1.0, "pos_neg_anchor": 1.2,
              "pos_neg_kind": 1.2, "anchor_reject": 1.0, "anchor_trap": 1.0})

EXTRA_TPL["landmarks"] = [
    "dọc đường có {T3}, {T4}, chỉ để tham khảo", "đường đi thường qua {T3} rồi {T4}, đó là mốc thôi",
    "nhìn thấy {T3} rồi {T4} là gần tới", "mấy chỗ như {T3}, {T4} chỉ là cột mốc dọc đường",
]
EXTRA_TPL["no_stop_aff"] = [
    "hôm qua phải ghé {T} lấy hàng, hôm nay hàng có sẵn trên xe rồi",
    "đơn của {name} lấy ở {T}, còn đơn này hàng đã ở trên xe",
    "{T} là chỗ lấy hàng của đơn kia, đơn này đi thẳng",
]
EXTRA_TPL["reject_via"] += [
    "không phải là không ghé {TV}, nhưng chỗ lấy hàng là {V}",
    "đâu phải {TV} không có hàng, nhưng đơn này {PV}",
]
VIA_TRICK_AFF = [
    "hôm qua {PV_T}; hôm nay {PV}", "đơn của {name} lấy ở {TV}, còn đơn này {PV}",
    "{TV} {why}, nên {PV}", "{PV_T}... {corr}, {PV}",
    "lần trước lấy ở {TV}, lần này {PV}", "ghi chú cũ ghi lấy ở {TV}, thông tin mới là {V}",
    "hôm qua đã lấy ở {TV} rồi, hôm nay qua {V} lấy",
    "{V} mới là chỗ có hàng hôm nay, {TV} là của hôm qua",
    "nếu {TV} hết hàng thì mới qua {V}, mà {TV} đã hết rồi nên {PV}",
]

ATTR = {
    ("urgent", "yes", "std"): [
        "đơn này gấp", "gấp lắm", "gấp nhé", "đơn gấp", "cần giao gấp",
        "{me} đang cần gấp", "làm nhanh giúp {me}", "ưu tiên đơn này trước",
        "đơn này ưu tiên số một", "khách đang đứng chờ rồi",
        "khách chờ lâu rồi", "cần có trong {n} phút nữa",
        "trong vòng {n} phút phải tới", "càng sớm càng tốt",
        "giao ASAP nha", "đơn hỏa tốc", "đơn khẩn", "sát giờ lắm rồi",
        "đi liền giúp {me}", "chạy nhanh giúp {me}",
        "trễ là khách hủy đơn đó",
    ],
    ("urgent", "no", "std"): [
        "đơn này không gấp", "không gấp đâu", "chưa cần gấp", "không cần vội",
        "từ từ cũng được", "cứ thong thả", "khi nào tiện thì giao",
        "trong ngày mai cũng được", "giao chiều mai cũng ổn",
        "cuối tuần giao cũng không sao", "không có yêu cầu ưu tiên",
        "đi theo lịch bình thường là được", "đơn thường, không phải đơn gấp",
        "khỏi chạy nhanh", "không cần giao ngay", "{me} không vội",
        "thời gian thoải mái", "không áp lực thời gian",
        "ngày mốt tới cũng được", "tuần này giao xong là được",
    ],
    ("urgent", "yes", "hard"): [
        "không thể để trễ được", "không được chậm nhé",
        "tuyệt đối không được trễ", "đừng để khách chờ thêm",
        "không còn nhiều thời gian đâu", "chớ để trễ nhé",
        "nhỏ thôi nhưng cần gấp", "đồ nhỏ mà gấp lắm",
        "tuy đơn nhỏ nhưng khách chờ rồi",
    ],
    ("urgent", "no", "hard"): [
        "đơn của {name} mới gấp, đơn này thì không",
        "hôm qua thì gấp, hôm nay đã ổn rồi", "đừng tưởng đây là đơn gấp",
        "{name} bảo gấp nhưng đơn này thì không cần",
        "gấp là chuyện của đơn kia, đơn này cứ từ từ",
        "lần trước gấp, lần này thì thong thả",
        "không phải đơn gấp, cứ theo lịch",
        "đơn gấp xong rồi, còn đơn này thì không vội",
        "đừng chạy nhanh quá, đơn này chưa gấp",
    ],
    ("fragile", "yes", "std"): [
        "hàng {w}, nhớ nhẹ tay",
        "món này thuộc loại {w}, vận chuyển cẩn thận giúp {me}",
        "nhẹ tay nhé, đây là đồ {w}", "lưu ý: hàng {w}",
        "đừng xóc mạnh vì món này {w}", "kiện này {w}, bảo quản cẩn thận nha",
        "xử lý nhẹ tay một chút, hàng khá {w}",
        "đồ {w} đó, xếp riêng giúp {me}", "hàng dạng {w} nên xếp trên cùng",
        "bên trong là đồ {w}", "cẩn thận vì hàng khá {w}",
        "dán nhãn hàng {w} giúp {me}", "{o} thuộc loại {w}",
    ],
    ("fragile", "no", "std"): [
        "món này không {w}", "đây không phải hàng {w}",
        "không có lưu ý gì về đồ {w}", "kiện này không thuộc loại {w}",
        "hàng thường, không {w}", "không cần xem món này là đồ {w}",
        "hàng chắc chắn, không {w}",
        "đóng gói thường thôi, không phải đồ {w}", "không có gì {w} hết",
        "hàng không {w}, cứ xếp bình thường",
    ],
    ("fragile", "yes", "hard"): [
        "đừng làm rơi", "không được va đập",
        "tuyệt đối không xóc mạnh vì đồ {w}", "đừng quăng, đồ {w} đó",
        "đừng để bị đè lên vì {w}", "không được chồng đồ lên trên, hàng {w}",
    ],
    ("fragile", "no", "hard"): [
        "đơn của {name} mới {w}, đơn này thì không",
        "hôm trước gửi đồ {w}, lần này hàng thường",
        "đừng nhầm với đơn {w} kia",
        "không phải đồ {w} đâu, cứ xếp bình thường",
        "{w} là đơn kia, đơn này thì khác",
        "đơn {w} đã đi trước rồi, đơn này là hàng thường",
    ],
    ("fragile", "yes", "para"): [
        "đừng chồng đồ nặng lên trên", "tránh va đập mạnh", "đừng ném hay quăng",
        "cầm nhẹ tay thôi", "đặt cẩn thận, tránh rung lắc", "nhẹ tay, kẻo bể",
        "xếp riêng ra, đừng để đè",
    ],
    ("fragile", "no", "para"): [
        "đồ cứng cáp, không sợ va chạm", "hàng bình thường, xử lý như mọi khi",
        "hàng chắc chắn, xếp thoải mái",
        "đóng gói thường thôi, chồng đồ cũng được",
        "không cần giữ gìn đặc biệt", "cứ xếp như hàng bình thường",
    ],
}

ATTR[("urgent", "yes", "hard")] += [
    "hôm qua thì thong thả, hôm nay thì gấp", "sáng khách bảo không gấp, giờ đổi ý, cần gấp",
    "trước đó không vội nhưng giờ cần gấp rồi", "khách vừa gọi giục lần nữa",
    "đã nhắc hai lần rồi, khách đang sốt ruột", "đừng nghe câu 'không gấp' ở tin trước, đơn này gấp",
    "không gấp... à nhầm, gấp lắm", "tài liệu gấp làm ba cho vừa túi, nhưng đơn thì gấp thật",
]
ATTR[("urgent", "no", "hard")] += [
    "mai mới cần, hôm nay chưa", "khách dặn tuần sau mới cần",
    "nếu có gấp thì {me} đã báo trước rồi, đơn này thì chưa",
    "hồi sáng thì gấp, giờ khách báo thư thả rồi", "đã gấp từ hôm qua, giờ khách nói không cần nữa",
    "đơn gấp hôm qua đã xong rồi", "gấp... à nhầm, không gấp",
    "tài liệu đã gấp làm ba cho vừa túi, đơn thường thôi",
    "áo đã gấp sẵn trong túi, giao lúc nào cũng được",
    "đoạn qua cổng đi nhanh thôi vì có chó dữ, chứ đơn không gấp",
    "phụ phí cuối tuần gấp đôi, đơn thì không gấp",
]
ATTR[("fragile", "yes", "hard")] += [
    "hôm qua hàng thường, hôm nay là đồ {w}", "tin trước ghi nhầm hàng thường, thật ra là đồ {w}",
    "hàng thường... à nhầm, đồ {w}", "nếu là đồ thường thì cứ xếp, nhưng đây là hàng {w}",
    "{w} nên {me} mới dặn kỹ, đừng cho là hàng thường",
]
ATTR[("fragile", "no", "hard")] += [
    "tin trước ghi {w} là nhầm, hàng thường", "đồ {w}... nhầm, hàng thường",
    "nếu là đồ {w} thì {me} đã báo, đơn này thì không", "nếu {w} thì nhẹ tay, nhưng đây là hàng thường",
    "cẩn thận chó ở cổng nhé, còn hàng thì không {w}", "đi cẩn thận kẻo trơn, hàng thường thôi",
    "mục đồ {w} trên app {me} không chọn", "kiện {w} ở kho kia, kiện này thì không",
]
ATTR[("urgent", "yes", "dneg")] = ["không phải là không gấp", "chẳng phải không gấp đâu",
                                   "đâu phải đơn không gấp", "không thể nói là không gấp"]
ATTR[("fragile", "yes", "dneg")] = ["đâu phải hàng không {w}", "không phải đồ không {w}",
                                    "không thể bảo là không {w}", "chẳng phải món này không {w}"]
ATTR[("urgent", "yes", "sarc")] = ["ừ cứ từ từ đi, khách chờ có vài phút thôi mà",
                                   "thong thả nhé, khách chỉ đứng chờ ngoài mưa thôi"]

ATTR_COMB = {
    ("yes", "yes"): [
        "vừa gấp vừa {w}, ráng giúp {me}",
        "gấp mà lại {w}, đi nhanh nhưng nhẹ tay", "đơn gấp và là hàng {w}",
        "nhanh thì nhanh nhưng nhớ nhẹ tay, đồ {w}",
        "hàng {w} nên nhẹ tay, mà {me} cũng cần gấp",
    ],
    ("yes", "no"): [
        "gấp nhưng không {w}", "đơn gấp, còn hàng thì không {w}",
        "cần nhanh, hàng không {w} nên khỏi lo", "gấp chứ không phải hàng {w}",
        "ưu tiên đơn này, hàng thường thôi",
        "chạy nhanh giúp {me}, đồ không {w}",
    ],
    ("no", "yes"): [
        "không gấp, chỉ cần nhẹ tay vì hàng {w}", "hàng {w} nhưng không vội",
        "từ từ cũng được, miễn là hàng {w} được giữ cẩn thận",
        "đồ {w}, đi chậm mà chắc", "chưa cần gấp, nhớ là hàng {w}",
        "đừng vội, hàng {w} mà",
    ],
    ("no", "no"): [
        "không gấp, cũng không {w}", "đơn thường, không vội, không phải hàng {w}",
        "hàng bình thường và không cần ưu tiên", "không gấp mà cũng chẳng {w}",
        "thong thả thôi, hàng thường",
    ],
}

FILL_PRE = ["chào {you}", "hi {you}", "alo", "{you} ơi", "cho {me} hỏi chút",
            "ok {you}", "dạ", "ừ", "à", "này", "đơn mới nè",
            "chào {you}, {me} gửi thông tin đơn nhé"]
FILL_POST = ["cảm ơn {you} nhiều", "thanks nha", "cảm ơn nhé", "ok chốt vậy nhé",
             "có gì nhắn {me} nha", "xong báo {me} một tiếng", "{me} đợi tin nhé",
             "cảm ơn {you} ạ", "cám ơn nhiều", "chốt vậy nha"]
FILL_INFO = ["mã đơn #{k}", "đơn đặt lúc {h}h", "khách tên {name_short}",
             "sđt cuối {d4}", "đơn đã thanh toán rồi", "đơn thu hộ COD",
             "khách đã xác nhận rồi", "trời đang mưa nhỉ",
             "hôm nay đông khách quá", "đơn số {k}"]


# ---------------------------------------------------------------
# 2. BALANCER (giữ nguyên từ V2)
# ---------------------------------------------------------------
class Balancer:
    def __init__(self, rng):
        self.rng = rng
        self.c = collections.Counter()

    def pick(self, key, options):
        opts = list(dict.fromkeys(options))
        if not opts:
            raise ValueError(f"Không có option để pick: {key!r}")
        m = min(self.c[(key, o)] for o in opts)
        choices = [o for o in opts if self.c[(key, o)] == m]
        o = self.rng.choice(choices)
        self.c[(key, o)] += 1
        return o


# ---------------------------------------------------------------
# 3. TEXT / NOISE HELPERS
# ---------------------------------------------------------------
def cap(s):
    return s[:1].upper() + s[1:] if s else s

def first_upper(text):
    return cap(text)

_TONES = {"\u0301": "s", "\u0300": "f", "\u0309": "r", "\u0303": "x", "\u0323": "j"}

def to_telex(word):
    out, tone = [], ""
    for ch in unicodedata.normalize("NFD", word):
        if ch in _TONES:
            tone = _TONES[ch]
        elif ch == "\u0302" and out:
            out.append(out[-1])
        elif ch in ("\u031b", "\u0306"):
            out.append("w")
        elif ch == "đ":
            out.append("dd")
        elif ch == "Đ":
            out.append("DD")
        else:
            out.append(ch)
    return "".join(out) + tone

def _build_kb():
    rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]
    nb = {}
    for ri, row in enumerate(rows):
        for ci, ch in enumerate(row):
            s = []
            if ci > 0:
                s.append(row[ci - 1])
            if ci < len(row) - 1:
                s.append(row[ci + 1])
            for rj in (ri - 1, ri + 1):
                if 0 <= rj < len(rows) and ci < len(rows[rj]):
                    s.append(rows[rj][ci])
            nb[ch] = s
    return nb

_KB = _build_kb()

def _mutate_word(core, rng):
    op = rng.choices(["swap", "drop", "dup", "kb", "telex", "elong"],
                     weights=[0.28, 0.22, 0.12, 0.16, 0.14, 0.08], k=1)[0]
    n = len(core)
    if op == "telex":
        t = to_telex(core)
        return t if t != core else core + core[-1]
    if op == "elong":
        return core + core[-1] * rng.randint(1, 2)
    pos = rng.randint(1, n - 2)
    if op == "swap":
        return core[:pos] + core[pos + 1] + core[pos] + core[pos + 2:]
    if op == "drop":
        return core[:pos] + core[pos + 1:]
    if op == "dup":
        return core[:pos] + core[pos] + core[pos:]
    idxs = [i for i, ch in enumerate(core) if ch.lower() in _KB]
    if not idxs:
        return core
    i = rng.choice(idxs)
    return core[:i] + rng.choice(_KB[core[i].lower()]) + core[i + 1:]

# Hàm phá hủy thực thể (lỗi chí mạng khiến NER bối rối)
def extreme_typo(core, rng):
    if len(core) < 3: return core
    op = rng.choice(["insert_space", "drop_char", "dup_char", "kb_miss", "split_end"])
    pos = rng.randint(1, len(core)-1)
    if op == "insert_space":
        return core[:pos] + " " + core[pos:]
    if op == "drop_char":
        return core[:pos] + core[pos+1:]
    if op == "dup_char":
        return core[:pos] + core[pos] + core[pos:]
    if op == "kb_miss":
        c = core[pos].lower()
        if c in _KB:
            return core[:pos] + rng.choice(_KB[c]) + core[pos+1:]
    if op == "split_end":
        if pos < len(core) - 1:
            return core[:pos] + core[pos] + " " + core[pos+1:]
    return core

def add_typo(text, rng):
    if not text:
        return text
    words = text.split()
    cand = [i for i, w in enumerate(words)
            if "@" not in w and not re.search(r"\d", w)
            and len(re.sub(r"[^\w]", "", w)) >= 4]
    if not cand:
        return text
    n = rng.choices([1, 2, 3], weights=[0.62, 0.28, 0.10], k=1)[0]
    rng.shuffle(cand)
    for idx in cand[:n]:
        m = re.match(r"^(\W*)(.*?)(\W*)$", words[idx], flags=re.UNICODE)
        pre, core, post = m.group(1), m.group(2), m.group(3)
        if len(core) < 4:
            continue
        words[idx] = pre + _mutate_word(core, rng) + post
    if rng.random() < 0.10 and len(words) > 3:      # dính chữ (thiếu space)
        i = rng.randrange(len(words) - 1)
        if "@" not in words[i] and "@" not in words[i + 1]:
            words[i] = words[i] + words[i + 1]
            del words[i + 1]
    return " ".join(words)

_ABBR = {
    "không": ["ko", "k", "hok", "hông", "khong"], "được": ["đc", "dc"],
    "với": ["vs"], "bạn": ["bn"], "mình": ["mk", "mik"], "rồi": ["r", "rùi"],
    "luôn": ["lun"], "thôi": ["thui"], "vậy": ["zậy"], "nhưng": ["nhg"],
    "xong": ["xg"], "gửi": ["gởi"], "ok": ["oke", "okie"], "cảm": ["cám"],
    "người": ["ng"], "nhé": ["nhe"],
}
_ELONG = {"nhé", "nha", "với", "ạ", "nhen", "nghen", "nè", "nhe"}

def chat_noise(text, rng):
    toks = re.findall(r"\S+|\s+", text)
    out = []
    for t in toks:
        if t.isspace() or "@" in t:
            out.append(t)
            continue
        m = re.match(r"^([^\w]*)(.*?)([^\w]*)$", t, flags=re.UNICODE)
        pre, core, post = m.group(1), m.group(2), m.group(3)
        lc = core.lower()
        if lc in _ABBR and rng.random() < 0.55:
            core = rng.choice(_ABBR[lc])
        elif lc in _ELONG and rng.random() < 0.25:
            core = core + core[-1] * rng.randint(1, 3)
        out.append(pre + core + post)
    s = "".join(out)
    if rng.random() < 0.30:
        s = s.replace(",", "")
    return s

def norm_ws(txt):
    txt = re.sub(r"\s+", " ", txt)
    txt = re.sub(r"\s+([,.;!?:])", r"\1", txt)
    txt = re.sub(r"([,;:])\1+", r"\1", txt)
    txt = re.sub(r",\s*([.!?])", r"\1", txt)
    return txt.strip(" ,;")

def sentence(u, rng, p_excl=0.07):
    u = re.sub(r"\s+", " ", u).strip()
    if not u:
        return u
    u = cap(u)
    if u[-1] not in ".!?…)\"":
        u += "!" if rng.random() < p_excl else "."
    return u

_RECAP = re.compile(r"(?<![.])([.!?]\s+)([a-zà-ỹ])")

def recap(s):
    return _RECAP.sub(lambda m: m.group(1) + m.group(2).upper(), s)

def render_units(units, rng, style):
    units = [u for u in units if u and u.strip()]
    if style == "formal":
        return recap(" ".join(sentence(u, rng) for u in units))
    if style == "terse":
        sep = rng.choice([" | ", " / ", "; ", " - "])
        return sep.join(re.sub(r"[\s.;,]+$", "", u.strip()) for u in units)
    sep = rng.choice([", ", ", ", ". ", " - ", " . ", " "])
    parts = [re.sub(r"[\s.;,]+$", "", u.strip()).lower() for u in units]
    return sep.join(parts)


# ---------------------------------------------------------------
# 4. ĐỊA ĐIỂM: ALIAS / CLASH / PHRASE
# ---------------------------------------------------------------
_ALIAS_CACHE = {}
_CLASH = {}

def _alias_set(p):
    s = _ALIAS_CACHE.get(p)
    if s is None:
        s = {fold(a).lower().strip() for a in BANK.get(p, [p])}
        _ALIAS_CACHE[p] = s
    return s

def clash(p, q):
    if p == q:
        return True
    k = (p, q) if p < q else (q, p)
    v = _CLASH.get(k)
    if v is None:
        v = bool(_alias_set(p) & _alias_set(q))
        _CLASH[k] = v
    return v

def _distract(B, key, forb):
    opts = [p for p in PLACES if not any(clash(p, f) for f in forb)]
    if not opts:
        opts = [p for p in PLACES if p not in forb] or list(PLACES)
    return B.pick(key, opts)

def _relation(rng, B, raw, role):
    if rng.random() >= CFG["p_kind"] or not KINDS:
        return "none", "none"
    kind = B.pick((role, "kind"), KINDS)
    if kind not in ("near", "far"):
        return kind, "none"
    opts = [p for p in PLACES if not clash(p, raw)] or [p for p in PLACES if p != raw]
    return kind, B.pick((role, "anchor"), opts)

def alias_for(raw_place, role, B, has_typo, rng):
    aliases = BANK.get(raw_place, [raw_place])
    value = B.pick(("place_alias", role, raw_place), aliases)
    # Bổ sung logic Extreme Typo cho Alias khi có cờ has_typo
    if has_typo and rng.random() < CFG["p_alias_typo"]:
        words = value.split()
        if words:
            idx = rng.randrange(len(words))
            words[idx] = extreme_typo(words[idx], rng)
            value = " ".join(words)
    return value

KIND_ONLY_FORMS = ["{base} {k}", "{k} {base}", "{base}, {k}", "{base} ({k})",
                   "chỗ {base} {k}", "{base} — {k}"]
ANCHOR_FORMS = ["{base} {k} {a}", "{base}, {k} {a}", "{base} ({k} {a})",
                "{base}, chỗ {k} {a}", "{k} {a}, chỗ {base}",
                "{base} nằm {k} {a}", "{base} — {k} {a}",
                "chỗ {base} ấy, {k} {a}", "{base} {k} {a} đó"]

def place_parts(raw_place, kind="none", anchor="none", role="place", B=None,
                has_typo=False, rng=None, base=None, kw=None, aa=None):
    rng = rng or random.Random()
    B = B or Balancer(rng)
    if base is None:
        base = alias_for(raw_place, role, B, has_typo, rng)
    if kind == "none":
        return {"text": base, "base": base, "k": None, "a": None}
    if kw is None:
        kw = B.pick(("kind_alias", kind), BANK.get(kind, [kind]))
    if anchor == "none":
        t = B.pick(("kind_form", role, kind), KIND_ONLY_FORMS).format(base=base, k=kw)
        return {"text": t, "base": base, "k": kw, "a": None}
    if aa is None:
        aa = alias_for(anchor, f"anchor_{role}", B, has_typo, rng)
    t = B.pick(("kind_anchor_form", role, kind), ANCHOR_FORMS).format(base=base, k=kw, a=aa)
    return {"text": t, "base": base, "k": kw, "a": aa}

def place_phrase(raw_place, kind="none", anchor="none", role="place", B=None,
                 has_typo=False, rng=None):
    return place_parts(raw_place, kind, anchor, role, B, has_typo, rng)["text"]

def _forced_relation(B, raw, role, kinds):
    kind = B.pick((role, "fkind"), kinds)
    if kind in ("near", "far"):
        opts = [p for p in PLACES if not clash(p, raw)] or [p for p in PLACES if p != raw]
        return kind, B.pick((role, "fanchor"), opts)
    return kind, "none"


# ---------------------------------------------------------------
# 5. CTX: bộ sinh mệnh đề (ghép từ thành phần)
# ---------------------------------------------------------------
class _Fmt(dict):
    def __init__(self, base, lazy):
        super().__init__(base)
        self._lazy = lazy

    def __missing__(self, k):
        if k in self._lazy:
            v = self._lazy[k]()
            self[k] = v
            return v
        raise KeyError(k)

def render_vp(P, place, with_obj=True):
    o = (" " + P["o"]) if (with_obj and P["o"]) else ""
    return f"{P['v']}{P['adv']}{o} {P['p']} {place}"

class Ctx:
    def __init__(self, rng, B, bk, persona, item):
        self.rng, self.B, self.bk = rng, B, bk
        self.me, self.you, self.south, self.polite = persona
        self.item = item

    def tail(self, mid=False, p_none=0.5):
        r = self.rng
        if r.random() < p_none:
            return ""
        pool = list(TAIL_COMMON) + (TAIL_SOUTH if self.south else TAIL_NORTH)
        if self.polite:
            pool += TAIL_POLITE
        if not mid:
            pool += TAIL_END_ONLY
        return r.choice(pool).format(me=self.me)

    def obj(self, allow_empty=False):
        r = self.rng
        if allow_empty and r.random() < 0.22:
            return ""
        x = r.random()
        if x < 0.38:
            return self.item
        if x < 0.50:
            return "đơn " + self.item
        if x < 0.58:
            return "món " + self.item
        return r.choice(OBJ_PRON)

    def vp_parts(self):
        B, r, bk = self.B, self.rng, self.bk
        verb = B.pick((bk, "verb"), list(GIVE_VERBS))
        prep = B.pick((bk, "prep", verb), GIVE_VERBS[verb])
        adv = ""
        if verb != "để lại" and r.random() < 0.16:
            adv = " " + r.choice(["thẳng", "luôn", "trực tiếp"])
        return {"v": verb, "p": prep, "adv": adv, "o": self.obj(allow_empty=True)}

    def vp(self, place):
        return render_vp(self.vp_parts(), place)

    def vp_contrast(self, G, T):
        P = self.vp_parts()
        fG, sG = render_vp(P, G), render_vp(P, G, False)
        fT, sT = render_vp(P, T), render_vp(P, T, False)
        p = P["p"]
        forms = [
            f"{fG}, chứ không {sT}", f"không {sT} mà {fG}",
            f"{fG} thôi, đừng {sT}", f"đừng {fT}; {sG} mới đúng",
            f"{sG}, không phải {p} {T}", f"đừng {fT}, hãy {sG}",
            f"{fG} nhé, đừng nhầm sang {T}",
            f"{P['v']}{P['adv']} {P['o']} không phải {p} {T} mà là {p} {G}",
        ]
        return self.B.pick((self.bk, "contrast"), forms)

    def gs(self, G, composable=True, family=None):
        r, me, you = self.rng, self.me, self.you
        fams = GS_FAMS if composable else GS_FAMS + ["ask"]
        fam = family or self.B.pick((self.bk, "gs_fam", composable), fams)
        t = self.tail()
        if fam == "imp":
            s = f"{self.vp(G)} {t}"
        elif fam == "nho":
            s = f"{r.choice(['nhờ', 'phiền', 'xin nhờ', 'làm phiền'])} {you} {self.vp(G)} {t}"
        elif fam == "you":
            s = f"{you} {self.vp(G)} {t}"
        elif fam == "want":
            s = r.choice([f"{me} muốn nhờ {you} {self.vp(G)}",
                          f"{me} cần {you} {self.vp(G)}",
                          f"{me} muốn {self.vp(G)}"]) + " " + t
        elif fam == "pass":
            P = self.vp_parts()
            aux = r.choice(["cần được", "phải được", "muốn được", "sẽ được"])
            s = f"{self.obj(False)} {aux} {P['v']}{P['adv']} {P['p']} {G} {t}"
        elif fam == "topic":
            P = self.vp_parts()
            s = f"{self.obj(False)}{r.choice([' thì', ''])} {render_vp(P, G, False)} {t}"
        elif fam == "formal":
            lead = r.choice(["vui lòng", "đề nghị", "yêu cầu", "kính nhờ " + you, "xin vui lòng"])
            s = f"{lead} {self.vp(G)}"
        elif fam == "nominal":
            s = r.choice(NOMINAL).format(G=G)
        elif fam == "recv":
            s = r.choice(RECV).format(G=G)
        elif fam == "frag":
            s = r.choice(FRAG).format(G=G, t=t)
        elif fam == "go":
            s = r.choice(GO).format(G=G, me=me, t=t)
        else:
            s = r.choice([f"có thể {self.vp(G)} không {you}?",
                          f"{you} {self.vp(G)} giúp {me} được không?",
                          f"{self.vp(G)} được không?"])
        return re.sub(r"\s+", " ", s).strip()

    def pv(self, V):
        r, B = self.rng, self.B
        if r.random() < 0.12:
            return B.pick((self.bk, "stop"), STOP_PATTERNS).format(V=V)
        pat = B.pick((self.bk, "pick"), PICK_PATTERNS)
        o = r.choice(PICK_OBJ).format(item=self.item)
        if "nhận" in pat and not o:
            o = "hàng"
        return re.sub(r"\s+", " ", pat.format(o=o, V=V)).strip()

    def ps(self, V):
        tpl = self.B.pick((self.bk, "ps"), PS_FRAMES)
        return re.sub(r"\s+", " ", self.fmt(tpl, V=V)).strip()

    def redir(self, G):
        return self.fmt(self.B.pick((self.bk, "redir"), REDIR_FORMS), G=G)

    def qa(self):
        q = self.fmt(self.B.pick((self.bk, "qa_q"), QA_Q))
        a = self.fmt(self.B.pick((self.bk, "qa_a"), QA_A))
        r = self.rng
        x = r.random()
        if x < 0.25:
            return f"A: {q} B: {a}"
        if x < 0.35:
            return f"- {q} - {a}"
        return q + r.choice([" ", " ", " — ", " - "]) + a

    def obj_plain(self):
        x = self.rng.random()
        if x < 0.30:
            return ""
        if x < 0.65:
            return self.item
        return ("đơn " if x < 0.85 else "món ") + self.item

    def vp0(self, place):
        P = self.vp_parts()
        P["o"] = self.obj_plain()
        return render_vp(P, place)

    def coref(self, G, T, neg=False):
        r = self.rng
        g_first = r.random() < 0.5
        a, b = (G, T) if g_first else (T, G)
        ref_g = r.choice(REF_FIRST if g_first else REF_SECOND)
        ref_t = r.choice(REF_SECOND if g_first else REF_FIRST)
        lead = self.B.pick((self.bk, "coref_lead"), COREF_LEAD).format(a=a, b=b)
        body = self.vp_contrast(ref_g, ref_t) if neg else self.vp(ref_g)
        return f"{lead}, {body}"

    def gs_person(self, G):
        r = self.rng
        verb = r.choice(["giao", "đưa", "gửi", "chuyển", "trao", "để lại"])
        person = self.B.pick((self.bk, "person"), PERSONS)
        tm = (" " + r.choice(TIME_MOD)) if r.random() < 0.4 else ""
        core = f"{verb} {self.obj(allow_empty=True)} cho {person} {r.choice(['ở', 'tại'])} {G}{tm}"
        x, t = r.random(), self.tail()
        if x < 0.4:
            s = f"{core} {t}"
        elif x < 0.7:
            s = f"nhờ {self.you} {core} {t}"
        else:
            s = f"{self.me} muốn nhờ {self.you} {core}"
        return re.sub(r"\s+", " ", s).strip()

    def fmt(self, tpl, **kw):
        G = kw.get("G", PH["G"])
        V = kw.get("V", PH["V"])
        T = kw.get("T", PH["T1"])
        B, r, bk = self.B, self.rng, self.bk
        base = dict(G=G, V=V, T=T, TV=PH["TV"], T1=PH["T1"], T2=PH["T2"],
                    T3=PH["T3"], T4=PH["T4"], me=self.me, you=self.you,
                    Me=cap(self.me), You=cap(self.you),
                    GW=PH["GW"], GB=PH["GB"], AW=PH["AW"], KT=PH["KT"], KO=PH["KO"],
                    AT=PH["AT"], TG=PH["TG"])
        base.update({k: v for k, v in kw.items() if k not in ("G", "V", "T")})
        nm = r.choice(NAMES)
        lazy = {
            "o": lambda: self.obj(False),
            "tail": lambda: self.tail(),
            "tailm": lambda: self.tail(mid=True),
            "name": lambda: nm,
            "name_short": lambda: nm.split()[-1],
            "n": lambda: r.choice([10, 15, 20, 30, 45]),
            "k": lambda: r.randint(100, 9999),
            "h": lambda: r.randint(6, 21),
            "d4": lambda: f"{r.randint(0, 9999):04d}",
            "GV": lambda: self.vp(G),
            "GS": lambda: self.gs(G),
            "GV_T": lambda: self.vp(T),
            "GV_T0": lambda: self.vp0(T),
            "GV_W": lambda: self.vp(PH["GW"]),
            "PV_T": lambda: self.pv(PH["TV"]),
            "GS_T": lambda: self.gs(T),
            "GV_C": lambda: self.vp_contrast(G, T),
            "PV": lambda: self.pv(V),
            "PS": lambda: self.ps(V),
            "REDIR": lambda: self.redir(G),
            "corr": lambda: self.fmt(B.pick((bk, "corr"), CORR_SOFT)),
            "Corr": lambda: cap(self.fmt(B.pick((bk, "corr"), CORR_SOFT))),
            "not_T": lambda: self.fmt(B.pick((bk, "not_T"), NOT_T), T=T),
            "notT_lead": lambda: self.fmt(B.pick((bk, "notT_lead"), NOT_LEAD), T=T),
            "why": lambda: B.pick((bk, "why"), WHY),
            "conv": lambda: B.pick((bk, "conv"), CONV),
        }
        return tpl.format_map(_Fmt(base, lazy))


# ---------------------------------------------------------------
# 6. THUỘC TÍNH (urgent / fragile)
# ---------------------------------------------------------------
def one_attr(ctx, which, val, w):
    r = ctx.rng
    x = r.random()
    cuts = [("hard", CFG["p_attr_hard"])]
    if CFG["use_para_attr"] and which == "fragile":
        cuts.append(("para", CFG["p_attr_para"]))
    if CFG["use_double_neg"] and val == "yes":
        cuts.append(("dneg", CFG["p_attr_dneg"]))
    if CFG["use_sarcasm"] and which == "urgent" and val == "yes":
        cuts.append(("sarc", CFG["p_attr_sarc"]))
    kind, acc = "std", 0.0
    for k, pr in cuts:
        acc += pr
        if x < acc:
            kind = k
            break
    tpl = ctx.B.pick((ctx.bk, "attr", which, val, kind), ATTR[(which, val, kind)])
    return ctx.fmt(tpl, w=w)

def attr_units(ctx, urgent, fragile, w):
    r = ctx.rng
    omit_u = urgent == "no" and r.random() < CFG["p_omit_no"]
    omit_f = fragile == "no" and r.random() < CFG["p_omit_no"]
    if not omit_u and not omit_f and r.random() < CFG["p_attr_combined"]:
        tpl = ctx.B.pick((ctx.bk, "attr_comb", urgent, fragile), ATTR_COMB[(urgent, fragile)])
        return [ctx.fmt(tpl, w=w)], "combined"
    units = []
    if not omit_u:
        units.append(one_attr(ctx, "urgent", urgent, w))
    if not omit_f:
        units.append(one_attr(ctx, "fragile", fragile, w))
    return units, ("omit" if (omit_u or omit_f) else "separate")


# ---------------------------------------------------------------
# 7. TẠO 1 MẪU
# ---------------------------------------------------------------
def _eligible(t):
    kinds = [k for k in KINDS if k != "none"]
    if t in KIND_TYPES and len(kinds) < 2:
        return False
    if t in ANCHOR_TYPES and not ("near" in KINDS and "far" in KINDS):
        return False
    if t == "anchor_trap" and not ("near" in KINDS or "far" in KINDS):
        return False
    if t == "sarcasm" and not CFG["use_sarcasm"]:
        return False
    if t == "double_neg" and not CFG["use_double_neg"]:
        return False
    if t == "hypo" and not CFG["use_hypo"]:
        return False
    return True

def _weighted(table, allowed, exclude):
    return [(k, w) for k, w in table.items()
            if (allowed is None or k in allowed) and k not in exclude and _eligible(k)]

def _make_sample(rng, B, bk, is_neg, has_acc, has_typo, allowed, exclude):
    persona = B.pick((bk, "persona"), PERSONAS)
    item = B.pick((bk, "item"), ITEMS)
    ctx = Ctx(rng, B, bk, persona, item)

    pool = _weighted(NEG_W if is_neg else AFF_W, allowed, exclude)
    if pool and (not is_neg or rng.random() < CFG["p_neg_goal"]):
        gtype = rng.choices([k for k, _ in pool], weights=[w for _, w in pool], k=1)[0]
    else:
        gtype = "plain"
    is_neg_type = gtype in NEG_W

    goal = B.pick((bk, "goal"), PLACES)
    if gtype in KIND_TYPES:
        gk, ga = _forced_relation(B, goal, (bk, "grel"), [k for k in KINDS if k != "none"])
    elif gtype in ANCHOR_TYPES:
        gk, ga = _forced_relation(B, goal, (bk, "grel"), ["near", "far"])
    elif gtype in ("coref", "coref_neg"):
        gk, ga = "none", "none"
    else:
        gk, ga = _relation(rng, B, goal, (bk, "grel"))
    via = "none"
    if rng.random() < CFG["p_via"]:
        opts = [p for p in PLACES if not clash(p, goal)]
        if opts:
            via = B.pick((bk, "via"), opts)
    if via == "none":
        vk, va = "none", "none"
    else:
        vk, va = _relation(rng, B, via, (bk, "vrel"))

    forb = {goal} | ({via} if via != "none" else set())
    t1 = _distract(B, (bk, "t1"), forb)
    t2 = _distract(B, (bk, "t2"), forb | {t1})
    t3 = _distract(B, (bk, "t3"), forb | {t1, t2})
    t4 = _distract(B, (bk, "t4"), forb | {t1, t2, t3})
    tv = _distract(B, (bk, "tv"), forb | {t1, t2, t3, t4})

    ptr = (0.6 if gtype in ("other_order", "temporal", "old_info")
           else 0.0 if gtype in ("coref", "coref_neg") else CFG["p_trap_rel"])

    def trap(raw, role):
        if rng.random() < ptr:
            k, a = _relation(rng, B, raw, (bk, role + "_rel"))
            return place_phrase(raw, k, a, role, B, has_typo, rng)
        return alias_for(raw, role, B, has_typo, rng)

    gp = place_parts(goal, gk, ga, "goal", B, has_typo, rng)
    surf = {0: gp["text"], 8: gp["base"],
            2: alias_for(tv, "tv", B, has_typo, rng),
            3: trap(t1, "t1"), 4: trap(t2, "t2"),
            5: alias_for(t3, "t3", B, has_typo, rng),
            6: alias_for(t4, "t4", B, has_typo, rng)}
    if gp["k"] is not None:
        surf[10] = gp["k"]
    if gp["a"] is not None:
        surf[12] = gp["a"]
    if via != "none":
        surf[1] = place_phrase(via, vk, va, "via", B, has_typo, rng)

    if gtype in KIND_TYPES or gtype in ANCHOR_TYPES:
        wa_raw = _distract(B, (bk, "aw"), forb | ({ga} if ga != "none" else set()))
        surf[9] = alias_for(wa_raw, "aw", B, has_typo, rng)
        if gtype in KIND_TYPES:
            if gk in ("near", "far") and gtype == "pos_neg_kind":
                wk = "far" if gk == "near" else "near"
            else:
                alts = [k for k in KINDS if k not in (gk, "none")]
                if gtype == "pos_neg_kind":
                    alts = [k for k in alts if k not in ("near", "far")] or alts
                wk = B.pick((bk, "wk", gk), alts)
            wkw = B.pick(("kind_alias", wk), BANK.get(wk, [wk]))
            surf[11] = wkw
            if wk in ("near", "far"):
                wp = place_parts(goal, wk, wa_raw, "goalw", B, has_typo, rng,
                                 base=gp["base"], kw=wkw, aa=surf[9])
            else:
                wp = place_parts(goal, wk, "none", "goalw", B, has_typo, rng,
                                 base=gp["base"], kw=wkw)
            surf[7] = wp["text"]
    if gtype == "anchor_trap":
        kt = B.pick((bk, "tgk"), [k for k in ("near", "far") if k in KINDS])
        tgk = B.pick(("kind_alias", kt), BANK.get(kt, [kt]))
        surf[13] = place_parts(t1, kt, goal, "tg", B, has_typo, rng,
                               kw=tgk, aa=gp["base"])["text"]

    if gtype == "plain":
        goal_unit = ctx.gs(PH["G"], composable=False)
    elif gtype == "qa_reject":
        goal_unit = (ctx.fmt(B.pick((bk, "tpl", "qa_rev"), QA_REV))
                     if rng.random() < 0.50 else ctx.qa())
    elif gtype == "coref":
        goal_unit = ctx.coref(PH["G"], PH["T1"])
    elif gtype == "coref_neg":
        goal_unit = ctx.coref(PH["G"], PH["T1"], neg=True)
    elif gtype == "recipient":
        goal_unit = ctx.gs_person(PH["G"])
    elif gtype == "pos_neg_kind":
        if gk in ("near", "far"):
            lst, tag = POS_NEG_KIND_ANCHOR, "a"
        elif wk in ("near", "far"):
            lst, tag = [t.replace("{AT}", "{AW}") for t in POS_NEG_KIND_ANCHOR], "w"
        else:
            lst, tag = POS_NEG_KIND_PLAIN, "p"
        goal_unit = ctx.fmt(B.pick((bk, "tpl", gtype, tag), lst))
    else:
        goal_unit = ctx.fmt(B.pick((bk, "tpl", gtype), GOAL_TPL[gtype]))

    neg_units = 1 if is_neg_type else 0

    via_unit, via_mode = "", "none"
    if via != "none":
        if gtype == "plain" and rng.random() < CFG["p_via_composite"]:
            goal_unit = ctx.fmt(B.pick((bk, "tpl", "composite"), COMPOSITE))
            via_mode = "composite"
        elif is_neg and ((not is_neg_type) or rng.random() < 0.35):
            via_unit = ctx.fmt(B.pick((bk, "tpl", "reject_via"), EXTRA_TPL["reject_via"]))
            via_mode, neg_units = "reject", neg_units + 1
        elif (not is_neg) and rng.random() < CFG["p_via_trick"]:
            via_unit = ctx.fmt(B.pick((bk, "tpl", "via_trick"), VIA_TRICK_AFF))
            via_mode = "trick"
        else:
            via_unit, via_mode = ctx.ps(PH["V"]), "plain"

    extras, extra_kinds = [], []
    if is_neg:
        if neg_units == 0 or rng.random() < CFG["p_extra_neg"]:
            opts = ["route_no_stop"]
            if via == "none":
                opts.append("no_stop")
            elif via_mode != "reject":
                opts.append("reject_via")
            k = B.pick((bk, "extra_neg"), opts)
            extras.append(ctx.fmt(B.pick((bk, "tpl", k), EXTRA_TPL[k]), T=PH["T4"]))
            extra_kinds.append(k)
            neg_units += 1
    elif rng.random() < CFG["p_extra_aff"]:
        opts = ["route_pass", "landmarks"] + (["no_stop_aff"] if via == "none" else [])
        k = B.pick((bk, "extra_aff"), opts)
        extras.append(ctx.fmt(B.pick((bk, "tpl", k), EXTRA_TPL[k]), T=PH["T4"]))
        extra_kinds.append(k)

    urgent = B.pick((bk, "urgent"), ["yes", "no"])
    fragile = B.pick((bk, "fragile"), ["yes", "no"])
    fra_w = B.pick((bk, "fra_w"), BANK.get("fragile", ["dễ vỡ"]))
    attrs, attr_mode = attr_units(ctx, urgent, fragile, fra_w)

    pre = post = ""
    infos = []
    if rng.random() < CFG["p_filler_pre"]:
        pre = ctx.fmt(B.pick((bk, "fpre"), FILL_PRE))
    if rng.random() < CFG["p_filler_post"]:
        post = ctx.fmt(B.pick((bk, "fpost"), FILL_POST))
    if rng.random() < CFG["p_filler_info"]:
        infos.append(ctx.fmt(B.pick((bk, "finfo"), FILL_INFO)))

    core = [goal_unit] + ([via_unit] if via_unit else [])
    if via_unit and rng.random() < 0.5:
        core.reverse()
    body = list(core)
    for u in extras + attrs + infos:
        body.insert(rng.randint(0, len(body)), u)
    if pre:
        body.insert(0, pre)
    if post:
        body.append(post)

    r_style = rng.random()
    style = ("chat" if r_style < CFG["p_chat"]
             else "terse" if r_style < CFG["p_chat"] + CFG["p_terse"] else "formal")
    masked = render_units(body, rng, style)

    ok = (PH["G"] in masked
          and ((via != "none") == (PH["V"] in masked))
          and not (via == "none" and PH["TV"] in masked)
          and "{" not in masked and "}" not in masked)
    if not ok:
        return None

    pos_g, pos_t = masked.find(PH["G"]), masked.find(PH["T1"])

    # Ở BƯỚC NÀY, thực thể mới được gỡ placeholder ra (sau khi văn cảnh đã chèn typo)
    if has_typo and rng.random() < CFG["p_sentence_typo"]:
        masked = add_typo(masked, rng)
    if style == "chat":
        masked = chat_noise(masked, rng)

    txt = re.sub(r"@(\d+)@", lambda m: surf.get(int(m.group(1)), m.group(0)), masked)
    if "@" in txt:
        return None
    if rng.random() < CFG["p_lower"]:
        txt = txt.lower()
    if not has_acc:
        txt = fold(txt)
    txt = norm_ws(txt)

    y = {
        "goal": H2I["goal"][goal], "via": H2I["via"][via],
        "gkind": H2I["gkind"][gk], "ganchor": H2I["ganchor"][ga],
        "vkind": H2I["vkind"][vk], "vanchor": H2I["vanchor"][va],
        "urgent": H2I["urgent"][urgent], "fragile": H2I["fragile"][fragile],
    }
    meta = {"gtype": gtype, "via_mode": via_mode, "extras": extra_kinds,
            "attr_mode": attr_mode, "style": style, "persona": persona,
            "goal_first": (pos_t < 0 or pos_g < pos_t) if pos_g >= 0 else None,
            "has_distractor": gtype not in ("plain",) or bool(extras)}
    return txt, y, meta


# ---------------------------------------------------------------
# 8. GENERATOR CHÍNH (cùng chữ ký với V2 + tham số holdout)
# ---------------------------------------------------------------
GEN_META = {}

def generate_15k_dataset(seed=42, num_per_bucket=1875, dedupe=True,
                         goal_types=None, exclude_goal_types=()):
    rng = random.Random(seed)
    B = Balancer(rng)
    allowed = set(goal_types) if goal_types else None
    exclude = set(exclude_goal_types or ())
    GEN_META.clear()

    buckets = {n: [] for n in [
        "Aff_Acc_NoTypo", "Aff_Acc_Typo", "Aff_NoAcc_NoTypo", "Aff_NoAcc_Typo",
        "Neg_Acc_NoTypo", "Neg_Acc_Typo", "Neg_NoAcc_NoTypo", "Neg_NoAcc_Typo"]}
    global_seen = set()
    bad = 0

    for bk in buckets:
        is_neg = bk.startswith("Neg")
        has_acc = "NoAcc" not in bk
        has_typo = "NoTypo" not in bk
        attempts, max_attempts = 0, max(num_per_bucket * 60, 20000)
        while len(buckets[bk]) < num_per_bucket:
            attempts += 1
            if attempts > max_attempts:
                raise RuntimeError(f"Không tạo đủ dữ liệu unique cho {bk}: "
                                   f"{len(buckets[bk])}/{num_per_bucket}.")
            res = _make_sample(rng, B, bk, is_neg, has_acc, has_typo, allowed, exclude)
            if res is None:
                bad += 1
                continue
            txt, y, meta = res
            if dedupe and txt in global_seen:
                continue
            global_seen.add(txt)
            GEN_META[txt] = meta
            buckets[bk].append({"text": txt, "y": y, "syn": True})
    if bad:
        print(f"  (bỏ {bad} mẫu vi phạm bất biến label/text)")
    return buckets, B


# ---------------------------------------------------------------
# 9. AUDIT: đo độ đa dạng + cân bằng vị trí goal/distractor
# ---------------------------------------------------------------
def audit_dataset(buckets, top=5):
    texts = [it["text"] for l in buckets.values() for it in l]
    alias = sorted({a for p in PLACES for a in BANK.get(p, [p])} |
                   {a for k in (KINDS or []) for a in BANK.get(k, [k])},
                   key=len, reverse=True)
    rx = re.compile("|".join(re.escape(a) for a in alias), re.IGNORECASE) if alias else None
    sk = collections.Counter()
    for t in texts:
        s = rx.sub("<P>", t) if rx else t
        sk[re.sub(r"\d+", "<N>", s)] += 1
    n = len(texts)
    print(f"Tổng mẫu: {n} | skeleton khác nhau: {len(sk)} ({len(sk)/n:.1%})")
    print(f"Skeleton phổ biến nhất chiếm: {sk.most_common(1)[0][1]/n:.3%}")
    gt = collections.Counter(GEN_META[t]["gtype"] for t in texts if t in GEN_META)
    print("Phân bố kiểu goal-unit:", dict(gt))
    gf = [GEN_META[t]["goal_first"] for t in texts
          if t in GEN_META and GEN_META[t]["has_distractor"]
          and GEN_META[t]["goal_first"] is not None]
    if gf:
        print(f"Goal đứng TRƯỚC distractor: {sum(gf)/len(gf):.1%} (lý tưởng ~50%)")
    lens = [len(t.split()) for t in texts]
    print(f"Độ dài (từ): min {min(lens)}, TB {sum(lens)/n:.1f}, max {max(lens)}")


# ================================================================
# 10. BUILD TRAIN SET (giữ nguyên luồng cũ)
# ================================================================
print("Đang sinh 15.000 dữ liệu cân bằng + đa dạng cấu trúc (V3.1 - Enhanced Typos & Logic)...")
bal_buckets, BalancerObj = generate_15k_dataset(seed=42, num_per_bucket=6000, dedupe=True)

final_15k_items = []
for bucket_name, bucket_items in bal_buckets.items():
    final_15k_items.extend(bucket_items)
    print(f"  {bucket_name}: {len(bucket_items)}")

print("Đang gộp thêm dữ liệu Train gốc + no-accent...")
train_extended = []
for it in base_items:
    train_extended.append({"text": it["text"], "y": it["y"]})
    train_extended.append({"text": fold(it["text"]), "y": it["y"]})

final_train_items = train_extended + final_15k_items
print(f"✅ Synthetic: {len(final_15k_items)}")
print(f"✅ Train gốc mở rộng: {len(train_extended)}")
print(f"✅ Tổng train sau gộp: {len(final_train_items)}")
audit_dataset(bal_buckets)

Đang sinh 15.000 dữ liệu cân bằng + đa dạng cấu trúc (V3.1 - Enhanced Typos & Logic)...
  Aff_Acc_NoTypo: 6000
  Aff_Acc_Typo: 6000
  Aff_NoAcc_NoTypo: 6000
  Aff_NoAcc_Typo: 6000
  Neg_Acc_NoTypo: 6000
  Neg_Acc_Typo: 6000
  Neg_NoAcc_NoTypo: 6000
  Neg_NoAcc_Typo: 6000
Đang gộp thêm dữ liệu Train gốc + no-accent...
✅ Synthetic: 48000
✅ Train gốc mở rộng: 4000
✅ Tổng train sau gộp: 52000
Tổng mẫu: 48000 | skeleton khác nhau: 47999 (100.0%)
Skeleton phổ biến nhất chiếm: 0.004%
Phân bố kiểu goal-unit: {'canceled_order': 1140, 'old_info': 1159, 'out_of_bounds': 1159, 'plain': 7131, 'self_correct': 1534, 'hypo': 984, 'transfer_person': 1169, 'unavailable': 1164, 'pos_correct': 1109, 'recipient': 787, 'temporal': 1442, 'pickup_goal': 777, 'future_cond': 1166, 'other_order': 1217, 'via_not_goal': 1129, 'swap_package': 1167, 'wait_vs_deliver': 1155, 'preference_override': 1155, 'past_completed': 1185, 'coref': 1035, 'coref_neg': 1062, 'pos_neg_anchor': 1301, 'note_wrong': 1658, 'misdirected'

In [ ]:
import random

print("Đang sinh thêm 100 dữ liệu khó cho tập Validation...")

# Chỉ định các cấu trúc hiếm/khó (dựa trên Cell 2/3 cũ của bạn)
HARD_TYPES = ("note_wrong", "old_info", "qa_reject")

# Sinh dư ra một chút (num_per_bucket=50) để lát nữa lọc trùng lặp thoải mái
hard_buckets, _ = generate_15k_dataset(seed=888, num_per_bucket=200, goal_types=HARD_TYPES)

# Gom dữ liệu từ các bucket (ưu tiên các bucket không dính Typo để làm val cho chuẩn)
raw_hard_items = []
for bucket_name, items in hard_buckets.items():
    if "NoTypo" in bucket_name:
        raw_hard_items.extend(items)

# Tạo tập hợp các câu đã có trong Train và Val để chống rò rỉ dữ liệu (Data Leakage)
# Thay 'final_train_items' bằng 'train_items5' nếu bạn đã gộp như hướng dẫn trước đó
_existing_texts = {it['text'] for it in final_train_items} | {it['text'] for it in val_items}

# Lọc các câu chưa từng xuất hiện
unique_hard_items = [it for it in raw_hard_items if it['text'] not in _existing_texts]

# Trộn đều và cắt lấy đúng 100 câu
random.seed(888)
random.shuffle(unique_hard_items)
hard_val_items = unique_hard_items[:1000]

# Gộp vào tập Validation gốc
val_items.extend(hard_val_items)

print(f"✅ Đã chèn thêm {len(hard_val_items)} câu khó vào Validation.")
print(f"✅ Tổng dung lượng tập Val hiện tại: {len(val_items)} câu.")

# In thử 3 câu ra xem độ khó
for i, it in enumerate(hard_val_items[:3]):
    print(f"  {i+1}. {it['text'][:120]}")

Đang sinh thêm 100 dữ liệu khó cho tập Validation...
✅ Đã chèn thêm 800 câu khó vào Validation.
✅ Tổng dung lượng tập Val hiện tại: 1100 câu.
  1. Điểm giao: phía trên cùng khu học. sân tập chỉ là địa chỉ cũ. Vừa gấp vừa nhẹ tay, ráng giúp anh. Đơn thu hộ COD.
  2. Này. chỗ học tập là điểm giao đúng không? Không nha, đem trực tiếp đến khu ký túc xá nhé ạ. Càng sớm càng tốt. Hàng thườ
  3. Lần trước bàn giao đơn máy chiếu tại phòng đào tạo khá xa chỗ vận động, lần này thật ra là classroom, phía trái. Nhờ bạn


In [ ]:
print(len(final_train_items))

52000


In [ ]:
# ===== PHASE 5 / BLOCK 5: FINETUNE TIẾP TỪ FT5 TẠO RA FT6 =====
P5_IN    = "/content/bamibert_nlu_ft711.pt"
P5_OUT   = "/content/bamibert_nlu_ft711.pt"
P5_DRIVE = "/content/drive/MyDrive/bamibert_nlu_ft711.pt"
P5_LAST  = "/content/bamibert_nlu_ft711_last.pt" # <--- MỚI THÊM: Đường dẫn lưu file last

assert os.path.exists(P5_IN), f"Không thấy {P5_IN}"

# Khai báo bổ sung các hàm đánh giá bị thiếu
def probe_eval(model):
    res = {}
    for v, txt in PROBE_VIEWS.items():
        pred = decode(predict_probs(model, txt))
        core, full = exact_masks(pred, GOLD_PROBE)
        res[v] = (float(core.mean()), float(full.mean()))
    return res

def p4_score(model):
    rv = eval_views(model, SEL_VIEWS)
    rp = probe_eval(model)
    sv = sel_score(rv)
    sp = float(np.mean([0.5 * (c + f) for c, f in rp.values()]))
    return W_VAL * sv + W_PROBE * sp, sv, sp, rv, rp

LR_ENC_P5, LR_HEAD_P5 = 4e-6, 1.2e-4
P5_EPOCHS, P5_PATIENCE = 2, 30
NOISE_HI_P5, NOISE_LO_P5 = 0.5, 0.3
W_VAL, W_PROBE = 0.6, 0.4
VAL_MAX_DROP = 0.01

gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
model = MultiHeadNLU(MODEL_NAME).to(device)
model.load_state_dict(torch.load(P5_IN, map_location=device)['state']); print("Đã nạp:", P5_IN)

ds = TrainDS(final_train_items)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
print(f"mẫu / epoch: {len(ds)} | bước / epoch: {len(loader)}")
enc_p  = [p for k, p in model.named_parameters() if k.startswith('enc.')]
head_p = [p for k, p in model.named_parameters() if not k.startswith('enc.')]
opt = torch.optim.AdamW([{'params': enc_p, 'lr': LR_ENC_P5}, {'params': head_p, 'lr': LR_HEAD_P5}], weight_decay=WD)
total = len(loader) * P5_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
use_scaler = USE_AMP and AMP_DTYPE == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
ema = EMA(model, EMA_DECAY)

base_sc, base_sv, base_sp, _, rp0 = p4_score(model)
print(f"Điểm xuất phát (ft5): tổng={base_sc:.4f} | val={base_sv:.4f} | probe={base_sp:.4f}")
best, best_ep, bad = base_sc, 0, 0

for ep in range(1, P5_EPOCHS + 1):
    ds.noise_scale = NOISE_HI_P5 if ep <= 0.6 * P5_EPOCHS else NOISE_LO_P5
    model.train(); run = run_al = 0.0
    for ids, mask, y, pidx, ib, mb in tqdm(loader, desc=f"P5-{ep}", leave=False):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            lg_a, ea = model(ids.to(device), mask.to(device), return_emb=True)
            loss = compute_loss(lg_a, y); al = torch.zeros((), device=device)
            if pidx is not None:
                lg_b, eb = model(ib.to(device), mb.to(device), return_emb=True)
                loss = loss + compute_loss(lg_b, {h: y[h][pidx] for h in HEADS})
                al = (1 - (ea[pidx.to(device)] * eb).sum(-1)).mean()
                loss = loss + ALIGN_W * al
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if use_scaler:
            sb = scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale() >= sb: sched.step()
        else:
            opt.step(); sched.step()
        ema.update(model); run += loss.item(); run_al += float(al)

    ema.apply(model); sc, sv, sp, rv, rp = p4_score(model)

    # <--- MỚI THÊM: Lưu đè file last ở mỗi epoch bất kể điểm số ra sao
    torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, P5_LAST)

    msg = (f"P5-{ep:02d} loss={run / len(loader):.3f} align={run_al / len(loader):.3f} | tổng={sc:.4f} val={sv:.4f} probe={sp:.4f} | "
           + 'val: ' + ' '.join(f"{n}:{rv[n]['full']:.3f}" for n in SEL_VIEWS) + ' | probe: ' + ' '.join(f"{v}:{f:.3f}" for v, (c, f) in rp.items()))

    if sc > best + 1e-4 and sv >= base_sv - VAL_MAX_DROP:
        best, best_ep, bad = sc, ep, 0
        torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, P5_OUT)
        try: shutil.copy(P5_OUT, P5_DRIVE)
        except Exception as e: print('  (Drive lỗi:', e, ')')
        msg += '  * (kỷ lục mới)'
    else: bad += 1
    ema.restore(model); print(msg)
    if bad >= P5_PATIENCE: print("Dừng sớm."); break

print(f"\nHoàn tất. Tốt nhất ở epoch {best_ep}: tổng={best:.4f} -> {P5_OUT}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã nạp: /content/bamibert_nlu_ft711.pt
mẫu / epoch: 56000 | bước / epoch: 1750
Điểm xuất phát (ft5): tổng=0.9610 | val=0.9958 | probe=0.9088


P5-1:   0%|          | 0/1750 [00:00<?, ?it/s]

P5-01 loss=5.033 align=0.000 | tổng=0.9553 val=0.9958 probe=0.8946 | val: gốc:1.000 bỏ dấu:1.000 typo:0.987 bỏ dấu+typo:0.997 | probe: gốc:0.892 bỏ dấu:0.895 bỏ dấu+typo:0.873


P5-2:   0%|          | 0/1750 [00:00<?, ?it/s]

P5-02 loss=4.989 align=0.000 | tổng=0.9568 val=0.9967 probe=0.8971 | val: gốc:1.000 bỏ dấu:1.000 typo:0.987 bỏ dấu+typo:1.000 | probe: gốc:0.895 bỏ dấu:0.895 bỏ dấu+typo:0.875

Hoàn tất. Tốt nhất ở epoch 0: tổng=0.9610 -> /content/bamibert_nlu_ft711.pt


In [ ]:
# ===== PHASE 6 / BLOCK 6: EPOCH CUỐI - ép model bền với "mất 1 chữ" ở tên địa điểm (nặng / typo / bỏ dấu+typo) =====
# Chạy SAU: Block 4 (p4_score, probe_eval), cell hard-val (cell bọc p4_score), và khi `final_train_items` còn tồn tại.
from torch.utils.data import RandomSampler
import os

P6_IN    = "/content/bamibert_nlu_ft711.pt"      # Đã sửa: Trỏ về checkpoint hiện có sẵn trong môi trường của bạn
P6_OUT   = "/content/bamibert_nlu_ft712.pt"
P6_DRIVE = "/content/drive/MyDrive/bamibert_nlu_ft712.pt"
P6_LAST  = "/content/bamibert_nlu_ft64_last.pt"
assert os.path.exists(P6_IN), f"Không thấy {P6_IN}"
assert 'final_train_items' in globals(), "Thiếu final_train_items (tập train Block 5 đang dùng)"
P6_SRC = final_train_items

P6_EPOCHS, P6_STEPS = 3, 900
LR_ENC_P6, LR_HEAD_P6 = 7.5e-7, 2.5e-5
P6_MIX = dict(clean=0.10, std=0.25, heavy=0.30, drop=0.35)   # tỉ lệ các kiểu nhiễu khi train
P6_HARD_FRAC, P6_MINE_N, P6_REMINE, P6_HARD_MAX = 0.25, 8000, 2, 3000
P6_W = (0.6, 0.2, 0.2)                               # điểm chọn = 0.6*p4_score + 0.2*'nặng' + 0.2*nhiễu MỚI (không dùng để học)
P6_MAX_DROP = 0.01                                   # không lưu nếu val tụt >1 điểm % so với xuất phát
FRESH_REP = 2

# --- NOISE HELPERS START ---
def heavy_noise(t, rng):
    """Cùng công thức view 'nặng' nhưng tham số dao động (để không học thuộc đúng 1 mẫu nhiễu)."""
    return merge_or_split(
        multi_typo(strip_accents(t), rng, rng.choice([0.10, 0.15, 0.20]), rng.choice([4, 6, 8])),
        rng, rng.choice([0.05, 0.08, 0.12]))

def drop_in_place(text, rng=random, n=1):
    """Xóa ĐÚNG 1 ký tự (vị trí bất kỳ, kể cả đầu/cuối) trong tên địa điểm; 35% thêm 1 lỗi nữa trên cùng từ."""
    t = unicodedata.normalize('NFC', text); f = ''.join(strip_accents(c) for c in t).lower()
    if len(t) != len(f): return text
    spans = [m.span() for m in ALIAS_RE.finditer(f)]
    if not spans: return text
    for s, e in sorted(rng.sample(spans, min(n, len(spans))), reverse=True):
        ws = [m for m in _WORD2.finditer(t, s, e) if len(m.group()) >= 4]
        if not ws: continue
        m = rng.choice(ws); w = m.group(); i = rng.randrange(len(w))
        w = w[:i] + w[i + 1:]
        if rng.random() < 0.35: w = _typo_word(w, rng)
        t = t[:m.start()] + w + t[m.end():]
    return t

def p6_noise(base, y, rng, scale=0.5, mode=None):
    mode = mode or rng.choices(list(P6_MIX), list(P6_MIX.values()))[0]
    if mode == 'clean': return base
    if mode == 'std':   return augment_v2(base, y, rng, scale)
    if mode == 'heavy': return heavy_noise(base, rng)
    t = strip_accents(base) if rng.random() < 0.5 else base
    t = drop_in_place(t, rng, rng.choice([1, 1, 2]))
    if rng.random() < 0.5: t = multi_typo(t, rng, rng.choice([0.03, 0.05, 0.08]))
    if rng.random() < 0.2: t = merge_or_split(t, rng)
    return t
# --- NOISE HELPERS END ---

class P6DS(TrainDS):
    def __init__(self, items, hard=()):
        super().__init__(items); self.hard = list(hard); self.noise_scale = 0.5
    def __getitem__(self, i):
        if self.hard and random.random() < P6_HARD_FRAC:
            it = random.choice(self.hard); t = it['text']          # câu model từng làm SAI: giữ nguyên, không nhiễu thêm
        else:
            it = self.items[self.index[i]]; t = p6_noise(it['text'], it['y'], random, self.noise_scale)
        b = strip_accents(t) if (has_accent(t) and random.random() < P_PAIR) else None
        return {'a': t, 'b': b, 'y': it['y']}

@torch.no_grad()
def p6_mine(model, seed):
    """Cho model tự thi trên câu TRAIN bị nhiễu (nặng / mất chữ / typo); câu sai -> dữ liệu 'khó' cố định."""
    rng = random.Random(seed); model.eval()
    cand = []
    for it in rng.sample(P6_SRC, min(P6_MINE_N, len(P6_SRC))):
        for mode in ('heavy', 'drop', 'typo'):
            t = multi_typo(it['text'], rng, 0.08) if mode == 'typo' else p6_noise(it['text'], it['y'], rng, mode=mode)
            if t != it['text']: cand.append((t, it['y'], mode))
    pred = decode(predict_probs(model, [c[0] for c in cand]))
    gold = {h: np.array([c[1][h] for c in cand]) for h in HEADS}
    _, full = exact_masks(pred, gold)
    bad = [(dict(text=c[0], y=c[1], syn=True, p6=True), c[2]) for c, ok in zip(cand, full) if not ok]
    rng.shuffle(bad)
    return [b for b, _ in bad[:P6_HARD_MAX]], len(cand), collections.Counter(m for _, m in bad)

# --- nhiễu MỚI (seed khác val) để biết giảm lỗi là thật, không phải khớp đúng mẫu nhiễu của val ---
_V0 = val_items[:len(GOLD_VAL['goal'])]
_rf = random.Random(31337)
FRESH_VIEWS = {'nặng*': [], 'mất-chữ-địa-điểm*': [], 'typo*': []}
for _ in range(FRESH_REP):
    for _it in _V0:
        _t = _it['text']
        FRESH_VIEWS['nặng*'].append(merge_or_split(multi_typo(strip_accents(_t), _rf, 0.15, 6), _rf, 0.08))
        FRESH_VIEWS['mất-chữ-địa-điểm*'].append(drop_in_place(strip_accents(_t) if _rf.random() < 0.5 else _t, _rf, 1))
        FRESH_VIEWS['typo*'].append(multi_typo(_t, _rf, 0.08))
FRESH_GOLD = gold_of(_V0 * FRESH_REP)

def fresh_eval(model):
    out = {}
    for n, txt in FRESH_VIEWS.items():
        core, full = exact_masks(decode(predict_probs(model, txt)), FRESH_GOLD)
        out[n] = (int((~full).sum()), len(txt), 0.5 * (float(core.mean()) + float(full.mean())))
    return out

def p6_score(model):
    sc, sv, sp, rv, rp = p4_score(model)
    rn = eval_views(model, ['nặng'])['nặng']; sn = 0.5 * (rn['core'] + rn['full'])
    fr = fresh_eval(model); sf = float(np.mean([v[2] for v in fr.values()]))
    return P6_W[0] * sc + P6_W[1] * sn + P6_W[2] * sf, sv, sp, rv, rn, fr

def p6_errs(rv, rn, fr):
    e = {n: int((~rv[n]['ok_full']).sum()) for n in SEL_VIEWS}; e['nặng'] = int((~rn['ok_full']).sum())
    return ' '.join(f"{n}:{k}" for n, k in e.items()) + ' | mới: ' + ' '.join(f"{n}:{k}/{m}" for n, (k, m, _) in fr.items())

gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
model = MultiHeadNLU(MODEL_NAME).to(device)
model.load_state_dict(torch.load(P6_IN, map_location=device)['state']); print("Đã nạp:", P6_IN)

ds = P6DS(P6_SRC)
loader = DataLoader(ds, batch_size=BATCH_SIZE, sampler=RandomSampler(ds, replacement=True, num_samples=P6_STEPS * BATCH_SIZE), collate_fn=collate)
enc_p  = [p for k, p in model.named_parameters() if k.startswith('enc.')]
head_p = [p for k, p in model.named_parameters() if not k.startswith('enc.')]
opt = torch.optim.AdamW([{'params': enc_p, 'lr': LR_ENC_P6}, {'params': head_p, 'lr': LR_HEAD_P6}], weight_decay=WD)
total = len(loader) * P6_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.05 * total), total)
use_scaler = USE_AMP and AMP_DTYPE == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
ema = EMA(model, EMA_DECAY)

base_tot, base_sv, base_sp, rv0, rn0, fr0 = p6_score(model)
print(f"XUẤT PHÁT: điểm chọn={base_tot:.4f} | val={base_sv:.4f} | probe={base_sp:.4f}")
print("  số câu SAI -> " + p6_errs(rv0, rn0, fr0))
base_gốc = rv0['gốc']['full']
best, best_ep, bad = base_tot, 0, 0

for ep in range(1, P6_EPOCHS + 1):
    if (ep - 1) % P6_REMINE == 0:
        hard, n_c, st = p6_mine(model, seed=SEED + ep)
        ds.hard = hard; print(f"  [mine] model sai {len(hard)}/{n_c} câu nhiễu (trước khi cắt trần) | theo kiểu: {dict(st)}")
    model.train(); run = run_al = 0.0
    for ids, mask, y, pidx, ib, mb in tqdm(loader, desc=f"P6-{ep}", leave=False):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            lg_a, ea = model(ids.to(device), mask.to(device), return_emb=True)
            loss = compute_loss(lg_a, y); al = torch.zeros((), device=device)
            if pidx is not None:
                lg_b, eb = model(ib.to(device), mb.to(device), return_emb=True)
                loss = loss + compute_loss(lg_b, {h: y[h][pidx] for h in HEADS})
                al = (1 - (ea[pidx.to(device)] * eb).sum(-1)).mean()
                loss = loss + ALIGN_W * al
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if use_scaler:
            sb = scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale() >= sb: sched.step()
        else:
            opt.step(); sched.step()
        ema.update(model); run += loss.item(); run_al += float(al)

    ema.apply(model); tot, sv, sp, rv, rn, fr = p6_score(model)
    torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, P6_LAST)
    msg = (f"P6-{ep:02d} loss={run / len(loader):.3f} align={run_al / len(loader):.3f} | điểm={tot:.4f} val={sv:.4f} probe={sp:.4f}\n"
           f"      SAI -> {p6_errs(rv, rn, fr)}")
    if tot > best + 1e-4 and sv >= base_sv - P6_MAX_DROP and rv['gốc']['full'] >= base_gốc - 0.01:
        best, best_ep, bad = tot, ep, 0
        torch.save({'state': model.state_dict(), 'model_name': MODEL_NAME, 'rules': RULES}, P6_OUT)
        try: shutil.copy(P6_OUT, P6_DRIVE)
        except Exception as e: print('  (Drive lỗi:', e, ')')
        msg += '  * (kỷ lục mới)'
    else: bad += 1
    ema.restore(model); print(msg)

print(f"\nXUẤT PHÁT  : {p6_errs(rv0, rn0, fr0)}")
if best_ep == 0:
    print(f"⚠️ Không epoch nào vượt xuất phát ({base_tot:.4f}) -> KHÔNG lưu {P6_OUT}. Giữ nguyên {P6_IN}.")
else:
    model.load_state_dict(torch.load(P6_OUT, map_location=device)['state']); model.eval()
    _, _, _, rvB, rnB, frB = p6_score(model)
    print(f"TỐT NHẤT  : {p6_errs(rvB, rnB, frB)}   (epoch {best_ep}, điểm {best:.4f}) -> {P6_OUT}")
    res = eval_views(model, ['typo', 'bỏ dấu+typo', 'nặng'], constrained=True)
    for nm, r in res.items():
        idx = np.where(~r['ok_full'])[0]; print(f"\n=== CÒN SAI {len(idx)}/{len(VAL_VIEWS[nm])} ở '{nm}' ===")
        for i in idx:
            g = {h: HEADS[h][GOLD_VAL[h][i]] for h in HEADS}; p = {h: HEADS[h][r['pred'][h][i]] for h in HEADS}
            print(' •', VAL_VIEWS[nm][i][:170]); print('    (đúng -> đoán):', {h: (g[h], p[h]) for h in HEADS if g[h] != p[h]})

In [ ]:
# CELL MỚI — PHASE 7: sửa nhãn synthetic, học cặp đối chứng, chấm probe riêng.
# Chạy trong runtime đã có tokenizer/model/helpers và generator cell 46.
# Không cần chạy lại các vòng train cũ. P7_IN là checkpoint bắt đầu, sửa nếu cần.
import os, itertools

def p7_item(text, goal, family, via=None, gkind=None, ga=None,
            vkind=None, va=None, urgent=False, fragile=False):
    mission = dict(goal=goal, via=via, urgent=urgent, fragile=fragile,
                   goal_ref=dict(kind=gkind, anchor=ga) if gkind else None,
                   via_ref=dict(kind=vkind, anchor=va) if vkind else None)
    return dict(text=text, y=mission_to_labels(mission), syn=True, family=family)

def p7_pairs(holdout=False):
    # Hai bộ dùng khung câu khác nhau; probe không đưa vào train của phase này.
    places = dict(canteen='căn tin', clinic='trạm y tế', dorm='ký túc xá',
                  gate='cổng trường', lab='phòng thí nghiệm', lecture='giảng đường',
                  library='thư viện', office='phòng hành chính', parking='bãi xe',
                  sports='nhà thể thao')
    keys = list(places)
    rows = []
    def add(train_text, probe_text, goal, family, **labels):
        rows.append(p7_item(probe_text if holdout else train_text, goal, family, **labels))
    for i, g in enumerate(keys):
        v, a, b = (keys[(i + j) % len(keys)] for j in (1, 2, 3))
        G, V, A, B = (places[k] for k in (g, v, a, b))
        for kind, word in [('near', 'gần'), ('far', 'xa')]:
            add(f'Giao bưu kiện tới {G} {word} {V} hơn.',
                f'Trong các {G}, nơi nhận hàng là chỗ nằm {word} {V} hơn.',
                g, 'goal_anchor', gkind=kind, ga=v)
            add(f'Giao bưu kiện tới {V} {word} {G} hơn.',
                f'Trong các {V}, nơi nhận hàng là chỗ nằm {word} {G} hơn.',
                v, 'goal_anchor', gkind=kind, ga=g)
        for rev in (False, True):
            vk, gk = ('far', 'near') if rev else ('near', 'far')
            vw, gw = ('xa', 'gần') if rev else ('gần', 'xa')
            add(f'Ghé {V} {vw} {A} hơn lấy hàng trước, rồi giao tới {G} {gw} {B} hơn.',
                f'Nơi giao cuối là {G} {gw} {B} hơn; trước khi tới đó, lấy hàng tại {V} {vw} {A} hơn.',
                g, 'two_refs', via=v, gkind=gk, ga=b, vkind=vk, va=a)
        add(f'Robot ghé {V} lấy hàng rồi giao tới {G}.',
            f'Nơi lấy kiện hàng là {V}; nơi nhận cuối cùng là {G}.',
            g, 'pickup_subject', via=v)
        add(f'Giao tới {G} để người đang ở {V} tự đến lấy.',
            f'Người nhận sẽ rời {V} để tự tới {G} nhận hàng; robot chỉ cần giao tại {G}.',
            g, 'pickup_subject')
        add(f'Ghé {V} lấy phiếu xác nhận trước, rồi giao hàng tới {G}.',
            f'Trước lúc bàn giao ở {G}, robot phải dừng tại {V} nhận phiếu.',
            g, 'stop_vs_pass', via=v)
        add(f'Giao hàng tới {G}; đi ngang {V} thôi, không dừng ở đó.',
            f'Điểm nhận là {G}. {V} chỉ là mốc dọc đường, không cần ghé.',
            g, 'stop_vs_pass')
        add(f'Giao thẻ thư viện tới {G}.',
            f'Nơi nhận thẻ thư viện là {G}.', g, 'cargo_name')
        add(f'Người nhận đang chờ bưu kiện tại {G}.',
            f'Có người đợi nhận hàng ở {G}.', g, 'waiting_not_urgent')
        add(f'Người nhận đang chờ bưu kiện tại {G}; cần giao gấp.',
            f'Có người đợi nhận hàng ở {G}; phải giao ngay.',
            g, 'waiting_not_urgent', urgent=True)
        add(f'Mai giao tới {G} cũng được; gọi báo cho tôi ngay.',
            f'Nhắn xác nhận ngay nhé; bưu kiện tới {G} thì ngày mai mới cần.',
            g, 'urgent_scope')
        add(f'Giao tới {G}. Hàng bền; đi cẩn thận vì đường trơn.',
            f'Đường trơn nên robot đi cẩn thận. Kiện giao tới {G} là hàng chắc chắn.',
            g, 'fragile_scope')
        add(f'Giao tới {G}. Hàng dễ vỡ; cần tránh rung lắc.',
            f'Kiện gửi tới {G} rất mong manh, phải giữ nhẹ tay.',
            g, 'fragile_scope', fragile=True)
        for u, f in itertools.product((False, True), repeat=2):
            U, F = ('gấp' if u else 'không gấp'), ('dễ vỡ' if f else 'bền chắc')
            old_u, old_f = ('không gấp' if u else 'gấp'), ('bền chắc' if f else 'dễ vỡ')
            add(f'Ghé {V} lấy hàng rồi giao tới {G}. Đơn này {U}, hàng {F}.',
                f'Hàng {F}. Nơi nhận cuối là {G}; lấy hàng ở {V} trước. Đơn này {U}.',
                g, 'two_flags', via=v, urgent=u, fragile=f)
            add(f'Đơn kia {old_u}, hàng {old_f}. Đơn này giao tới {G}, {U}, hàng {F}.',
                f'Kiện hiện tại gửi tới {G}: {U}, hàng {F}. Kiện {old_u} và {old_f} là của người khác.',
                g, 'other_order', urgent=u, fragile=f)
    return rows

def p7_self_check():
    train, probe = p7_pairs(), p7_pairs(True)
    assert len(train) == len(probe) == 240
    assert not ({fold(x['text']) for x in train} & {fold(x['text']) for x in probe})
    for rows in (train, probe):
        for x in rows:
            y = {h: HEADS[h][v] for h, v in x['y'].items()}
            assert (y['gkind'] in ('near', 'far')) == (y['ganchor'] != 'none')
            assert (y['vkind'] in ('near', 'far')) == (y['vanchor'] != 'none')
            if y['via'] == 'none':
                assert y['vkind'] == y['vanchor'] == 'none'
        flags = [x for x in rows if x['family'] == 'two_flags']
        assert {(x['y']['urgent'], x['y']['fragile']) for x in flags} == set(itertools.product((0, 1), repeat=2))
        # Same mission, only the two flags change across each group of four.
        for i in range(0, len(flags), 4):
            assert len({tuple(x['y'][h] for h in HEADS if h not in ('urgent', 'fragile'))
                        for x in flags[i:i + 4]}) == 1
    # Independent expected labels for the first role-swap and dual-reference cases.
    assert train[0]['y']['goal'] == H2I['goal']['canteen']
    assert train[0]['y']['ganchor'] == H2I['ganchor']['clinic']
    assert train[1]['y']['goal'] == H2I['goal']['clinic']
    assert train[1]['y']['ganchor'] == H2I['ganchor']['canteen']
    assert train[4]['y']['via'] == H2I['via']['clinic']
    assert train[4]['y']['vanchor'] == H2I['vanchor']['dorm']
    assert train[4]['y']['ganchor'] == H2I['ganchor']['gate']
    print('P7 self-check: nhãn, đổi vai trò, bốn tổ hợp cờ, train/probe tách biệt — OK')

# --- CẤU HÌNH: giữ LR thấp đang dùng ở Phase 6; không giảm thêm lần nữa. ---
P7_IN = '/content/bamibert_nlu_ft711.pt'
P7_OUT = '/content/bamibert_nlu_ft713_best.pt'
P7_LAST = '/content/bamibert_nlu_ft713_last.pt'
P7_DRIVE = '/content/drive/MyDrive/bamibert_nlu_ft713_best.pt'
P7_EPOCHS = 3
P7_LR_ENC, P7_LR_HEAD = 7.5e-7, 2.5e-5
assert os.path.isfile(P7_IN), f'Chỉnh P7_IN tới checkpoint muốn fine-tune: {P7_IN}'
assert os.path.abspath(P7_IN) not in {os.path.abspath(P7_OUT), os.path.abspath(P7_LAST), os.path.abspath(P7_DRIVE)}
p7_self_check()

# Sửa generator cho những lần sinh dữ liệu tiếp theo; không dùng lại synthetic cũ.
GOAL_TPL['via_not_goal'] = [s for s in GOAL_TPL['via_not_goal'] if 'đổ xăng' not in s]
for key, words in {
    ('urgent', 'yes', 'std'): {'khách đang đứng chờ rồi', 'khách chờ lâu rồi'},
    ('urgent', 'yes', 'hard'): {'tuy đơn nhỏ nhưng khách chờ rồi'},
}.items():
    ATTR[key] = [s for s in ATTR[key] if s not in words]
BANK['fragile'] = ['dễ vỡ', 'dễ bể', 'dễ nứt', 'mong manh', 'dễ hỏng khi va đập']

# Đọc lại split gốc, tránh val_items đã bị nối hard-val từ cell 47.
_p7_real = load_missions(TRAIN_SCENES)
_p7_public = load_missions(VAL_SCENES)
_p7_target, _p7_probe = p7_pairs(), p7_pairs(True)
_p7_rng = random.Random(2024)
P7_EVAL = {
    'public': _p7_public,
    'public_noaccent': [dict(x, text=strip_accents(x['text'])) for x in _p7_public],
    'public_typo': [dict(x, text=multi_typo(x['text'], _p7_rng, 0.08)) for x in _p7_public],
    'public_heavy': [dict(x, text=merge_or_split(multi_typo(strip_accents(x['text']), _p7_rng, 0.15, 6), _p7_rng, 0.08)) for x in _p7_public],
    'probe': _p7_probe,
    'probe_noaccent': [dict(x, text=strip_accents(x['text'])) for x in _p7_probe],
}
_p7_held = {fold(x['text']) for rows in P7_EVAL.values() for x in rows}
_p7_train = []
for group, repeats in ((_p7_real, 1), (_p7_target, 3)):
    for x in group:
        if fold(x['text']) in _p7_held:
            continue
        for text in dict.fromkeys((x['text'], strip_accents(x['text']))):
            _p7_train.extend([dict(a=text, b=None, y=x['y'])] * repeats)
assert _p7_train and not ({fold(x['a']) for x in _p7_train} & _p7_held)

# Kiểm tra token thật; không âm thầm cắt mất cờ hoặc địa điểm ở cuối câu.
_p7_texts = [x['a'] for x in _p7_train] + [x['text'] for rows in P7_EVAL.values() for x in rows]
_p7_lengths = [len(ids) for ids in tok(_p7_texts, truncation=False, padding=False)['input_ids']]
assert max(_p7_lengths) <= MAX_LEN, f'Có câu dài {max(_p7_lengths)} token > MAX_LEN={MAX_LEN}; xử lý trước khi train.'
print('P7 train:', len(_p7_train), '| eval:', {k: len(v) for k, v in P7_EVAL.items()}, '| max tokens:', max(_p7_lengths))

@torch.no_grad()
def p7_eval(model):
    report = {}
    for name, rows in P7_EVAL.items():
        pred = decode(predict_probs(model, [x['text'] for x in rows]))
        gold = gold_of(rows)
        _, full = exact_masks(pred, gold)
        report[name] = dict(full=float(full.mean()), errors={h: int((pred[h] != gold[h]).sum()) for h in HEADS})
        print(name, f'full={full.mean():.4f}', 'sai/head:', report[name]['errors'])
    return report

def p7_score(report):
    hard = (
        report['probe']['full']
        + report['probe_noaccent']['full']
    ) / 2

    public = (
        report['public']['full']
        + report['public_noaccent']['full']
    ) / 2

    return 0.8 * hard + 0.2 * public


def p7_preserves_public(report, baseline):
    return True  # Cho phép val giảm; chọn checkpoint bằng điểm phía trên.

set_seed(SEED)
for _p7_name in ('model', 'opt', 'ema', 'scaler'):
    globals().pop(_p7_name, None)
gc.collect()
torch.cuda.empty_cache()
model = MultiHeadNLU(MODEL_NAME).to(device)
_p7_ck = torch.load(P7_IN, map_location='cpu')
model.load_state_dict(_p7_ck['state'])
RULES = _p7_ck.get('rules', RULES)
del _p7_ck
print('P7 checkpoint đầu vào:', P7_IN)
_p7_base = p7_eval(model)
_p7_best, _p7_best_ep = p7_score(_p7_base), 0
loader = DataLoader(_p7_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if n.startswith('enc.')], 'lr': P7_LR_ENC},
    {'params': [p for n, p in model.named_parameters() if not n.startswith('enc.')], 'lr': P7_LR_HEAD},
], weight_decay=WD)
total = len(loader) * P7_EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(0.05 * total), total)
use_scaler = USE_AMP and AMP_DTYPE == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
ema = EMA(model, EMA_DECAY)
for ep in range(1, P7_EPOCHS + 1):
    model.train()
    running = 0.0
    for ids, mask, y, _, _, _ in tqdm(loader, desc=f'P7-{ep}', leave=False):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = compute_loss(model(ids.to(device), mask.to(device)), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if use_scaler:
            old_scale = scaler.get_scale()
            scaler.step(opt)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                sched.step()
        else:
            opt.step()
            sched.step()
        ema.update(model)
        running += loss.item()
    ema.apply(model)
    try:
        print(f'P7 epoch {ep}: loss={running / len(loader):.4f}')
        report = p7_eval(model)
        payload = dict(state=model.state_dict(), model_name=MODEL_NAME, rules=RULES,
                       epoch=ep, source_checkpoint=P7_IN, p7_metrics=report)
        torch.save(payload, P7_LAST)
        score = p7_score(report)
        if score > _p7_best + 1e-8 and p7_preserves_public(report, _p7_base):
            torch.save(payload, P7_OUT)
            _p7_best, _p7_best_ep = score, ep
            try:
                shutil.copy2(P7_OUT, P7_DRIVE)
            except OSError as exc:
                print('Best đã lưu local; copy Drive lỗi:', exc)
            print('Đã lưu best:', P7_OUT)
    finally:
        ema.restore(model)
# Nếu không cải thiện, nạp lại checkpoint đầu vào; không dùng best cũ sót trên đĩa.
_p7_selected = P7_OUT if _p7_best_ep else P7_IN
_p7_ck = torch.load(_p7_selected, map_location='cpu')
model.load_state_dict(_p7_ck['state'])
del _p7_ck
model.eval()
print('P7 hoàn tất; checkpoint được chọn:', _p7_selected, '| best epoch:', _p7_best_ep)


P7 self-check: nhãn, đổi vai trò, bốn tổ hợp cờ, train/probe tách biệt — OK
P7 train: 5198 | eval: {'public': 300, 'public_noaccent': 300, 'public_typo': 300, 'public_heavy': 300, 'probe': 240, 'probe_noaccent': 240} | max tokens: 97


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


P7 checkpoint đầu vào: /content/bamibert_nlu_ft711.pt
public full=1.0000 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 0}
public_noaccent full=1.0000 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 0}
public_typo full=0.9867 sai/head: {'goal': 2, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 2}
public_heavy full=0.9633 sai/head: {'goal': 6, 'via': 2, 'gkind': 2, 'ganchor': 0, 'vkind': 1, 'vanchor': 0, 'urgent': 0, 'fragile': 0}
probe full=0.7542 sai/head: {'goal': 31, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 28, 'fragile': 0}
probe_noaccent full=0.6500 sai/head: {'goal': 46, 'via': 10, 'gkind': 0, 'ganchor': 0, 'vkind': 1, 'vanchor': 1, 'urgent': 28, 'fragile': 10}


P7-1:   0%|          | 0/163 [00:00<?, ?it/s]

P7 epoch 1: loss=2.5529
public full=0.9900 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 3}
public_noaccent full=0.9967 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 1}
public_typo full=0.9800 sai/head: {'goal': 2, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 4}
public_heavy full=0.9433 sai/head: {'goal': 6, 'via': 2, 'gkind': 2, 'ganchor': 0, 'vkind': 1, 'vanchor': 0, 'urgent': 0, 'fragile': 7}
probe full=0.7917 sai/head: {'goal': 31, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 19, 'fragile': 0}
probe_noaccent full=0.7708 sai/head: {'goal': 46, 'via': 9, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 9, 'fragile': 0}
Đã lưu best: /content/bamibert_nlu_ft713_best.pt


P7-2:   0%|          | 0/163 [00:00<?, ?it/s]

P7 epoch 2: loss=2.4940
public full=0.9867 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 4}
public_noaccent full=0.9967 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 1}
public_typo full=0.9733 sai/head: {'goal': 2, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 6}
public_heavy full=0.9467 sai/head: {'goal': 6, 'via': 2, 'gkind': 2, 'ganchor': 0, 'vkind': 1, 'vanchor': 0, 'urgent': 0, 'fragile': 6}
probe full=0.7750 sai/head: {'goal': 31, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 20, 'fragile': 3}
probe_noaccent full=0.7750 sai/head: {'goal': 46, 'via': 9, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 8, 'fragile': 0}


P7-3:   0%|          | 0/163 [00:00<?, ?it/s]

P7 epoch 3: loss=2.4910
public full=0.9900 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 3}
public_noaccent full=0.9967 sai/head: {'goal': 0, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 1}
public_typo full=0.9800 sai/head: {'goal': 2, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 0, 'fragile': 4}
public_heavy full=0.9467 sai/head: {'goal': 6, 'via': 2, 'gkind': 2, 'ganchor': 0, 'vkind': 1, 'vanchor': 0, 'urgent': 0, 'fragile': 5}
probe full=0.7917 sai/head: {'goal': 31, 'via': 0, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 19, 'fragile': 0}
probe_noaccent full=0.7708 sai/head: {'goal': 46, 'via': 9, 'gkind': 0, 'ganchor': 0, 'vkind': 0, 'vanchor': 0, 'urgent': 9, 'fragile': 0}
P7 hoàn tất; checkpoint được chọn: /content/bamibert_nlu_ft713_best.pt | best epoch: 1
